In [ ]:
import pandas as pd
import numpy as np

# =========================================
# LOAD DATA
# =========================================
path = "/Users/kanishq/Documents/Projects/CO2_to_DME/vs_code/data/CO2_to_DME_Data.xlsx - Sheet1.csv"
df = pd.read_csv(path)

print("\n===== BASIC INFO =====")
print("Shape:", df.shape)
display(df.head())


# =========================================
# COLUMN INFO
# =========================================
print("\n===== COLUMN INFO =====")
df.info()


# =========================================
# DUPLICATE CHECK (IMPORTANT)
# =========================================
print("\n===== DUPLICATE CHECK =====")
duplicates = df.duplicated().sum()
print("Total duplicate rows:", duplicates)


# =========================================
# MISSING VALUE ANALYSIS (COLUMN LEVEL)
# =========================================
print("\n===== MISSING VALUES (COLUMN LEVEL) =====")

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct
})

display(missing_df.head(15))


# =========================================
# MISSING VALUE ANALYSIS (ROW LEVEL) 🚨
# =========================================
print("\n===== MISSING VALUES (ROW LEVEL) =====")

rows_with_nan = df.isna().any(axis=1).sum()
print("Rows with ANY NaN:", rows_with_nan)
print("Total rows:", len(df))
print("Rows WITHOUT NaN:", len(df) - rows_with_nan)


# =========================================
# NUMERICAL vs CATEGORICAL
# =========================================
print("\n===== FEATURE TYPES =====")

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))


# =========================================
# CONSTANT / LOW VARIANCE FEATURES 🚨
# =========================================
print("\n===== CONSTANT / LOW VARIANCE FEATURES =====")

low_var_cols = []

for col in num_cols:
    if df[col].nunique() <= 2:
        low_var_cols.append(col)

print("Low variance columns:", low_var_cols)


# =========================================
# QUICK DATA HEALTH VERDICT
# =========================================
print("\n===== DATA HEALTH SUMMARY =====")

if rows_with_nan > 0.5 * len(df):
    print("🚨 Major issue: Too many rows have NaN → dropna will destroy dataset")
elif rows_with_nan > 0.2 * len(df):
    print("⚠️ Moderate issue: NaNs present → careful preprocessing needed")
else:
    print("✅ Data looks manageable")

if len(low_var_cols) > 0:
    print("⚠️ Low variance features detected → may hurt ML")
else:
    print("✅ No low variance issue")

In [ ]:
# =========================================
# CODE 2: COLUMN CLEANING + VERIFICATION
# =========================================

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)

# Fix specific known issue
df.rename(columns={"Metal-acid_ratio": "Metal_acid_ratio"}, inplace=True)

# =========================================
# VERIFICATION
# =========================================
print("\n===== CLEANED COLUMN NAMES =====")
print(df.columns.tolist())

# Check for duplicate column names
print("\n===== DUPLICATE COLUMN NAMES CHECK =====")
duplicates = df.columns[df.columns.duplicated()].tolist()
print("Duplicate columns:", duplicates)

# Check if any weird characters still exist
print("\n===== COLUMN SANITY CHECK =====")
for col in df.columns:
    if "\t" in col or "  " in col:
        print("⚠️ Issue in column:", col)

# Final shape (should NOT change)
print("\n===== SHAPE CHECK (SHOULD BE SAME) =====")
print(df.shape)

In [ ]:
numeric_cols = [

    "Reaction_temperature_C",
    "metal_1_atomic",
    "metal_2_atomic",
    "metal_3_atomic",
    "metal_4_atomic",
    "Pressure_MPa",
    "H2_CO2_ratio",
    "GHSV_mL_gcat_h",
    "catalyst_mass_g",
    "Cu_Metal_atomic_fraction",
    "Surface_area_m2_g",
    "Crystallite_size_nm",
    "Si/Al_ratio",
    "Pore_volume_cm3_g",
    "Total_acid_sites_mmol_g",
    "Metal_acid_ratio",   # only if you renamed it
    "CO_Selectivity_(%)",
    "DME_Selectivity_(%)",
    "Methanol_Selectivity_(%)",
    "CO2_conversion_efficiency_(%)"
]


In [ ]:
print(df.columns.tolist())

In [ ]:
# =========================================
# CATEGORY DIAGNOSTIC (BEFORE CLEANING)
# =========================================

selected_cat_cols = [
    "acid_component_type",
    "Integration_method",
    "Dimentionality_of_Zeolite"
]

for col in selected_cat_cols:
    print("\n" + "="*50)
    print(f"🔍 COLUMN: {col}")
    print("="*50)

    # Raw unique values
    raw_values = df[col].unique()
    print("\n--- RAW UNIQUE VALUES ---")
    print(raw_values)
    print("Count:", len(raw_values))

    # After strip
    stripped = df[col].astype(str).str.strip()
    print("\n--- AFTER STRIP ---")
    print(stripped.unique())
    print("Count:", stripped.nunique())

    # After upper
    upper = stripped.str.upper()
    print("\n--- AFTER UPPER ---")
    print(upper.unique())
    print("Count:", upper.nunique())

    # Value counts (top 10)
    print("\n--- TOP FREQUENCIES ---")
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
# =========================================
# FIND POTENTIAL DUPLICATES
# =========================================

for col in selected_cat_cols:
    print(f"\n🔍 Possible duplicates in {col}")

    cleaned = df[col].astype(str).str.strip().str.upper()

    mapping = {}

    for original, cleaned_val in zip(df[col], cleaned):
        mapping.setdefault(cleaned_val, set()).add(original)

    for k, v in mapping.items():
        if len(v) > 1:
            print(f"{k} -> {v}")

In [ ]:
# -----------------------------
# FIX NUMERIC FORMATTING (FINAL VERSION)
# -----------------------------
for col in numeric_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(" ", "", regex=False)   # fix "1 250"
        )

        df[col] = pd.to_numeric(df[col], errors="coerce")


# -----------------------------
# VERIFY (OUTSIDE LOOP)
# -----------------------------
print("\n===== NAN COUNT AFTER NUMERIC FIX =====")
print(df[numeric_cols].isna().sum().sort_values(ascending=False).head())

In [ ]:
# =========================================
# CODE 3.1: NUMERIC CLEANING VERIFICATION
# =========================================

print("\n===== NUMERIC CLEANING VERIFICATION =====")

# 1) Check dtypes (all numeric_cols should be numeric)
non_numeric = [col for col in numeric_cols if not pd.api.types.is_numeric_dtype(df[col])]
print("Non-numeric columns after cleaning:", non_numeric)

# 2) Check for any leftover spaces (should be zero)
space_issues = []
for col in numeric_cols:
    if df[col].astype(str).str.contains(" ").any():
        space_issues.append(col)

print("Columns still containing spaces:", space_issues)

# 3) NaN summary (top 10)
print("\nTop NaN counts after numeric fix:")
print(df[numeric_cols].isna().sum().sort_values(ascending=False).head(10))

# 4) Basic stats sanity check (look for weird values)
print("\n===== BASIC STATS CHECK =====")
display(df[numeric_cols].describe().T)

# 5) Extreme values check (optional but useful)
print("\n===== EXTREME VALUE CHECK =====")
for col in numeric_cols:
    if df[col].notna().sum() > 0:
        min_val = df[col].min()
        max_val = df[col].max()
        if min_val < 0:
            print(f"⚠️ Negative values in {col}: min = {min_val}")
        if max_val > 1e7:
            print(f"⚠️ Very large values in {col}: max = {max_val}")

In [ ]:
# # =========================================
# # OUTLIER REMOVAL (MEASURED FEATURES ONLY)
# # =========================================

# outlier_cols = [
#     "Si/Al_ratio",
#     "Total_acid_sites_mmol_g",
#     "Crystallite_size_nm",
#     "Metal_acid_ratio"
# ]

# df_before = df.copy()

# rows_to_remove = set()
# outlier_summary = {}

# for col in outlier_cols:
#     Q1 = df[col].quantile(0.25)
#     Q3 = df[col].quantile(0.75)
#     IQR = Q3 - Q1

#     lower = Q1 - 3 * IQR
#     upper = Q3 + 3 * IQR

#     mask = (df[col] < lower) | (df[col] > upper)

#     flagged_indices = df[mask].index
#     rows_to_remove.update(flagged_indices)

#     outlier_summary[col] = {
#         "rows_flagged": int(mask.sum()),
#         "lower_bound": lower,
#         "upper_bound": upper
#     }

# # Remove rows
# df = df.drop(index=rows_to_remove)

# # Convert summary to DataFrame
# outlier_df = pd.DataFrame(outlier_summary).T

# print("\n===== OUTLIER REMOVAL SUMMARY =====")
# display(outlier_df)

# print("\n===== DATASET SIZE CHANGE =====")
# print("Before:", len(df_before))
# print("After :", len(df))
# print("Rows removed:", len(df_before) - len(df))


# # =========================================
# # EXTRA VERIFICATION
# # =========================================

# print("\n===== % ROWS REMOVED =====")
# removed_pct = (len(df_before) - len(df)) / len(df_before) * 100
# print(f"{removed_pct:.2f}% removed")

# print("\n===== REMAINING NaNs IN THESE COLUMNS =====")
# print(df[outlier_cols].isna().sum())

In [ ]:
# # =========================================
# # PHYSICS-BASED FILTERING (SAFE)
# # =========================================

# df_before = df.copy()

# # Define realistic ranges (based on catalysis knowledge)
# conditions_filter = (
#     (df["Reaction_temperature_C"] >= 150) & (df["Reaction_temperature_C"] <= 400) &
#     (df["Pressure_MPa"] >= 0.1) & (df["Pressure_MPa"] <= 10) &
#     (df["H2_CO2_ratio"] >= 1) & (df["H2_CO2_ratio"] <= 20) &
#     (df["GHSV_mL_gcat_h"] >= 100) & (df["GHSV_mL_gcat_h"] <= 200000)
# )

# df = df[conditions_filter]

# print("\n===== PHYSICAL FILTER APPLIED =====")
# print("Before:", len(df_before))
# print("After :", len(df))
# print("Rows removed:", len(df_before) - len(df))

In [ ]:
# =========================================
# CODE 3: NUMERIC COLUMN CHECK + VERIFICATION
# =========================================

print("\n===== CHECKING NUMERIC COLS =====")

missing_cols = [col for col in numeric_cols if col not in df.columns]
print("Missing columns from df:", missing_cols)


# =========================================
# CHECK DATA TYPES
# =========================================
print("\n===== DATA TYPES OF NUMERIC COLS =====")
for col in numeric_cols:
    print(col, "→", df[col].dtype)


# =========================================
# FIX OBJECT COLUMNS (SAFE CONVERSION)
# =========================================



# =========================================
# VERIFY AFTER CONVERSION
# =========================================
print("\n===== AFTER CONVERSION =====")
print("GHSV dtype:", df["GHSV_mL_gcat_h"].dtype)
print("Metal_acid_ratio dtype:", df["Metal_acid_ratio"].dtype)


# =========================================
# CRITICAL CHECK: ARE YOU DROPPING ROWS HERE?
# =========================================
print("\n===== SHAPE BEFORE ANY DROP =====")
print(df.shape)

# Simulate dangerous step
df_numeric_only = df[numeric_cols]

print("\nShape after selecting numeric_cols:", df_numeric_only.shape)

df_dropna_test = df_numeric_only.dropna()
print("Shape after dropna on numeric_cols:", df_dropna_test.shape)


# =========================================
# FINAL WARNING CHECK
# =========================================
print("\n===== NAN COUNT IN NUMERIC COLS =====")
print(df_numeric_only.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# =========================================
# CODE 4: STANDARDIZE MISSING VALUES
# =========================================

missing_markers = [
    "NR", "nr", "NA", "na", "N/A", "n/a",
    "-", "--", "—", ""
]

# Count NaNs BEFORE
nan_before = df.isna().sum().sum()

df.replace(missing_markers, np.nan, inplace=True)

# Count NaNs AFTER
nan_after = df.isna().sum().sum()

print("\n===== MISSING VALUE STANDARDIZATION =====")
print("NaNs before:", nan_before)
print("NaNs after :", nan_after)


# =========================================
# COLUMN-WISE IMPACT CHECK
# =========================================
print("\n===== TOP COLUMNS WITH NaNs AFTER CLEANING =====")
display(df.isna().sum().sort_values(ascending=False).head(10))


# =========================================
# SHAPE CHECK (SHOULD NOT CHANGE)
# =========================================
print("\n===== SHAPE CHECK =====")
print(df.shape)

In [ ]:
# # -----------------------------
# # CLEAN ALL CATEGORICAL COLUMNS
# # -----------------------------
# selected_cat_cols = [
#     "acid_component_type",
#     "Integration_method",
#     "Dimentionality_of_Zeolite"
# ]

# for col in selected_cat_cols:
#     df[col] = (
#         df[col]
#         .astype(str)
#         .str.strip()
#         .str.upper()
#     )

In [ ]:
print(df["Dimentionality_of_Zeolite"].unique())
print(df["Integration_method"].unique())
print(df["acid_component_type"].unique())  

In [ ]:
essential_cols = [
    "Reaction_temperature_C",
    "Pressure_MPa",
    "H2_CO2_ratio",
    "GHSV_mL_gcat_h"
]

df_before = len(df)

df = df.dropna(subset=essential_cols)

df_after = len(df)

df_before, df_after

In [ ]:
missing_table = (
    pd.DataFrame({
        "missing_count": df[numeric_cols].isna().sum(),
        "missing_percent": df[numeric_cols].isna().sum() / len(df) * 100
    })
    .sort_values("missing_percent", ascending=False)
)
print("\n===== ESSENTIAL FILTER =====")
print("Before:", df_before)
print("After :", df_after)
print("Rows removed:", df_before - df_after)

missing_table


In [ ]:
impute_cols = [
    "Surface_area_m2_g",
    "Crystallite_size_nm",
    "Pore_volume_cm3_g",
    "Total_acid_sites_mmol_g",
    "Si/Al_ratio",
    "Cu_Metal_atomic_fraction",
    "Metal_acid_ratio",
    "catalyst_mass_g",
    "GHSV_mL_gcat_h"
]
X = df[impute_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce")

print("\n===== X SHAPE =====")
print(X.shape)

print("\n===== NaNs in X =====")
print(X.isna().sum().sort_values(ascending=False).head(10))

In [ ]:

# Force numeric conversion (kills '–', stray text, etc.)
df[impute_cols] = df[impute_cols].apply(pd.to_numeric, errors="coerce")
df[impute_cols].info()

missing_markers = ["NA", "NR", "N/A", "na", "nr", "", " ", "-", "--", "—"]
df.replace(missing_markers, np.nan, inplace=True)


X = df[impute_cols].copy()

rng = np.random.RandomState(42)

X_masked = X.copy()
mask = (~X.isna()) & (rng.rand(*X.shape) < 0.1)

X_true = X[mask]
X_masked[mask] = np.nan

print("\n===== SHAPE CHECK =====")
print("X:", X.shape)
print("X_masked:", X_masked.shape)

print("\n===== TOTAL NaNs =====")
print("Original NaNs:", X.isna().sum().sum())
print("After masking:", X_masked.isna().sum().sum())

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
import numpy as np

mean_imputer = SimpleImputer(strategy="mean")
X_mean = mean_imputer.fit_transform(X_masked)

# CORRECT RMSE calculation
y_true = X.values[mask.values]
y_pred = X_mean[mask.values]

mean_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mean_rmse

from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(
    n_neighbors=5,
    weights="distance"
)

X_knn = knn_imputer.fit_transform(X_masked)

y_pred_knn = X_knn[mask.values]

knn_rmse = np.sqrt(mean_squared_error(y_true, y_pred_knn))
knn_rmse

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

mice_imputer = IterativeImputer(
    estimator=ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    max_iter=10,
    random_state=42
)

X_mice = mice_imputer.fit_transform(X_masked)

y_pred_mice = X_mice[mask.values]

mice_rmse = np.sqrt(mean_squared_error(y_true, y_pred_mice))
mice_rmse

import pandas as pd

pd.DataFrame({
    "Method": ["Mean", "KNN", "MICE"],
    "RMSE": [mean_rmse, knn_rmse, mice_rmse]
})

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

final_imputer = IterativeImputer(
    estimator=ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    max_iter=15,
    random_state=42
)

df[impute_cols] = final_imputer.fit_transform(df[impute_cols])
# =========================================
# GLOBAL METAL HANDLING (RUN ONCE)
# =========================================

metal_cols = [
    "metal_1_atomic",
    "metal_2_atomic",
    "metal_3_atomic",
    "metal_4_atomic"
]

# Fill missing metals as 0 (physically: metal not present)
df[metal_cols] = df[metal_cols].fillna(0)

print("\n===== METAL FIX APPLIED =====")
print("Remaining NaNs in metals:")
print(df[metal_cols].isna().sum())

print("\nMetal column stats:")
print(df[metal_cols].describe())

In [ ]:
print("\n===== AFTER FINAL IMPUTATION =====")
print("Shape:", df.shape)

print("\nRemaining NaNs:")
print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
df[impute_cols].isna().sum()


In [ ]:
# =========================================
# PROFILE REPORT (sweetviz - works on Python 3.14+)
# =========================================
!pip install sweetviz

import sweetviz as sv

report = sv.analyze(df)
report.show_html("output(After Impute).html")

In [ ]:
# # @title
# final_path = "/content/drive/MyDrive/CO2 to DME/CO2_to_DME_preprocessed_final.xlsx"
# df.to_excel(final_path, index=False)

# final_path


In [ ]:
# final_path_csv = "/content/drive/MyDrive/CO2 to DME/CO2_to_DME_preprocessed_final.csv"
# df.to_csv(final_path_csv, index=False)

# final_path_csv


In [ ]:
# #Outlier removal

# outlier_features = [
#     "Reaction_temperature_C",
#     "Pressure_MPa",
#     "H2_CO2_ratio",
#     "GHSV_mL_gcat_h",
#     "Surface_area_m2_g",
#     "Crystallite_size_nm",
#     "Pore_volume_cm3_g",
#     "Total_acid_sites_mmol_g",
#     "Si/Al_ratio",
#     "Cu_Metal_atomic_fraction",
#     "Metal_acid_ratio",
#     "catalyst_mass_g"
# ]

# # Remove spaces inside numeric strings and coerce to float
# df[outlier_features] = (
#     df[outlier_features]
#     .astype(str)
#     .apply(lambda col: col.str.replace(" ", "", regex=False))
#     .apply(pd.to_numeric, errors="coerce")
# )


# from sklearn.preprocessing import StandardScaler

# X_outlier = df[outlier_features].copy()

# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X_outlier)

# from sklearn.ensemble import IsolationForest

# iso = IsolationForest(
#     n_estimators=300,
#     contamination=0.05,
#     random_state=42
# )

# df["outlier_flag"] = iso.fit_predict(X_scaled)

# # Count the number of outliers (-1)
# num_outliers = df[df["outlier_flag"] == -1].shape[0]
# print(f"Number of outliers: {num_outliers}")

# df["outlier_flag"].value_counts()

# # Keep the flag for later analysis
# df["outlier_flag"] = df["outlier_flag"]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap

# ── Define features ───────────────────────────────────────────────────────────
heatmap_features = [
    "Reaction_temperature_C",
    "metal_1_atomic",
    "metal_2_atomic",
    "metal_3_atomic",
    "metal_4_atomic",
    "Pressure_MPa",
    "H2_CO2_ratio",
    "GHSV_mL_gcat_h",
    "Surface_area_m2_g",
    "Crystallite_size_nm",
    "Pore_volume_cm3_g",
    "Total_acid_sites_mmol_g",
    "Si/Al_ratio",
    "Cu_Metal_atomic_fraction",
    "Metal_acid_ratio",
    "catalyst_mass_g",
    "CO2_conversion_efficiency_(%)",
    "DME_Selectivity_(%)",
    "Methanol_Selectivity_(%)",
    "CO_Selectivity_(%)"
]

# ── Custom diverging colormap: deep blue → white → deep red ─────────────────
colors_list = [
    "#08306b",  # deep navy
    "#2171b5",
    "#6baed6",
    "#c6dbef",
    "#ffffff",  # white center
    "#fcbba1",
    "#cb181d",
    "#67000d",  # deep crimson
]
cmap = LinearSegmentedColormap.from_list("research_div", colors_list, N=512)

# ── Compute correlation & mask upper triangle ────────────────────────────────
corr = df[heatmap_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
corr_masked = np.where(mask, np.nan, corr.values)

n      = len(corr.columns)
labels = corr.columns.tolist()

# ── Figure setup ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 15))
fig.patch.set_facecolor("#ffffff")
ax.set_facecolor("#ffffff")

# ── Draw heatmap ─────────────────────────────────────────────────────────────
im = ax.imshow(
    corr_masked,
    aspect="auto",
    cmap=cmap,
    vmin=-1,
    vmax=1,
    interpolation="nearest",
)

# ── Cell border lines ────────────────────────────────────────────────────────
for v in np.arange(-0.5, n, 1):
    ax.axhline(v, color="#cccccc", linewidth=0.4, zorder=2)
    ax.axvline(v, color="#cccccc", linewidth=0.4, zorder=2)

# ── Annotate each cell ───────────────────────────────────────────────────────
for i in range(n):
    for j in range(n):
        val = corr.iloc[i, j]
        if not mask[i, j] and not np.isnan(val):
            abs_val = abs(val)
            # Dark text on light cells, white on saturated cells
            if abs_val > 0.55:
                text_color = "white"
            else:
                text_color = "#1a1a2e"
            fontweight = "bold" if abs_val > 0.6 else "normal"
            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=7.5,
                color=text_color,
                fontweight=fontweight,
                family="monospace",
            )

# ── Colorbar ─────────────────────────────────────────────────────────────────
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, aspect=30)
cbar.ax.yaxis.set_tick_params(color="#222222", labelsize=10)
cbar.outline.set_edgecolor("#aaaaaa")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#222222", family="monospace")
cbar.set_label("Pearson r", color="#333333", fontsize=11, labelpad=12)
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar.set_ticklabels(["-1.0", "-0.5", "0.0", "+0.5", "+1.0"])

# ── Axis ticks & labels ──────────────────────────────────────────────────────
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9.5,
                   color="#222222", family="sans-serif")
ax.set_yticklabels(labels, fontsize=9.5, color="#222222", family="sans-serif")
ax.tick_params(axis="both", which="both", length=0, pad=6)

# ── Black border around entire heatmap ──────────────────────────────────────
for spine in ax.spines.values():
    spine.set_edgecolor("#000000")
    spine.set_linewidth(1.5)

# ── Title ────────────────────────────────────────────────────────────────────
ax.set_title(
    "Correlation Heatmap — Reaction Conditions, Catalyst Properties & Outputs",
    fontsize=14,
    color="#111111",
    fontweight="bold",
    pad=18,
    loc="center",
)

fig.text(
    0.5, 0.01,
    "Lower triangle shown  •  Pearson correlation  •  Values ≥ |0.6| bolded",
    ha="center",
    fontsize=9,
    color="#666666",
    style="italic",
)

plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.show()

In [ ]:
print("\n===== HEATMAP FEATURE CHECK =====")
missing_cols = [col for col in heatmap_features if col not in df.columns]
print("Missing columns:", missing_cols)

print("\nShape used for correlation:", df[heatmap_features].shape)

In [ ]:
target_cols = [
    "CO2_conversion_efficiency_(%)",
    "DME_Selectivity_(%)",
    "Methanol_Selectivity_(%)",
    "CO_Selectivity_(%)"
]
numeric_features = [
    "Reaction_temperature_C",
    "metal_1_atomic",
    "metal_2_atomic",
    "metal_3_atomic",
    "metal_4_atomic",
    "Pressure_MPa",
    "H2_CO2_ratio",
    "GHSV_mL_gcat_h",
    "Surface_area_m2_g",
    "Crystallite_size_nm",
    "Pore_volume_cm3_g",
    "Total_acid_sites_mmol_g",
    "Si/Al_ratio",
    "Cu_Metal_atomic_fraction",
    "Metal_acid_ratio",
    "catalyst_mass_g"
]
categorical_features = [
    "Integration_method",
    "acid_component_type",
    "Dimentionality_of_Zeolite"
]

df[numeric_features + categorical_features + target_cols].isna().sum()

In [ ]:
print("\n===== CURRENT DATASET STATUS =====")
print("Total rows:", len(df))

print("\nNaNs in targets:")
print(df[target_cols].isna().sum())

print("\nNaNs in features:")
print(df[numeric_features + categorical_features].isna().sum().sum())

In [ ]:
# ========================================
# PREPROCESSOR PIPELINE (CORRECTED)
# ========================================

import numpy as np

# ---------------------------------------
# Normalize textual missing markers
# ---------------------------------------
missing_markers = [
    "NA", "na", "NR", "nr", "N/A", "n/a",
    "-", "--", "—", ""
]

df.replace(missing_markers, np.nan, inplace=True)

# ---------------------------------------
# Fix categorical columns (IMPORTANT)
# ---------------------------------------
for col in categorical_features:
    # Fix previous mistake (string "nan")
    df[col] = df[col].replace("nan", np.nan)

    # Proper handling
    df[col] = df[col].fillna("Unknown").astype(str)

# ---------------------------------------
# SELECT TARGET (IMPORTANT)
# ---------------------------------------
target = "CO2_conversion_efficiency_(%)"   # change later as needed

df_model = df[df[target].notna()].copy()

# ---------------------------------------
# DEFINE X and y
# ---------------------------------------
X = df_model[numeric_features + categorical_features]
y = df_model[target]

# ---------------------------------------
# PIPELINE
# ---------------------------------------
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

In [ ]:
print("\n===== VERIFICATION =====")

# 1. Check shape (NO collapse)
print("\nShape check:")
print("X:", X.shape)
print("y:", y.shape)

# 2. Check target NaNs (must be 0)
print("\nTarget NaNs:")
print(y.isna().sum())

# 3. Check categorical 'nan' issue fixed
print("\nCategorical 'nan' string check:")
for col in categorical_features:
    print(col, (X[col] == "nan").sum())

# 4. Check real NaNs in categorical (should be 0)
print("\nCategorical actual NaNs:")
print(X[categorical_features].isna().sum())

# 5. Check numeric NaNs (allowed)
print("\nNumeric NaNs (should exist but OK):")
print(X[numeric_features].isna().sum().sum())

# 6. FINAL sanity
print("\nFinal row count:", len(X))

In [ ]:
# # =========================================
# # EXPORT CLEAN DATA FOR REVIEW
# # =========================================

# # Save full cleaned dataset
# save_path = "/content/drive/MyDrive/CO2 to DME/cleaned_dataset_v1.csv"

# df.to_csv(save_path, index=False)

# print("✅ Full dataset saved at:", save_path)


# # =========================================
# # ALSO SAVE MODEL-READY DATA (IMPORTANT)
# # =========================================

# model_save_path = "/content/drive/MyDrive/CO2 to DME/model_dataset_v1.csv"

# df_model.to_csv(model_save_path, index=False)

# print("✅ Model dataset saved at:", model_save_path)


# # =========================================
# # OPTIONAL: SAVE X and y separately
# # =========================================

# X.to_csv("/content/drive/MyDrive/CO2 to DME/X_features_v1.csv", index=False)
# y.to_csv("/content/drive/MyDrive/CO2 to DME/y_target_v1.csv", index=False)

# print("✅ X and y saved separately")

In [ ]:
# Choose target variable
target = "CO2_conversion_efficiency_(%)"

# Keep only rows where this target is available
# Only filter based on target
data_model = df[df[target].notna()].copy()


# Define X and y
X_model = data_model[numeric_features + categorical_features]
y_model = data_model[target]

# Split inputs and output
X_model = data_model[numeric_features + categorical_features]
y_model = data_model[target]

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Baseline Random Forest (no tuning yet)
baseline_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Full pipeline = preprocessing + model
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", baseline_model)
])
# Train the pipeline
model_pipeline.fit(X_train, y_train)



In [ ]:
print("\n===== FINAL MODEL DATA CHECK =====")
print("Rows:", len(X_model))
print("Target NaNs:", y_model.isna().sum())
print("Feature NaNs:", X_model.isna().sum().sum())

In [ ]:
# ============================================================
# Validation — Baseline RF (CO2_conversion_efficiency_(%))
# ============================================================

from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# ------------------------------------------------------------
# 1. Generate predictions FIRST (this was missing)
# ------------------------------------------------------------
y_train_pred = model_pipeline.predict(X_train)
y_val_pred = model_pipeline.predict(X_val)

# ------------------------------------------------------------
# 2. Compute R2 scores
# ------------------------------------------------------------
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

# ------------------------------------------------------------
# 3. Compute RMSE
# ------------------------------------------------------------
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

# ------------------------------------------------------------
# 4. Print results
# ------------------------------------------------------------
print("Train R2:", train_r2)
print("Validation R2:", val_r2)
print("Train RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)
print("\n===== OVERFITTING GAP =====")
print("R2 Gap:", train_r2 - val_r2)
print("RMSE Gap:", val_rmse - train_rmse)

In [ ]:
print("\n===== FINAL DATA USAGE CHECK =====")

# Total available rows after preprocessing
print("Total df_model rows:", len(df_model))

# Rows actually used in modeling
print("X_model rows:", len(X_model))

# Train + Val split check
print("Train rows:", len(X_train))
print("Val rows:", len(X_val))
print("Train + Val:", len(X_train) + len(X_val))


# Check consistency
if len(X_model) == len(df_model):
    print("\n✅ All available rows are used for modeling")
else:
    print("\n❌ Some rows are lost before modeling")

if len(X_train) + len(X_val) == len(X_model):
    print("✅ Train/Val split is correct")
else:
    print("❌ Data lost during split")


# Extra safety: check if any drop happened silently
print("\n===== SAFETY CHECK =====")
print("Any NaNs in X_model:", X_model.isna().sum().sum())
print("Any NaNs in y_model:", y_model.isna().sum())

In [ ]:
# ============================
# K-Fold RF — CO2 Conversion (CORRECTED)
# ============================

from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

target = "CO2_conversion_efficiency_(%)"

# ✅ ONLY filter target (NO dropna on features)
data_co2 = df[df[target].notna()].copy()

X_co2 = data_co2[numeric_features + categorical_features]
y_co2 = data_co2[target]

print("\n===== CV DATA CHECK =====")
print("Rows:", len(X_co2))
print("NaNs in X:", X_co2.isna().sum().sum())
print("NaNs in y:", y_co2.isna().sum())


# Model
rf_co2 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline_co2 = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_co2)
])

# K-Fold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

cv_r2_scores_co2 = cross_val_score(
    pipeline_co2,
    X_co2,
    y_co2,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("\n===== CV RESULTS =====")
print("Scores:", cv_r2_scores_co2)
print("Mean R2:", cv_r2_scores_co2.mean())
print("Std:", cv_r2_scores_co2.std())

In [ ]:
# =========================================
# RF - CO2_conversion_efficiency_(%) Graph
# PUBLICATION EDITION — LIGHT THEME
# =========================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.ticker import AutoMinorLocator
import matplotlib.patches as mpatches
from sklearn.model_selection import learning_curve, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats

# ── Typography & palette (journal-grade) ─────────────────────────────────────
plt.rcParams.update({
    "font.family":       "DejaVu Serif",
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "axes.titleweight":  "bold",
    "figure.dpi":        150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"   # steel blue  — training
C2       = "#c0392b"   # brick red   — CV / error
C3       = "#1e8449"   # forest green— positive residuals / reference
C4       = "#7d3c98"   # purple      — annotations
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ╔══════════════════════════════════════════╗
# ║  1. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline_co2, X_co2, y_co2,
    cv=kf,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

# Gap annotation
gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.78, val_rmse[-1] * 1.05),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — Random Forest  |  CO₂ Conversion Efficiency")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. CV R² BOXPLOT                        ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(6, 5.5))
fig.patch.set_facecolor(BG)
style_ax(ax, minor_ticks=False)

bp = ax.boxplot(
    cv_r2_scores_co2,
    patch_artist=True,
    widths=0.38,
    medianprops=dict(color=C2, linewidth=2.8),
    whiskerprops=dict(color=C1, linewidth=1.6, linestyle="--"),
    capprops=dict(color=C1, linewidth=2.2),
    flierprops=dict(marker="D", color=C4, markersize=5,
                    markerfacecolor=C4, alpha=0.8),
    boxprops=dict(linewidth=1.6),
)
bp["boxes"][0].set_facecolor("#dbeeff")
bp["boxes"][0].set_edgecolor(C1)

np.random.seed(42)
jitter = np.random.normal(1, 0.045, size=len(cv_r2_scores_co2))
ax.scatter(jitter, cv_r2_scores_co2, color=C1, alpha=0.65,
           s=45, zorder=4, label="Individual fold scores",
           edgecolors="white", linewidths=0.5)

mean_r2 = np.mean(cv_r2_scores_co2)
std_r2  = np.std(cv_r2_scores_co2)
ax.axhline(mean_r2, color=C2, linewidth=1.8, linestyle="-.",
           zorder=3, label=f"Mean R² = {mean_r2:.3f}")

# Stats text box
stats_txt = (f"Mean  = {mean_r2:.3f}\n"
             f"Std    = {std_r2:.3f}\n"
             f"Min   = {min(cv_r2_scores_co2):.3f}\n"
             f"Max  = {max(cv_r2_scores_co2):.3f}")
ax.text(1.32, mean_r2, stats_txt,
        fontsize=8.5, color=TEXT_COL, family="monospace",
        va="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff9e6",
                  edgecolor="#ccaa00", alpha=0.95))

ax.set_ylabel("R² Score")
ax.set_title("10-Fold Cross-Validation R²  |  CO₂ Conversion Efficiency")
ax.set_xticks([])
ax.set_xlim(0.5, 1.8)
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Each point represents one fold  •  Dashed line = median  •  "
         "Dash-dot line = mean",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "boxplot_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  3. OOF PREDICTIONS                      ║
# ╚══════════════════════════════════════════╝

y_oof_pred = cross_val_predict(pipeline_co2, X_co2, y_co2, cv=kf, n_jobs=-1)
residuals  = np.array(y_co2) - y_oof_pred


# ╔══════════════════════════════════════════╗
# ║  4. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(np.array(y_co2).min(), y_oof_pred.min())
max_val = max(np.array(y_co2).max(), y_oof_pred.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_co2, y_oof_pred])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    np.array(y_co2)[idx], y_oof_pred[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

# Perfect fit
ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

# ±10% envelope
band = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band, max_val - band],
                [min_val + band, max_val + band],
                color=C2, alpha=0.07, label="±10% error band")

# ±5% envelope
band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

# Metrics box
r2   = r2_score(y_co2, y_oof_pred)
rmse = np.sqrt(np.mean(residuals**2))
mae  = mean_absolute_error(y_co2, y_oof_pred)
metrics_txt = (f"R²   = {r2:.4f}\n"
               f"RMSE = {rmse:.3f}\n"
               f"MAE  = {mae:.3f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual CO₂ Conversion Efficiency (%)")
ax.set_ylabel("Predicted CO₂ Conversion Efficiency (%)")
ax.set_title("Parity Plot — Out-of-Fold CV Predictions  |  CO₂ Conversion")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  Predictions from 10-fold CV",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. RESIDUAL PLOT                        ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals >= 0
ax.scatter(y_oof_pred[pos],  residuals[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_oof_pred[~pos], residuals[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals.std()
ax.axhspan(-std_r, std_r, color=C1, alpha=0.07,
           label=f"±1σ band  (σ = {std_r:.2f})")
ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
           label=f"±2σ band  (2σ = {2*std_r:.2f})")

# Annotate std lines
for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#888888",
                   linewidth=0.8, linestyle=ls, alpha=0.7)
        ax.text(ax.get_xlim()[1] if ax.get_xlim()[1] != 0 else y_oof_pred.max(),
                sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5,
                color="#666666")

ax.set_xlabel("Predicted CO₂ Conversion Efficiency (%)")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — CV Predictions  |  CO₂ Conversion Efficiency")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "residual_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  6. RESIDUAL DISTRIBUTION (FIXED BARS)   ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

# Gradient by position (left=blue, right=red) — never goes white
bin_centers  = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos     = (bin_centers - bin_centers.min()) / (bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap     = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf",   # deep blue (left / under-prediction)
        "#5ba3d0",   # mid blue
        "#a8d1e7",   # light blue
        "#f4a97f",   # light orange
        "#e05c3a",   # mid red
        "#c0392b",   # deep red  (right / over-prediction)
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

# KDE
kde_x     = np.linspace(residuals.min(), residuals.max(), 400)
kde_y     = stats.gaussian_kde(residuals)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals) * bin_width
ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

# Normal reference
mu, sigma = residuals.mean(), residuals.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

# Shapiro-Wilk
from scipy.stats import shapiro
sw_stat, sw_p = shapiro(residuals)

ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

# Stats box
stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — CV Predictions  |  CO₂ Conversion Efficiency")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "residual_dist_co2.png")
plt.show()

# ╔══════════════════════════════════════════╗
# ║  7. DIAGNOSTICS SUMMARY                  ║
# ╚══════════════════════════════════════════╝

corr_val = np.corrcoef(y_co2, y_oof_pred)[0, 1]
rmse_oof = np.sqrt(np.mean(residuals ** 2))
mae_oof  = mean_absolute_error(y_co2, y_oof_pred)
r2_oof   = r2_score(y_co2, y_oof_pred)

print("\n" + "═" * 46)
print("   CV DIAGNOSTICS  —  CO₂ Conversion Efficiency")
print("═" * 46)
print(f"  Pearson r              : {corr_val:.4f}")
print(f"  OOF R²                 : {r2_oof:.4f}")
print(f"  OOF RMSE               : {rmse_oof:.4f}")
print(f"  OOF MAE                : {mae_oof:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals.mean():.4f}")
print(f"  Residual Std           : {residuals.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 46)

In [ ]:
# ============================
# RF — DME Selectivity (FIXED)
# ============================

target = "DME_Selectivity_(%)"

# ✅ ONLY filter target
data_model = df[df[target].notna()].copy()

X_model = data_model[numeric_features + categorical_features]
y_model = data_model[target]

print("Samples used:", len(y_model))


# ============================
# SPLIT
# ============================

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42
)


# ============================
# MODEL
# ============================

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

pipeline.fit(X_train, y_train)


# ============================
# VERIFICATION (MANDATORY)
# ============================

print("\n===== DATA CHECK =====")
print("Total rows:", len(X_model))
print("Train + Val:", len(X_train) + len(X_val))
print("NaNs in X:", X_model.isna().sum().sum())
print("NaNs in y:", y_model.isna().sum())

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

y_train_pred = pipeline.predict(X_train)
y_val_pred = pipeline.predict(X_val)

train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print("Train R2:", train_r2)
print("Validation R2:", val_r2)
print("Train RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)

print("\n===== OVERFITTING GAP =====")
print("R2 Gap:", train_r2 - val_r2)
print("RMSE Gap:", val_rmse - train_rmse)

In [ ]:
# ============================
# K-Fold RF — DME Selectivity (FIXED)
# ============================

from sklearn.model_selection import cross_val_score

target = "DME_Selectivity_(%)"

# ✅ ONLY filter target
data_dme = df[df[target].notna()].copy()

X_dme = data_dme[numeric_features + categorical_features]
y_dme = data_dme[target]

print("\n===== CV DATA CHECK =====")
print("Rows:", len(X_dme))
print("NaNs in X:", X_dme.isna().sum().sum())
print("NaNs in y:", y_dme.isna().sum())


rf_dme = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline_dme = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_dme)
])

cv_r2_scores_dme = cross_val_score(
    pipeline_dme,
    X_dme,
    y_dme,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("\n===== CV RESULTS =====")
print("Scores:", cv_r2_scores_dme)
print("Mean R2:", cv_r2_scores_dme.mean())
print("Std:", cv_r2_scores_dme.std())

In [ ]:
# ==============================
# RF — DME Selectivity
# PUBLICATION EDITION — LIGHT THEME
# ==============================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from sklearn.model_selection import KFold, learning_curve, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats
from scipy.stats import shapiro

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"   # steel blue
C2       = "#c0392b"   # brick red
C3       = "#1e8449"   # forest green
C4       = "#7d3c98"   # purple
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

# ── CV Setup ──────────────────────────────────────────────────────────────────
kf = KFold(n_splits=10, shuffle=True, random_state=42)


# ╔══════════════════════════════════════════╗
# ║  1. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline, X_model, y_model,
    cv=kf,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.78, val_rmse[-1] * 1.05),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — Random Forest  |  DME Selectivity")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_dme.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. OOF PREDICTIONS                      ║
# ╚══════════════════════════════════════════╝

y_pred_all = cross_val_predict(pipeline, X_model, y_model, cv=kf, n_jobs=-1)
y_true_all = y_model.values
residuals  = y_true_all - y_pred_all


# ╔══════════════════════════════════════════╗
# ║  3. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(y_true_all.min(), y_pred_all.min())
max_val = max(y_true_all.max(), y_pred_all.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_true_all, y_pred_all])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    y_true_all[idx], y_pred_all[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

band10 = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band10, max_val - band10],
                [min_val + band10, max_val + band10],
                color=C2, alpha=0.07, label="±10% error band")

band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

r2   = r2_score(y_true_all, y_pred_all)
rmse = np.sqrt(np.mean(residuals**2))
mae  = mean_absolute_error(y_true_all, y_pred_all)

metrics_txt = (f"R²   = {r2:.4f}\n"
               f"RMSE = {rmse:.3f}\n"
               f"MAE  = {mae:.3f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual DME Selectivity (%)")
ax.set_ylabel("Predicted DME Selectivity (%)")
ax.set_title("Parity Plot — Out-of-Fold CV Predictions  |  DME Selectivity")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  Predictions from 10-fold CV",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_dme.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. RESIDUAL DISTRIBUTION                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

# Left→right positional gradient (always visible, never white)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos    = (bin_centers - bin_centers.min()) / (
               bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap    = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf",  # deep blue
        "#5ba3d0",  # mid blue
        "#a8d1e7",  # light blue
        "#f4a97f",  # light orange
        "#e05c3a",  # mid red
        "#c0392b",  # deep red
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

kde_x     = np.linspace(residuals.min(), residuals.max(), 400)
kde_y     = stats.gaussian_kde(residuals)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals) * bin_width

ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

mu, sigma = residuals.mean(), residuals.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

sw_stat, sw_p = shapiro(residuals)
ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — CV Predictions  |  DME Selectivity")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "residual_dist_dme.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. RESIDUAL VS PREDICTED                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals >= 0
ax.scatter(y_pred_all[pos],  residuals[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_pred_all[~pos], residuals[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals.std()
ax.axhspan(-std_r,    std_r,    color=C1, alpha=0.07,
           label=f"±1σ  ({std_r:.2f})")
ax.axhspan(-2*std_r,  2*std_r,  color=C1, alpha=0.04,
           label=f"±2σ  ({2*std_r:.2f})")

for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#aaaaaa",
                   linewidth=0.8, linestyle=ls, alpha=0.8)
        ax.text(y_pred_all.max(), sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5, color="#666666")

ax.set_xlabel("Predicted DME Selectivity (%)")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — CV Predictions  |  DME Selectivity")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "residual_vs_pred_dme.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  6. DIAGNOSTICS SUMMARY                  ║
# ╚══════════════════════════════════════════╝

corr = np.corrcoef(y_true_all, y_pred_all)[0, 1]

print("\n" + "═" * 46)
print("   CV DIAGNOSTICS  —  DME Selectivity")
print("═" * 46)
print(f"  Pearson r              : {corr:.4f}")
print(f"  OOF R²                 : {r2:.4f}")
print(f"  OOF RMSE               : {rmse:.4f}")
print(f"  OOF MAE                : {mae:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals.mean():.4f}")
print(f"  Residual Std           : {residuals.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 46)

In [ ]:
# -------------------------------
# Target definition - RF Methanol selectivity (FIXED)
# -------------------------------

target = "Methanol_Selectivity_(%)"

# ✅ ONLY filter target
data_model = df[df[target].notna()].copy()

# Separate inputs and output
X_model = data_model[numeric_features + categorical_features]
y_model = data_model[target]

print("Number of samples used:", len(y_model))


# -------------------------------
# Train-test split
# -------------------------------
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42
)


# -------------------------------
# Model
# -------------------------------
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

pipeline.fit(X_train, y_train)


# -------------------------------
# VERIFICATION (MANDATORY)
# -------------------------------
print("\n===== DATA CHECK =====")
print("Total rows:", len(X_model))
print("Train + Val:", len(X_train) + len(X_val))
print("NaNs in X:", X_model.isna().sum().sum())
print("NaNs in y:", y_model.isna().sum())

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Predictions
y_train_pred = pipeline.predict(X_train)
y_val_pred = pipeline.predict(X_val)

# R2 scores
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

# RMSE
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print("Train R2:", train_r2)
print("Validation R2:", val_r2)
print("Train RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)
print("\n===== OVERFITTING GAP =====")
print("R2 Gap:", train_r2 - val_r2)
print("RMSE Gap:", val_rmse - train_rmse)

In [ ]:
# =================================
# K-Fold RF — Methanol Selectivity (FIXED)
# =================================

from sklearn.model_selection import cross_val_score

target = "Methanol_Selectivity_(%)"

# ✅ ONLY filter target
data_meoh = df[df[target].notna()].copy()

X_meoh = data_meoh[numeric_features + categorical_features]
y_meoh = data_meoh[target]

print("\n===== CV DATA CHECK =====")
print("Rows:", len(X_meoh))
print("NaNs in X:", X_meoh.isna().sum().sum())
print("NaNs in y:", y_meoh.isna().sum())


rf_meoh = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline_meoh = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_meoh)
])

cv_r2_scores_meoh = cross_val_score(
    pipeline_meoh,
    X_meoh,
    y_meoh,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("\n===== CV RESULTS =====")
print("Scores:", cv_r2_scores_meoh)
print("Mean R2:", cv_r2_scores_meoh.mean())
print("Std:", cv_r2_scores_meoh.std())

In [ ]:
# -------------------------------
# Target definition - CO selectivity (FIXED)
# -------------------------------
target = "CO_Selectivity_(%)"

# ✅ ONLY filter target
data_model = df[df[target].notna()].copy()

# Inputs and output
X_model = data_model[numeric_features + categorical_features]
y_model = data_model[target]

print("Samples used:", len(y_model))


# -------------------------------
# Split
# -------------------------------
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42
)


# -------------------------------
# Model
# -------------------------------
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

pipeline.fit(X_train, y_train)


# -------------------------------
# VERIFICATION
# -------------------------------
print("\n===== DATA CHECK =====")
print("Total rows:", len(X_model))
print("Train + Val:", len(X_train) + len(X_val))
print("NaNs in X:", X_model.isna().sum().sum())
print("NaNs in y:", y_model.isna().sum())

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Predictions
y_train_pred = pipeline.predict(X_train)
y_val_pred = pipeline.predict(X_val)

# Metrics
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print("Train R2:", train_r2)
print("Validation R2:", val_r2)
print("Train RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)
print("\n===== OVERFITTING GAP =====")
print("R2 Gap:", train_r2 - val_r2)
print("RMSE Gap:", val_rmse - train_rmse)

In [ ]:
# ============================
# K-Fold RF — CO Selectivity (FIXED)
# ============================

from sklearn.model_selection import cross_val_score

target = "CO_Selectivity_(%)"

# ✅ ONLY filter target
data_co = df[df[target].notna()].copy()

X_co = data_co[numeric_features + categorical_features]
y_co = data_co[target]

print("\n===== CV DATA CHECK =====")
print("Rows:", len(X_co))
print("NaNs in X:", X_co.isna().sum().sum())
print("NaNs in y:", y_co.isna().sum())


rf_co = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline_co = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_co)
])

cv_r2_scores_co = cross_val_score(
    pipeline_co,
    X_co,
    y_co,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("\n===== CV RESULTS =====")
print("Scores:", cv_r2_scores_co)
print("Mean R2:", cv_r2_scores_co.mean())
print("Std:", cv_r2_scores_co.std())

In [ ]:
# =========================================
# FULL SANITY CHECK BEFORE PERMUTATION
# =========================================

print("\n===== SHAPE CHECK =====")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)


# -----------------------------------------
# NaN check
# -----------------------------------------
print("\n===== NaN CHECK =====")
print("NaNs in X_train:", X_train.isna().sum().sum())
print("NaNs in X_val  :", X_val.isna().sum().sum())
print("NaNs in y_train:", y_train.isna().sum())
print("NaNs in y_val  :", y_val.isna().sum())


# -----------------------------------------
# Categorical column check
# -----------------------------------------
print("\n===== CATEGORICAL CHECK =====")

for col in categorical_features:
    print(f"\n{col}:")
    print("Unique values:", X_train[col].nunique())
    print("Sample values:", X_train[col].unique()[:5])


# -----------------------------------------
# Check encoding output shape
# -----------------------------------------
print("\n===== ENCODING CHECK =====")

X_transformed = preprocessor.fit_transform(X_train)

print("Original feature count:", X_train.shape[1])
print("Transformed feature count:", X_transformed.shape[1])


# -----------------------------------------
# Feature name expansion (VERY IMPORTANT)
# -----------------------------------------
try:
    feature_names = preprocessor.get_feature_names_out()
    print("\nTotal encoded features:", len(feature_names))
    print("Sample encoded features:", feature_names[:10])
except:
    print("\n⚠️ Could not extract feature names")


# -----------------------------------------
# Pipeline consistency check
# -----------------------------------------
print("\n===== PIPELINE CHECK =====")

test_pred = pipeline.predict(X_val[:5])
print("Prediction sample:", test_pred)

In [ ]:
# =========================================
# Permutation Importance — CO2 Conversion (FIXED & ROBUST)
# =========================================

from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

# ---------------------------
# 1. Baseline performance
# ---------------------------
baseline_pred = pipeline.predict(X_val)
baseline_r2 = r2_score(y_val, baseline_pred)

print("Baseline R2:", baseline_r2)


# ---------------------------
# 2. Permutation importance
# ---------------------------
n_repeats = 5   # IMPORTANT → reduces noise
rng = np.random.RandomState(42)

feature_importance = []

for col in X_val.columns:

    scores = []

    for _ in range(n_repeats):
        X_permuted = X_val.copy()

        X_permuted[col] = rng.permutation(X_permuted[col].values)

        permuted_pred = pipeline.predict(X_permuted)
        permuted_r2 = r2_score(y_val, permuted_pred)

        scores.append(baseline_r2 - permuted_r2)

    # average importance
    importance_mean = np.mean(scores)

    feature_importance.append((col, importance_mean))


# ---------------------------
# 3. Convert to DataFrame
# ---------------------------
perm_importance_df = pd.DataFrame(
    feature_importance,
    columns=["feature", "importance"]
).sort_values("importance", ascending=False)


# ---------------------------
# 4. Output
# ---------------------------
print("\n===== TOP FEATURES (Permutation Importance) =====")
print(perm_importance_df.head(10))


# ---------------------------
# 5. SANITY CHECK
# ---------------------------
print("\n===== CHECK =====")
print("Any negative importance (noise expected):", (perm_importance_df["importance"] < 0).sum())
print("Total features:", len(perm_importance_df))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

# ── Shared style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax_h(ax):
    """Horizontal bar chart style — grid on x only."""
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.set_axisbelow(True)
    ax.grid(True,  axis="x", color=GRID_COL,
            linewidth=0.7, linestyle="--", alpha=1.0)
    ax.grid(False, axis="y")
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which="minor", length=3, color="#888888", direction="in")
    ax.tick_params(which="major", length=5, color="#333333", direction="in")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

# ── Prep data ─────────────────────────────────────────────────────────────────
top = perm_importance_df.head(10).copy()
top = top.sort_values("importance", ascending=True)

features   = top["feature"].values
importance = top["importance"].values

# Gradient: light blue → deep blue (positional)
norm       = (importance - importance.min()) / (importance.max() - importance.min() + 1e-9)
cmap       = plt.cm.get_cmap("Blues")
bar_colors = [cmap(0.35 + 0.55 * n) for n in norm]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6.5))
fig.patch.set_facecolor(BG)
style_ax_h(ax)

bars = ax.barh(
    features, importance,
    color=bar_colors,
    edgecolor="white",
    linewidth=0.7,
    height=0.6,
    zorder=3,
)

# Value labels
for bar, val in zip(bars, importance):
    ax.text(
        bar.get_width() + importance.max() * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center", ha="left",
        color=TEXT_COL, fontsize=8.5, family="monospace"
    )

# Top feature highlight — black border
ax.barh(
    [features[-1]], [importance[-1]],
    color="none",
    edgecolor="#000000",
    linewidth=2.0,
    height=0.6,
    zorder=4,
)

# Rank badges on left of each bar
for i, (feat, val) in enumerate(zip(features, importance)):
    rank = len(features) - i
    ax.text(
        -importance.max() * 0.015, i,
        f"#{rank}",
        va="center", ha="right",
        fontsize=7.5, color="#888888", family="monospace"
    )

ax.axvline(0, color="#333333", linewidth=1.2, zorder=2)

# Top feature callout
top_feat = features[-1]
top_val  = importance[-1]
# Top feature callout — bottom right, never overlaps bars
ax.annotate(
    f"Most important:\n{top_feat}",
    xy=(top_val, len(features) - 1),
    xytext=(importance.max() * 0.75, 1.2),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.3,
                    connectionstyle="arc3,rad=0.3"),
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.9)
)
ax.set_xlabel("Permutation Importance  (ΔR²)")
ax.set_ylabel("Feature")
ax.set_title("Feature Importance — Random Forest  |  CO₂ Conversion Efficiency")
ax.set_xlim(left=-importance.max() * 0.06,
            right=importance.max() * 1.22)

fig.text(0.5, -0.02,
         "Permutation importance = mean decrease in R² when feature values are randomly shuffled",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "feature_importance_co2.png")
plt.show()

In [ ]:
# =========================================
# STEP 0: Prepare DME dataset (CORRECT WAY)
# =========================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

target = "DME_Selectivity_(%)"

# ✅ ONLY filter target
data_model = df[df[target].notna()].copy()

X_model = data_model[numeric_features + categorical_features]
y_model = data_model[target]

# Split
X_train_dme, X_val_dme, y_train_dme, y_val_dme = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42
)

# Model
pipeline_dme = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

pipeline_dme.fit(X_train_dme, y_train_dme)

print("DME model ready ✅")
print("Train shape:", X_train_dme.shape)
print("Val shape:", X_val_dme.shape)

In [ ]:
# =========================================
# Permutation Importance — DME Selectivity
# =========================================

from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

# ---------------------------
# Baseline
# ---------------------------
baseline_pred = pipeline_dme.predict(X_val_dme)
baseline_r2 = r2_score(y_val_dme, baseline_pred)

print("Baseline R2:", baseline_r2)


# ---------------------------
# Permutation
# ---------------------------
rng = np.random.RandomState(42)
n_repeats = 5

feature_importance = []

for col in X_val_dme.columns:

    scores = []

    for _ in range(n_repeats):
        X_permuted = X_val_dme.copy()
        X_permuted[col] = rng.permutation(X_permuted[col].values)

        permuted_pred = pipeline_dme.predict(X_permuted)
        permuted_r2 = r2_score(y_val_dme, permuted_pred)

        scores.append(baseline_r2 - permuted_r2)

    feature_importance.append((col, np.mean(scores)))


# ---------------------------
# DataFrame
# ---------------------------
perm_importance_dme = pd.DataFrame(
    feature_importance,
    columns=["feature", "importance"]
).sort_values("importance", ascending=False)

print("\n===== TOP FEATURES (DME) =====")
print(perm_importance_dme.head(10))


print("\nNegative importance count:",
      (perm_importance_dme["importance"] < 0).sum())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax_h(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.set_axisbelow(True)
    ax.grid(True,  axis="x", color=GRID_COL,
            linewidth=0.7, linestyle="--", alpha=1.0)
    ax.grid(False, axis="y")
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which="minor", length=3, color="#888888", direction="in")
    ax.tick_params(which="major", length=5, color="#333333", direction="in")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

top = perm_importance_dme.head(10).copy()
top = top.sort_values("importance", ascending=True)

features   = top["feature"].values
importance = top["importance"].values

# Gradient: light orange → deep red
norm       = (importance - importance.min()) / (importance.max() - importance.min() + 1e-9)
cmap       = plt.cm.get_cmap("Oranges")
bar_colors = [cmap(0.35 + 0.55 * n) for n in norm]

fig, ax = plt.subplots(figsize=(10, 6.5))
fig.patch.set_facecolor(BG)
style_ax_h(ax)

bars = ax.barh(
    features, importance,
    color=bar_colors,
    edgecolor="white",
    linewidth=0.7,
    height=0.6,
    zorder=3,
)

for bar, val in zip(bars, importance):
    ax.text(
        bar.get_width() + importance.max() * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center", ha="left",
        color=TEXT_COL, fontsize=8.5, family="monospace"
    )

ax.barh(
    [features[-1]], [importance[-1]],
    color="none", edgecolor="#000000",
    linewidth=2.0, height=0.6, zorder=4,
)

for i, (feat, val) in enumerate(zip(features, importance)):
    rank = len(features) - i
    ax.text(
        -importance.max() * 0.015, i,
        f"#{rank}",
        va="center", ha="right",
        fontsize=7.5, color="#888888", family="monospace"
    )

ax.axvline(0, color="#333333", linewidth=1.2, zorder=2)

top_feat = features[-1]
top_val  = importance[-1]
ax.annotate(
    f"Most important:\n{top_feat}",
    xy=(top_val, len(features) - 1),
    xytext=(importance.max() * 0.75, 1.2),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.3,
                    connectionstyle="arc3,rad=0.3"),
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.9)
)

ax.set_xlabel("Permutation Importance  (ΔR²)")
ax.set_ylabel("Feature")
ax.set_title("Feature Importance — Random Forest  |  DME Selectivity")
ax.set_xlim(left=-importance.max() * 0.06,
            right=importance.max() * 1.22)

fig.text(0.5, -0.02,
         "Permutation importance = mean decrease in R² when feature values are randomly shuffled",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "feature_importance_dme.png")
plt.show()

In [ ]:
# ============================
# Prepare Methanol model
# ============================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

target = "Methanol_Selectivity_(%)"

data_model = df[df[target].notna()].copy()

X_model = data_model[numeric_features + categorical_features]
y_model = data_model[target]

X_train_meoh, X_val_meoh, y_train_meoh, y_val_meoh = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42
)

pipeline_meoh = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

pipeline_meoh.fit(X_train_meoh, y_train_meoh)

print("Methanol model ready ✅")

In [ ]:
# =========================================
# Permutation Importance — Methanol
# =========================================

from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

baseline_pred = pipeline_meoh.predict(X_val_meoh)
baseline_r2 = r2_score(y_val_meoh, baseline_pred)

print("Baseline R2:", baseline_r2)

rng = np.random.RandomState(42)
n_repeats = 5

feature_importance = []

for col in X_val_meoh.columns:

    scores = []

    for _ in range(n_repeats):
        X_permuted = X_val_meoh.copy()
        X_permuted[col] = rng.permutation(X_permuted[col].values)

        permuted_pred = pipeline_meoh.predict(X_permuted)
        permuted_r2 = r2_score(y_val_meoh, permuted_pred)

        scores.append(baseline_r2 - permuted_r2)

    feature_importance.append((col, np.mean(scores)))

perm_importance_meoh = pd.DataFrame(
    feature_importance,
    columns=["feature", "importance"]
).sort_values("importance", ascending=False)

print("\n===== TOP FEATURES (Methanol) =====")
print(perm_importance_meoh.head(10))

print("\nNegative importance count:",
      (perm_importance_meoh["importance"] < 0).sum())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax_h(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.set_axisbelow(True)
    ax.grid(True,  axis="x", color=GRID_COL,
            linewidth=0.7, linestyle="--", alpha=1.0)
    ax.grid(False, axis="y")
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which="minor", length=3, color="#888888", direction="in")
    ax.tick_params(which="major", length=5, color="#333333", direction="in")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

top = perm_importance_meoh.head(10).copy()
top = top.sort_values("importance", ascending=True)

features   = top["feature"].values
importance = top["importance"].values

# Gradient: light green → deep green
norm       = (importance - importance.min()) / (importance.max() - importance.min() + 1e-9)
cmap       = plt.cm.get_cmap("Greens")
bar_colors = [cmap(0.35 + 0.55 * n) for n in norm]

fig, ax = plt.subplots(figsize=(10, 6.5))
fig.patch.set_facecolor(BG)
style_ax_h(ax)

bars = ax.barh(
    features, importance,
    color=bar_colors,
    edgecolor="white",
    linewidth=0.7,
    height=0.6,
    zorder=3,
)

for bar, val in zip(bars, importance):
    ax.text(
        bar.get_width() + importance.max() * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center", ha="left",
        color=TEXT_COL, fontsize=8.5, family="monospace"
    )

ax.barh(
    [features[-1]], [importance[-1]],
    color="none", edgecolor="#000000",
    linewidth=2.0, height=0.6, zorder=4,
)

for i, (feat, val) in enumerate(zip(features, importance)):
    rank = len(features) - i
    ax.text(
        -importance.max() * 0.015, i,
        f"#{rank}",
        va="center", ha="right",
        fontsize=7.5, color="#888888", family="monospace"
    )

ax.axvline(0, color="#333333", linewidth=1.2, zorder=2)

top_feat = features[-1]
top_val  = importance[-1]
ax.annotate(
    f"Most important:\n{top_feat}",
    xy=(top_val, len(features) - 1),
    xytext=(importance.max() * 0.75, 1.2),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.3,
                    connectionstyle="arc3,rad=0.3"),
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.9)
)

ax.set_xlabel("Permutation Importance  (ΔR²)")
ax.set_ylabel("Feature")
ax.set_title("Feature Importance — Random Forest  |  Methanol Selectivity")
ax.set_xlim(left=-importance.max() * 0.06,
            right=importance.max() * 1.22)

fig.text(0.5, -0.02,
         "Permutation importance = mean decrease in R² when feature values are randomly shuffled",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "feature_importance_meoh.png")
plt.show()

In [ ]:
# ============================
# CO MODEL (CLEAN SETUP)
# ============================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

target = "CO_Selectivity_(%)"

data_co = df[df[target].notna()].copy()

X_co = data_co[numeric_features + categorical_features]
y_co = data_co[target]

X_train_co, X_val_co, y_train_co, y_val_co = train_test_split(
    X_co,
    y_co,
    test_size=0.2,
    random_state=42
)

pipeline_co = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

pipeline_co.fit(X_train_co, y_train_co)

print("CO model ready ✅")

In [ ]:
# =========================================
# Permutation Importance — CO Selectivity
# =========================================

from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

baseline_pred = pipeline_co.predict(X_val_co)
baseline_r2 = r2_score(y_val_co, baseline_pred)

print("Baseline R2:", baseline_r2)

rng = np.random.RandomState(42)
n_repeats = 5

feature_importance = []

for col in X_val_co.columns:

    scores = []

    for _ in range(n_repeats):
        X_permuted = X_val_co.copy()
        X_permuted[col] = rng.permutation(X_permuted[col].values)

        permuted_pred = pipeline_co.predict(X_permuted)
        permuted_r2 = r2_score(y_val_co, permuted_pred)

        scores.append(baseline_r2 - permuted_r2)

    feature_importance.append((col, np.mean(scores)))

perm_importance_co = pd.DataFrame(
    feature_importance,
    columns=["feature", "importance"]
).sort_values("importance", ascending=False)

print("\n===== TOP FEATURES (CO) =====")
print(perm_importance_co.head(10))

print("\nNegative importance count:",
      (perm_importance_co["importance"] < 0).sum())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax_h(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.set_axisbelow(True)
    ax.grid(True,  axis="x", color=GRID_COL,
            linewidth=0.7, linestyle="--", alpha=1.0)
    ax.grid(False, axis="y")
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which="minor", length=3, color="#888888", direction="in")
    ax.tick_params(which="major", length=5, color="#333333", direction="in")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

top = perm_importance_co.head(10).copy()
top = top.sort_values("importance", ascending=True)

features   = top["feature"].values
importance = top["importance"].values

# Gradient: light purple → deep purple
norm       = (importance - importance.min()) / (importance.max() - importance.min() + 1e-9)
cmap       = plt.cm.get_cmap("Purples")
bar_colors = [cmap(0.35 + 0.55 * n) for n in norm]

fig, ax = plt.subplots(figsize=(10, 6.5))
fig.patch.set_facecolor(BG)
style_ax_h(ax)

bars = ax.barh(
    features, importance,
    color=bar_colors,
    edgecolor="white",
    linewidth=0.7,
    height=0.6,
    zorder=3,
)

for bar, val in zip(bars, importance):
    ax.text(
        bar.get_width() + importance.max() * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center", ha="left",
        color=TEXT_COL, fontsize=8.5, family="monospace"
    )

ax.barh(
    [features[-1]], [importance[-1]],
    color="none", edgecolor="#000000",
    linewidth=2.0, height=0.6, zorder=4,
)

for i, (feat, val) in enumerate(zip(features, importance)):
    rank = len(features) - i
    ax.text(
        -importance.max() * 0.015, i,
        f"#{rank}",
        va="center", ha="right",
        fontsize=7.5, color="#888888", family="monospace"
    )

ax.axvline(0, color="#333333", linewidth=1.2, zorder=2)

top_feat = features[-1]
top_val  = importance[-1]
ax.annotate(
    f"Most important:\n{top_feat}",
    xy=(top_val, len(features) - 1),
    xytext=(importance.max() * 0.75, 1.2),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.3,
                    connectionstyle="arc3,rad=0.3"),
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.9)
)

ax.set_xlabel("Permutation Importance  (ΔR²)")
ax.set_ylabel("Feature")
ax.set_title("Feature Importance — Random Forest  |  CO Selectivity")
ax.set_xlim(left=-importance.max() * 0.06,
            right=importance.max() * 1.22)

fig.text(0.5, -0.02,
         "Permutation importance = mean decrease in R² when feature values are randomly shuffled",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "feature_importance_co.png")
plt.show()

In [ ]:
summary_df = pd.DataFrame({
    "Target": [
        "CO2 Conversion (%)",
        "DME Selectivity (%)",
        "Methanol Selectivity (%)",
        "CO Selectivity (%)"
    ],
    "Mean CV R2": [
        np.mean(cv_r2_scores_co2),
        np.mean(cv_r2_scores_dme),
        np.mean(cv_r2_scores_meoh),
        np.mean(cv_r2_scores_co)
    ],
    "CV Std": [
        np.std(cv_r2_scores_co2),
        np.std(cv_r2_scores_dme),
        np.std(cv_r2_scores_meoh),
        np.std(cv_r2_scores_co)
    ],
    "Primary Controlling Factors": [
        "Temperature-dominated kinetics (reaction conditions)",
        "Metal–acid synergy (dual-function catalysis)",
        "Acid-driven structure (Si/Al, pore architecture)",
        "Temperature-driven RWGS pathway"
    ]
})

summary_df[["Mean CV R2", "CV Std"]] = summary_df[["Mean CV R2", "CV Std"]].round(3)

summary_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
import numpy as np
import pandas as pd

# ── Typography ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  10,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"
C4       = "#7d3c98"   # purple for annotations

# One color per target — matches light-theme feature importance plots
TARGET_COLORS = [
    "#1a6faf",   # CO₂  — steel blue  (Blues cmap)
    "#d4621a",   # DME  — burnt orange (Oranges cmap)
    "#1e8449",   # MeOH — forest green (Greens cmap)
    "#6b3fa0",   # CO   — deep purple  (Purples cmap)
]

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.set_axisbelow(True)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

# ── Prep data ─────────────────────────────────────────────────────────────────
plot_df = summary_df.sort_values("Mean CV R2", ascending=False).reset_index(drop=True)
r2_vals = plot_df["Mean CV R2"].values
targets = plot_df["Target"].values

# Reorder colors to follow sorted order
color_map = {
    "CO2 Conversion":     TARGET_COLORS[0],
    "DME Selectivity":    TARGET_COLORS[1],
    "Methanol Selectivity": TARGET_COLORS[2],
    "CO Selectivity":     TARGET_COLORS[3],
}
bar_colors = [color_map.get(t, TARGET_COLORS[i % 4])
              for i, t in enumerate(targets)]

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

bars = ax.bar(
    targets, r2_vals,
    color=bar_colors,
    edgecolor="white",
    linewidth=0.8,
    width=0.52,
    zorder=3,
)

# Black border on best bar
best_idx = int(np.argmax(r2_vals))
ax.bar(
    [targets[best_idx]], [r2_vals[best_idx]],
    color="none",
    edgecolor="#000000",
    linewidth=2.2,
    width=0.52,
    zorder=4,
)

# Value labels above each bar
for bar, val in zip(bars, r2_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.012,
        f"{val:.3f}",
        ha="center", va="bottom",
        color=TEXT_COL, fontsize=10,
        fontweight="bold", family="monospace"
    )

# Threshold reference lines with proper light-theme styling
for thresh, label in [(0.95, "R² = 0.95"),
                       (0.90, "R² = 0.90"),
                       (0.80, "R² = 0.80")]:
    ax.axhline(thresh, color="#aaaaaa", linewidth=1.1,
               linestyle="--", zorder=2)
    ax.text(len(targets) - 0.42, thresh + 0.006,
            label, color="#888888", fontsize=8,
            style="italic")

# Best model callout annotation
best_target = targets[best_idx]
best_val    = r2_vals[best_idx]
ax.annotate(
    f"Best model: {best_target}\nR² = {best_val:.3f}",
    xy=(best_idx, best_val),
    xytext=(best_idx + 0.55, best_val - 0.12),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.3,
                    connectionstyle="arc3,rad=-0.2"),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.92)
)

# Std error bars if available
if "Std CV R2" in plot_df.columns:
    std_vals = plot_df["Std CV R2"].values
    ax.errorbar(
        targets, r2_vals,
        yerr=std_vals,
        fmt="none",
        ecolor="#555555",
        elinewidth=1.5,
        capsize=5,
        capthick=1.5,
        zorder=5,
    )

ax.set_ylabel("Mean CV R²")
ax.set_xlabel("Target Variable")
ax.set_title("Model Performance Comparison — All Targets  |  Random Forest")
ax.set_ylim(0, 1.12)
ax.tick_params(axis="x", rotation=15)
ax.grid(True,  axis="y", color=GRID_COL,
        linewidth=0.7, linestyle="--", alpha=1.0)
ax.grid(False, axis="x")

fig.text(0.5, -0.02,
         "Error bars show ±1 std across CV folds  •  "
         "Black border = best performing model  •  "
         "Dashed lines = R² thresholds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "model_performance_comparison.png")
plt.show()

In [ ]:
perm_importance_co2 = perm_importance_df.copy()

In [ ]:
print("CO2:", 'perm_importance_co2' in globals())
print("DME:", 'perm_importance_dme' in globals())
print("MeOH:", 'perm_importance_meoh' in globals())
print("CO :", 'perm_importance_co' in globals())

In [ ]:
feature_groups = {
    "Reaction Conditions": [
        "Reaction_temperature_C",
        "Pressure_MPa",
        "H2_CO2_ratio",
        "GHSV_mL_gcat_h"
    ],

    "Acid Properties": [
        "Total_acid_sites_mmol_g",
        "Si/Al_ratio",
        "Metal_acid_ratio",
        "acid_component_type"
    ],

    "Structural Properties": [
        "Surface_area_m2_g",
        "Crystallite_size_nm",
        "Pore_volume_cm3_g",
        "Dimentionality_of_Zeolite"
    ],

    "Metal Properties": [   # ✅ NEW (IMPORTANT FIX)
        "metal_1_atomic",
        "metal_2_atomic",
        "metal_3_atomic",
        "metal_4_atomic",
        "Cu_Metal_atomic_fraction"
    ],

    "System Design": [
        "Integration_method",
        "catalyst_mass_g"
    ]
}

def group_importance(perm_importance_df, feature_groups):

    # Convert to dictionary for faster lookup
    importance_dict = dict(zip(
        perm_importance_df["feature"],
        perm_importance_df["importance"]
    ))

    grouped_importance = {}

    for group_name, features in feature_groups.items():
        grouped_importance[group_name] = sum(
            importance_dict.get(feature, 0)
            for feature in features
        )

    return grouped_importance


group_co2 = group_importance(perm_importance_co2, feature_groups)
group_dme = group_importance(perm_importance_dme, feature_groups)
group_meoh = group_importance(perm_importance_meoh, feature_groups)
group_co = group_importance(perm_importance_co, feature_groups)

group_df = pd.DataFrame({
    "CO2 Conversion": group_co2,
    "DME Selectivity": group_dme,
    "Methanol Selectivity": group_meoh,
    "CO Selectivity": group_co
}).T

group_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

# ── Typography ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   12,
    "axes.labelsize":   10,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

GROUP_COLORS = [
    "#2166ac",   # Reaction Conditions   — deep blue
    "#1a9850",   # Acid Properties       — forest green
    "#d4621a",   # Structural Properties — burnt orange
    "#8e44ad",   # Metal Properties      — purple
    "#b5282a",   # System Design         — crimson
]

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.set_axisbelow(True)
    ax.grid(True,  axis="y", color=GRID_COL,
            linewidth=0.7, linestyle="--", alpha=1.0)
    ax.grid(False, axis="x")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which="minor", length=3,
                   color="#888888", direction="in")
    ax.tick_params(which="major", length=5,
                   color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

# ── Build dataframe ───────────────────────────────────────────────────────────
final_group_df = pd.DataFrame({
    "Reaction Conditions": {
        "CO₂ Conversion":       group_co2["Reaction Conditions"],
        "DME Selectivity":      group_dme["Reaction Conditions"],
        "Methanol Selectivity": group_meoh["Reaction Conditions"],
        "CO Selectivity":       group_co["Reaction Conditions"],
    },
    "Acid Properties": {
        "CO₂ Conversion":       group_co2["Acid Properties"],
        "DME Selectivity":      group_dme["Acid Properties"],
        "Methanol Selectivity": group_meoh["Acid Properties"],
        "CO Selectivity":       group_co["Acid Properties"],
    },
    "Structural Properties": {
        "CO₂ Conversion":       group_co2["Structural Properties"],
        "DME Selectivity":      group_dme["Structural Properties"],
        "Methanol Selectivity": group_meoh["Structural Properties"],
        "CO Selectivity":       group_co["Structural Properties"],
    },
    "Metal Properties": {
        "CO₂ Conversion":       group_co2["Metal Properties"],
        "DME Selectivity":      group_dme["Metal Properties"],
        "Methanol Selectivity": group_meoh["Metal Properties"],
        "CO Selectivity":       group_co["Metal Properties"],
    },
    "System Design": {
        "CO₂ Conversion":       group_co2["System Design"],
        "DME Selectivity":      group_dme["System Design"],
        "Methanol Selectivity": group_meoh["System Design"],
        "CO Selectivity":       group_co["System Design"],
    },
}).round(3)

group_norm   = final_group_df.div(final_group_df.sum(axis=1), axis=0)
groups       = final_group_df.columns.tolist()
target_names = final_group_df.index.tolist()
n_targets    = len(target_names)
n_groups     = len(groups)

print(final_group_df)
print("\n===== NORMALIZED GROUP IMPORTANCE =====")
print(group_norm)

x      = np.arange(n_targets)
width  = 0.14
offset = np.linspace(-(n_groups - 1) / 2,
                      (n_groups - 1) / 2, n_groups) * width

# ── Figure ────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))
fig.patch.set_facecolor(BG)

# ╔══════════════════════════════════════════╗
# ║  LEFT — Grouped bar (absolute)           ║
# ╚══════════════════════════════════════════╝
style_ax(ax1)

# Track dominant bar position for star placement
dom_positions = {}

for i, (grp, col, off) in enumerate(zip(groups, GROUP_COLORS, offset)):
    vals = final_group_df[grp].values
    bars = ax1.bar(
        x + off, vals,
        width=width * 0.88,
        color=col,
        edgecolor="white",
        linewidth=0.6,
        label=grp,
        zorder=3,
        alpha=0.88,
    )
    for xi, (b, v) in enumerate(zip(bars, vals)):
        if v == final_group_df.iloc[xi].max():
            # Store for star placement AFTER all labels drawn
            dom_positions[xi] = (b.get_x() + b.get_width() / 2,
                                  v, col)
            # Value label: placed just above bar
            ax1.text(
                b.get_x() + b.get_width() / 2,
                v + 0.006,
                f"{v:.3f}",
                ha="center", va="bottom",
                fontsize=7.5, color=col,
                fontweight="bold", family="monospace",
                zorder=6,
            )

# FIX 1: Star placed well above the label — use ax1.get_ylim dynamically
y_max = final_group_df.values.max()
star_offset = y_max * 0.07   # 7% of y-range above bar top

for xi, (bx, bv, col) in dom_positions.items():
    ax1.plot(
        bx, bv + star_offset,
        marker="*", color="#111111",
        markersize=10, zorder=7,
        markeredgewidth=0.5, markeredgecolor="#555555"
    )

# Set ylim with enough headroom for star + label
ax1.set_ylim(0, y_max * 1.22)

ax1.set_xticks(x)
ax1.set_xticklabels(target_names, rotation=15, ha="right")
ax1.set_ylabel("Aggregated Permutation Importance (ΔR²)")
ax1.set_title("Dominant Control Layers — Absolute Importance")

star_entry = Line2D([0], [0], marker="*", color="#111111",
                    linestyle="none", markersize=9,
                    label="Dominant group")
handles, labels = ax1.get_legend_handles_labels()
ax1.legend(
    handles + [star_entry],
    labels  + ["Dominant group"],
    title="Feature Group", title_fontsize=9,
    fontsize=8.5, facecolor="white",
    edgecolor="#cccccc", labelcolor=TEXT_COL,
    loc="upper right", framealpha=1,
)

# ╔══════════════════════════════════════════╗
# ║  RIGHT — Stacked bar (normalized 100%)   ║
# ╚══════════════════════════════════════════╝
style_ax(ax2)

bottoms = np.zeros(n_targets)
for grp, col in zip(groups, GROUP_COLORS):
    vals = group_norm[grp].values * 100
    bars = ax2.bar(
        x, vals,
        bottom=bottoms,
        color=col,
        edgecolor="white",
        linewidth=0.7,
        label=grp,
        width=0.55,
        zorder=3,
        alpha=0.88,
    )
    # ── % label inside every segment ≥ 5% ────────────────────────────────────
    for xi, (v, b) in enumerate(zip(vals, bottoms)):
        if v >= 5:                          # lower threshold → more labels
            lum = (int(col[1:3], 16) * 0.299 +
                   int(col[3:5], 16) * 0.587 +
                   int(col[5:7], 16) * 0.114)
            txt_col = "white" if lum < 140 else "#1a1a1a"
            ax2.text(
                xi, b + v / 2,
                f"{v:.1f}%",
                ha="center", va="center",
                color=txt_col, fontsize=8,
                fontweight="bold", family="monospace",
                zorder=5,
            )
    bottoms += vals

# ▲ dominant group label above each bar
for xi in range(n_targets):
    dom_grp = group_norm.iloc[xi].idxmax()
    ax2.text(
        xi, 101.5,
        f"▲ {dom_grp.split()[0]}",
        ha="center", va="bottom",
        fontsize=7.5, color="#333333",
        fontweight="bold",
    )

ax2.set_xticks(x)
ax2.set_xticklabels(target_names, rotation=15, ha="right")
ax2.set_ylabel("Relative Contribution (%)")
ax2.set_ylim(0, 114)
ax2.set_title("Dominant Control Layers — Normalized (100%)")

# Legend — upper RIGHT but OUTSIDE the plot area so it never covers bars
ax2.legend(
    title="Feature Group", title_fontsize=9,
    fontsize=8.5, facecolor="white",
    edgecolor="#cccccc", labelcolor=TEXT_COL,
    loc="upper left",
    bbox_to_anchor=(1.01, 1),   # ← pushed outside right edge
    borderaxespad=0,
    framealpha=1,
)

# ── Super-title & caption ─────────────────────────────────────────────────────
fig.suptitle(
    "Feature Group Importance Across All Targets  |  Random Forest",
    fontsize=14, fontweight="bold",
    color=TEXT_COL, y=1.02
)
fig.text(
    0.5, -0.03,
    "Left: absolute aggregated permutation importance (ΔR²)  •  "
    "Right: normalised relative contribution per target  •  "
    "★ = dominant group  •  ▲ = leading group label",
    ha="center", fontsize=8, color="#666666", style="italic"
)

plt.tight_layout()
save_fig(fig, "group_importance_all_targets.png")
plt.show()

In [ ]:
# # ============================================================
# # EXPORT HOT-ENCODED DATA (WHAT THE ML MODEL ACTUALLY SEES)
# # ============================================================

# import pandas as pd

# # ------------------------------------------------------------
# # 1. Select the exact input features used by the model
# #    (numeric + categorical, NO targets yet)
# # ------------------------------------------------------------
# X_full = df[numeric_features + categorical_features].copy()

# # ------------------------------------------------------------
# # 2. Use the FITTED preprocessor inside the trained pipeline
# #    This applies:
# #    - StandardScaler on numeric features
# #    - OneHotEncoder on categorical features
# # ------------------------------------------------------------
# X_encoded = pipeline.named_steps["preprocessor"].transform(X_full)

# # ------------------------------------------------------------
# # 3. Get feature names after one-hot encoding
# # ------------------------------------------------------------
# encoded_feature_names = pipeline.named_steps[
#     "preprocessor"
# ].get_feature_names_out()

# # ------------------------------------------------------------
# # 4. Convert encoded array to a DataFrame
# # ------------------------------------------------------------
# X_encoded_df = pd.DataFrame(
#     X_encoded,
#     columns=encoded_feature_names,
#     index=df.index
# )

# # ------------------------------------------------------------
# # 5. (Optional but recommended)
# #    Append target columns for completeness / reuse
# # ------------------------------------------------------------
# for col in target_cols:
#     X_encoded_df[col] = df[col].values

# # ------------------------------------------------------------
# # 6. Save the hot-encoded dataset to Google Drive
# # ------------------------------------------------------------
# encoded_csv_path = "/content/drive/MyDrive/CO2 to DME/CO2_to_DME_hot_encoded.csv"
# encoded_xlsx_path = "/content/drive/MyDrive/CO2 to DME/CO2_to_DME_hot_encoded.xlsx"

# X_encoded_df.to_csv(encoded_csv_path, index=False)
# X_encoded_df.to_excel(encoded_xlsx_path, index=False)

# print("Hot-encoded files saved:")
# print(encoded_csv_path)
# print(encoded_xlsx_path)

# # ------------------------------------------------------------
# # 7. Quick sanity check
# # ------------------------------------------------------------
# print("Encoded data shape:", X_encoded_df.shape)
# X_encoded_df.head()


In [ ]:
# ============================================================
# Master results container for model comparison (IMPROVED)
# ============================================================

rf_results = {
    "CO2 Conversion": {
        "cv_r2_scores": cv_r2_scores_co2,
    },
    "DME Selectivity": {
        "cv_r2_scores": cv_r2_scores_dme,
    },
    "Methanol Selectivity": {
        "cv_r2_scores": cv_r2_scores_meoh,
    },
    "CO Selectivity": {
        "cv_r2_scores": cv_r2_scores_co,
    }
}

# ---------------------------
# Build summary table
# ---------------------------
import pandas as pd
import numpy as np

rf_summary_df = pd.DataFrame.from_dict(
    rf_results,
    orient="index"
).reset_index().rename(columns={"index": "Target"})

# Compute metrics
rf_summary_df["Mean CV R²"] = rf_summary_df["cv_r2_scores"].apply(np.mean)
rf_summary_df["CV Std"] = rf_summary_df["cv_r2_scores"].apply(np.std)

# Round
rf_summary_df[["Mean CV R²", "CV Std"]] = rf_summary_df[
    ["Mean CV R²", "CV Std"]
].round(3)

# Keep only useful columns
rf_summary_df = rf_summary_df[["Target", "Mean CV R²", "CV Std"]]

print(rf_summary_df)

In [ ]:
# Save summary as CSV (for paper use)
rf_summary_df.to_csv("rf_summary.csv", index=False)

# Save full results
import pickle
with open("rf_results.pkl", "wb") as f:
    pickle.dump(rf_results, f)

print("All RF outputs saved ✅")


In [ ]:
# ============================================================
# ExtraTrees — K-Fold CV — CO2 Conversion Efficiency (FIXED)
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

target = "CO2_conversion_efficiency_(%)"

# ✅ ONLY drop target NaNs
data_co2 = df[df[target].notna()].copy()

X_co2 = data_co2[numeric_features + categorical_features]
y_co2 = data_co2[target]

# Model
et_co2 = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

pipeline_et_co2 = Pipeline([
    ("preprocessor", preprocessor),
    ("model", et_co2)
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_r2_scores_et_co2 = cross_val_score(
    pipeline_et_co2,
    X_co2,
    y_co2,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_et_co2)
print("Mean R2:", cv_r2_scores_et_co2.mean())
print("Std:", cv_r2_scores_et_co2.std())

# Verification
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_co2))
print("NaNs in X:", X_co2.isna().sum().sum())
print("NaNs in y:", y_co2.isna().sum())

In [ ]:
# ============================================================
# ExtraTrees — K-Fold CV — DME Selectivity (FIXED)
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

target = "DME_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_dme = df[df[target].notna()].copy()

X_dme = data_dme[numeric_features + categorical_features]
y_dme = data_dme[target]

et_dme = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

pipeline_et_dme = Pipeline([
    ("preprocessor", preprocessor),
    ("model", et_dme)
])

cv_r2_scores_et_dme = cross_val_score(
    pipeline_et_dme,
    X_dme,
    y_dme,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_et_dme)
print("Mean R2:", cv_r2_scores_et_dme.mean())
print("Std:", cv_r2_scores_et_dme.std())

# Verification
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_dme))
print("NaNs in X:", X_dme.isna().sum().sum())
print("NaNs in y:", y_dme.isna().sum())

In [ ]:
# ============================================================
# ExtraTrees — K-Fold CV — Methanol Selectivity (FIXED)
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

target = "Methanol_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_meoh = df[df[target].notna()].copy()

X_meoh = data_meoh[numeric_features + categorical_features]
y_meoh = data_meoh[target]

et_meoh = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

pipeline_et_meoh = Pipeline([
    ("preprocessor", preprocessor),
    ("model", et_meoh)
])

cv_r2_scores_et_meoh = cross_val_score(
    pipeline_et_meoh,
    X_meoh,
    y_meoh,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_et_meoh)
print("Mean R2:", cv_r2_scores_et_meoh.mean())
print("Std:", cv_r2_scores_et_meoh.std())

# Verification
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_meoh))
print("NaNs in X:", X_meoh.isna().sum().sum())
print("NaNs in y:", y_meoh.isna().sum())

In [ ]:
# ============================================================
# ExtraTrees — K-Fold CV — CO Selectivity (FIXED)
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

target = "CO_Selectivity_(%)"

# ✅ ONLY drop target NaNs (IMPORTANT)
data_co = df[df[target].notna()].copy()

X_co = data_co[numeric_features + categorical_features]
y_co = data_co[target]

et_co = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

pipeline_et_co = Pipeline([
    ("preprocessor", preprocessor),
    ("model", et_co)
])

cv_r2_scores_et_co = cross_val_score(
    pipeline_et_co,
    X_co,
    y_co,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_et_co)
print("Mean R2:", cv_r2_scores_et_co.mean())
print("Std:", cv_r2_scores_et_co.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_co))
print("NaNs in X:", X_co.isna().sum().sum())
print("NaNs in y:", y_co.isna().sum())

In [ ]:
# ============================================================
# Store ExtraTrees results for later comparison (UPDATED)
# ============================================================

et_results = {
    "CO2 Conversion": {
        "cv_r2_scores": cv_r2_scores_et_co2,
        "mean_cv_r2": np.mean(cv_r2_scores_et_co2),
        "std_cv_r2": np.std(cv_r2_scores_et_co2),
    },
    "DME Selectivity": {
        "cv_r2_scores": cv_r2_scores_et_dme,
        "mean_cv_r2": np.mean(cv_r2_scores_et_dme),
        "std_cv_r2": np.std(cv_r2_scores_et_dme),
    },
    "Methanol Selectivity": {
        "cv_r2_scores": cv_r2_scores_et_meoh,
        "mean_cv_r2": np.mean(cv_r2_scores_et_meoh),
        "std_cv_r2": np.std(cv_r2_scores_et_meoh),
    },
    "CO Selectivity": {   # ✅ ADD THIS
        "cv_r2_scores": cv_r2_scores_et_co,
        "mean_cv_r2": np.mean(cv_r2_scores_et_co),
        "std_cv_r2": np.std(cv_r2_scores_et_co),
    }
}

# Quick view
import pandas as pd

et_summary_df = pd.DataFrame.from_dict(
    et_results,
    orient="index"
).reset_index().rename(columns={"index": "Target"})

et_summary_df[["mean_cv_r2", "std_cv_r2"]] = et_summary_df[
    ["mean_cv_r2", "std_cv_r2"]
].round(3)

et_summary_df

In [ ]:
# ============================================================
# XGBoost — CO2 Conversion (SAFE EARLY STOPPING)
# ============================================================

from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np

target = "CO2_conversion_efficiency_(%)"

data = df[df[target].notna()].copy()

X = data[numeric_features + categorical_features]
y = data[target]

kf = KFold(n_splits=10, shuffle=True, random_state=42)

r2_scores = []

for train_idx, val_idx in kf.split(X):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Preprocessing
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)

    model = XGBRegressor(
        n_estimators=300,   # 🔻 reduced instead of early stopping
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1,
        reg_alpha=0.1,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    # ✅ NORMAL FIT (NO early stopping)
    model.fit(X_train_processed, y_train)

    preds = model.predict(X_val_processed)
    r2 = r2_score(y_val, preds)
    r2_scores.append(r2)

# Results
r2_scores = np.array(r2_scores)

print("Scores:", r2_scores)
print("Mean R2:", r2_scores.mean())
print("Std:", r2_scores.std())

In [ ]:
# ============================================================
# XGBoost — K-Fold CV — DME Selectivity (FIXED)
# ============================================================

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

target = "DME_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_dme = df[df[target].notna()].copy()

X_dme = data_dme[numeric_features + categorical_features]
y_dme = data_dme[target]

xgb_dme = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    reg_alpha=0.1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

pipeline_xgb_dme = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_dme)
])

cv_r2_scores_xgb_dme = cross_val_score(
    pipeline_xgb_dme,
    X_dme,
    y_dme,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_xgb_dme)
print("Mean R2:", cv_r2_scores_xgb_dme.mean())
print("Std:", cv_r2_scores_xgb_dme.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_dme))
print("NaNs in X:", X_dme.isna().sum().sum())
print("NaNs in y:", y_dme.isna().sum())

In [ ]:
# ============================================================
# XGBoost — K-Fold CV — Methanol Selectivity (FIXED)
# ============================================================

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

target = "Methanol_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_meoh = df[df[target].notna()].copy()

X_meoh = data_meoh[numeric_features + categorical_features]
y_meoh = data_meoh[target]

xgb_meoh = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    reg_alpha=0.1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

pipeline_xgb_meoh = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_meoh)
])

cv_r2_scores_xgb_meoh = cross_val_score(
    pipeline_xgb_meoh,
    X_meoh,
    y_meoh,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_xgb_meoh)
print("Mean R2:", cv_r2_scores_xgb_meoh.mean())
print("Std:", cv_r2_scores_xgb_meoh.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_meoh))
print("NaNs in X:", X_meoh.isna().sum().sum())
print("NaNs in y:", y_meoh.isna().sum())

In [ ]:
# ============================================================
# XGBoost — K-Fold CV — CO Selectivity (FIXED)
# ============================================================

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

target = "CO_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_co = df[df[target].notna()].copy()

X_co = data_co[numeric_features + categorical_features]
y_co = data_co[target]

xgb_co = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    reg_alpha=0.1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

pipeline_xgb_co = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_co)
])

cv_r2_scores_xgb_co = cross_val_score(
    pipeline_xgb_co,
    X_co,
    y_co,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_xgb_co)
print("Mean R2:", cv_r2_scores_xgb_co.mean())
print("Std:", cv_r2_scores_xgb_co.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_co))
print("NaNs in X:", X_co.isna().sum().sum())
print("NaNs in y:", y_co.isna().sum())

In [ ]:
# ============================================================
# Store XGBoost results for later comparison (UPDATED)
# ============================================================

cv_r2_scores_xgb_co2 = r2_scores # Assign the correct variable from previous cell

xgb_results = {
    "CO2 Conversion": {
        "cv_r2_scores": cv_r2_scores_xgb_co2,
        "mean_cv_r2": np.mean(cv_r2_scores_xgb_co2),
        "std_cv_r2": np.std(cv_r2_scores_xgb_co2),
    },
    "DME Selectivity": {
        "cv_r2_scores": cv_r2_scores_xgb_dme,
        "mean_cv_r2": np.mean(cv_r2_scores_xgb_dme),
        "std_cv_r2": np.std(cv_r2_scores_xgb_dme),
    },
    "Methanol Selectivity": {
        "cv_r2_scores": cv_r2_scores_xgb_meoh,
        "mean_cv_r2": np.mean(cv_r2_scores_xgb_meoh),
        "std_cv_r2": np.std(cv_r2_scores_xgb_meoh),
    },
    "CO Selectivity": {   # ✅ ADD THIS
        "cv_r2_scores": cv_r2_scores_xgb_co,
        "mean_cv_r2": np.mean(cv_r2_scores_xgb_co),
        "std_cv_r2": np.std(cv_r2_scores_xgb_co),
    }
}

# ============================
# SUMMARY TABLE
# ============================
import pandas as pd

xgb_summary_df = pd.DataFrame.from_dict(
    xgb_results,
    orient="index"
).reset_index().rename(columns={"index": "Target"})

xgb_summary_df[["mean_cv_r2", "std_cv_r2"]] = xgb_summary_df[
    ["mean_cv_r2", "std_cv_r2"]
].round(3)

xgb_summary_df

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
import numpy as np
numeric_features_hgbr = numeric_features.copy()


In [ ]:
# ============================================================
# HGBR (Numeric-only) — CO2 Conversion (FIXED)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

target = "CO2_conversion_efficiency_(%)"

# ✅ ONLY drop target NaNs
data_co2 = df[df[target].notna()].copy()

X_co2 = data_co2[numeric_features]
y_co2 = data_co2[target]

hgbr_co2 = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

cv_r2_scores_hgbr_co2 = cross_val_score(
    hgbr_co2,
    X_co2,
    y_co2,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_hgbr_co2)
print("Mean R2:", cv_r2_scores_hgbr_co2.mean())
print("Std:", cv_r2_scores_hgbr_co2.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_co2))
print("NaNs in X:", X_co2.isna().sum().sum())
print("NaNs in y:", y_co2.isna().sum())

In [ ]:
# ============================================================
# HGBR (Numeric-only) — DME Selectivity (FIXED)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

target = "DME_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_dme = df[df[target].notna()].copy()

X_dme = data_dme[numeric_features]
y_dme = data_dme[target]

hgbr_dme = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

cv_r2_scores_hgbr_dme = cross_val_score(
    hgbr_dme,
    X_dme,
    y_dme,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_hgbr_dme)
print("Mean R2:", cv_r2_scores_hgbr_dme.mean())
print("Std:", cv_r2_scores_hgbr_dme.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_dme))
print("NaNs in X:", X_dme.isna().sum().sum())
print("NaNs in y:", y_dme.isna().sum())

In [ ]:
# ============================================================
# HGBR (Numeric-only) — Methanol Selectivity (FIXED)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

target = "Methanol_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_meoh = df[df[target].notna()].copy()

X_meoh = data_meoh[numeric_features]
y_meoh = data_meoh[target]

hgbr_meoh = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

cv_r2_scores_hgbr_meoh = cross_val_score(
    hgbr_meoh,
    X_meoh,
    y_meoh,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_hgbr_meoh)
print("Mean R2:", cv_r2_scores_hgbr_meoh.mean())
print("Std:", cv_r2_scores_hgbr_meoh.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_meoh))
print("NaNs in X:", X_meoh.isna().sum().sum())
print("NaNs in y:", y_meoh.isna().sum())

In [ ]:
# ============================================================
# HGBR (Numeric-only) — CO Selectivity (FIXED)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

target = "CO_Selectivity_(%)"

# ✅ ONLY drop target NaNs (NOT features)
data_co = df[df[target].notna()].copy()

X_co = data_co[numeric_features]
y_co = data_co[target]

hgbr_co = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

cv_r2_scores_hgbr_co = cross_val_score(
    hgbr_co,
    X_co,
    y_co,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_hgbr_co)
print("Mean R2:", cv_r2_scores_hgbr_co.mean())
print("Std:", cv_r2_scores_hgbr_co.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_co))
print("NaNs in X:", X_co.isna().sum().sum())
print("NaNs in y:", y_co.isna().sum())

In [ ]:
# ============================================================
# Store HGBR (numeric-only) results for comparison
# ============================================================

hgbr_num_results = {
    "CO2 Conversion": {
        "cv_r2_scores": cv_r2_scores_hgbr_co2,
        "mean_cv_r2": np.mean(cv_r2_scores_hgbr_co2),
        "std_cv_r2": np.std(cv_r2_scores_hgbr_co2),
    },
    "DME Selectivity": {
        "cv_r2_scores": cv_r2_scores_hgbr_dme,
        "mean_cv_r2": np.mean(cv_r2_scores_hgbr_dme),
        "std_cv_r2": np.std(cv_r2_scores_hgbr_dme),
    },
    "Methanol Selectivity": {
        "cv_r2_scores": cv_r2_scores_hgbr_meoh,
        "mean_cv_r2": np.mean(cv_r2_scores_hgbr_meoh),
        "std_cv_r2": np.std(cv_r2_scores_hgbr_meoh),
    },
    "CO Selectivity": {
        "cv_r2_scores": cv_r2_scores_hgbr_co,
        "mean_cv_r2": np.mean(cv_r2_scores_hgbr_co),
        "std_cv_r2": np.std(cv_r2_scores_hgbr_co),
    }
}

hgbr_num_results

In [ ]:
# ============================================================
# HGBR (Full features) — CO2 Conversion (FIXED)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

target = "CO2_conversion_efficiency_(%)"

# ✅ ONLY drop target NaNs
data_co2 = df[df[target].notna()].copy()

X_co2 = data_co2[numeric_features + categorical_features]
y_co2 = data_co2[target]

hgbr_full_co2 = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

pipeline_hgbr_full_co2 = Pipeline([
    ("preprocessor", preprocessor),  # includes OneHotEncoder
    ("model", hgbr_full_co2)
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

cv_r2_scores_hgbr_full_co2 = cross_val_score(
    pipeline_hgbr_full_co2,
    X_co2,
    y_co2,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_hgbr_full_co2)
print("Mean R2:", cv_r2_scores_hgbr_full_co2.mean())
print("Std:", cv_r2_scores_hgbr_full_co2.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_co2))
print("NaNs in X:", X_co2.isna().sum().sum())
print("NaNs in y:", y_co2.isna().sum())

In [ ]:
# ============================================================
# HGBR (Full features) — DME Selectivity (FIXED)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

target = "DME_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_dme = df[df[target].notna()].copy()

X_dme = data_dme[numeric_features + categorical_features]
y_dme = data_dme[target]

hgbr_full_dme = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

pipeline_hgbr_full_dme = Pipeline([
    ("preprocessor", preprocessor),
    ("model", hgbr_full_dme)
])

cv_r2_scores_hgbr_full_dme = cross_val_score(
    pipeline_hgbr_full_dme,
    X_dme,
    y_dme,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_hgbr_full_dme)
print("Mean R2:", cv_r2_scores_hgbr_full_dme.mean())
print("Std:", cv_r2_scores_hgbr_full_dme.std())

# ============================
# VERIFICATION
# ============================
print("\n===== DATA CHECK =====")
print("Rows used:", len(X_dme))
print("NaNs in X:", X_dme.isna().sum().sum())
print("NaNs in y:", y_dme.isna().sum())

In [ ]:
# ============================================================
# HGBR (Full features) — Methanol Selectivity (FIXED)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

target = "Methanol_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_meoh = df[df[target].notna()].copy()

X_meoh = data_meoh[numeric_features + categorical_features]
y_meoh = data_meoh[target]

hgbr_full_meoh = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

pipeline_hgbr_full_meoh = Pipeline([
    ("preprocessor", preprocessor),
    ("model", hgbr_full_meoh)
])

cv_r2_scores_hgbr_full_meoh = cross_val_score(
    pipeline_hgbr_full_meoh,
    X_meoh,
    y_meoh,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_hgbr_full_meoh)
print("Mean R2:", cv_r2_scores_hgbr_full_meoh.mean())
print("Std:", cv_r2_scores_hgbr_full_meoh.std())

print("\n===== DATA CHECK =====")
print("Rows used:", len(X_meoh))
print("NaNs in X:", X_meoh.isna().sum().sum())
print("NaNs in y:", y_meoh.isna().sum())

In [ ]:
# ============================================================
# HGBR (Full features) — CO Selectivity (FIXED)
# ============================================================

target = "CO_Selectivity_(%)"

# ✅ ONLY drop target NaNs
data_co = df[df[target].notna()].copy()

X_co = data_co[numeric_features + categorical_features]
y_co = data_co[target]

hgbr_full_co = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

pipeline_hgbr_full_co = Pipeline([
    ("preprocessor", preprocessor),
    ("model", hgbr_full_co)
])

cv_r2_scores_hgbr_full_co = cross_val_score(
    pipeline_hgbr_full_co,
    X_co,
    y_co,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Scores:", cv_r2_scores_hgbr_full_co)
print("Mean R2:", cv_r2_scores_hgbr_full_co.mean())
print("Std:", cv_r2_scores_hgbr_full_co.std())

print("\n===== DATA CHECK =====")
print("Rows used:", len(X_co))
print("NaNs in X:", X_co.isna().sum().sum())
print("NaNs in y:", y_co.isna().sum())

In [ ]:
# ============================================================
# Store HGBR (Full features) results for comparison
# ============================================================

hgbr_full_results = {
    "CO2 Conversion": {
        "cv_r2_scores": cv_r2_scores_hgbr_full_co2,
        "mean_cv_r2": np.mean(cv_r2_scores_hgbr_full_co2),
        "std_cv_r2": np.std(cv_r2_scores_hgbr_full_co2),
    },
    "DME Selectivity": {
        "cv_r2_scores": cv_r2_scores_hgbr_full_dme,
        "mean_cv_r2": np.mean(cv_r2_scores_hgbr_full_dme),
        "std_cv_r2": np.std(cv_r2_scores_hgbr_full_dme),
    },
    "Methanol Selectivity": {
        "cv_r2_scores": cv_r2_scores_hgbr_full_meoh,
        "mean_cv_r2": np.mean(cv_r2_scores_hgbr_full_meoh),
        "std_cv_r2": np.std(cv_r2_scores_hgbr_full_meoh),
    },
    "CO Selectivity": {
        "cv_r2_scores": cv_r2_scores_hgbr_full_co,
        "mean_cv_r2": np.mean(cv_r2_scores_hgbr_full_co),
        "std_cv_r2": np.std(cv_r2_scores_hgbr_full_co),
    }
}

hgbr_full_results

In [ ]:
# ============================================================
# REBUILD ALL RESULT DICTIONARIES CLEANLY
# ============================================================

# RF
rf_results = {
    "CO2 Conversion": {"mean": np.mean(cv_r2_scores_co2), "std": np.std(cv_r2_scores_co2)},
    "DME Selectivity": {"mean": np.mean(cv_r2_scores_dme), "std": np.std(cv_r2_scores_dme)},
    "Methanol Selectivity": {"mean": np.mean(cv_r2_scores_meoh), "std": np.std(cv_r2_scores_meoh)},
    "CO Selectivity": {"mean": np.mean(cv_r2_scores_co), "std": np.std(cv_r2_scores_co)},
}

# ExtraTrees
et_results = {
    "CO2 Conversion": {"mean": np.mean(cv_r2_scores_et_co2), "std": np.std(cv_r2_scores_et_co2)},
    "DME Selectivity": {"mean": np.mean(cv_r2_scores_et_dme), "std": np.std(cv_r2_scores_et_dme)},
    "Methanol Selectivity": {"mean": np.mean(cv_r2_scores_et_meoh), "std": np.std(cv_r2_scores_et_meoh)},
    "CO Selectivity": {"mean": np.mean(cv_r2_scores_et_co), "std": np.std(cv_r2_scores_et_co)},
}

# XGBoost
xgb_results = {
    "CO2 Conversion": {"mean": np.mean(cv_r2_scores_xgb_co2), "std": np.std(cv_r2_scores_xgb_co2)},
    "DME Selectivity": {"mean": np.mean(cv_r2_scores_xgb_dme), "std": np.std(cv_r2_scores_xgb_dme)},
    "Methanol Selectivity": {"mean": np.mean(cv_r2_scores_xgb_meoh), "std": np.std(cv_r2_scores_xgb_meoh)},
    "CO Selectivity": {"mean": np.mean(cv_r2_scores_xgb_co), "std": np.std(cv_r2_scores_xgb_co)},
}

# HGBR numeric
hgbr_num_results = {
    "CO2 Conversion": {"mean": np.mean(cv_r2_scores_hgbr_co2), "std": np.std(cv_r2_scores_hgbr_co2)},
    "DME Selectivity": {"mean": np.mean(cv_r2_scores_hgbr_dme), "std": np.std(cv_r2_scores_hgbr_dme)},
    "Methanol Selectivity": {"mean": np.mean(cv_r2_scores_hgbr_meoh), "std": np.std(cv_r2_scores_hgbr_meoh)},
    "CO Selectivity": {"mean": np.mean(cv_r2_scores_hgbr_co), "std": np.std(cv_r2_scores_hgbr_co)},
}

# HGBR full
hgbr_full_results = {
    "CO2 Conversion": {"mean": np.mean(cv_r2_scores_hgbr_full_co2), "std": np.std(cv_r2_scores_hgbr_full_co2)},
    "DME Selectivity": {"mean": np.mean(cv_r2_scores_hgbr_full_dme), "std": np.std(cv_r2_scores_hgbr_full_dme)},
    "Methanol Selectivity": {"mean": np.mean(cv_r2_scores_hgbr_full_meoh), "std": np.std(cv_r2_scores_hgbr_full_meoh)},
    "CO Selectivity": {"mean": np.mean(cv_r2_scores_hgbr_full_co), "std": np.std(cv_r2_scores_hgbr_full_co)},
}
# ============================================================
# FINAL COMPARISON TABLE
# ============================================================

final_comparison = pd.DataFrame({
    "Target": ["CO2 Conversion", "DME Selectivity", "Methanol Selectivity", "CO Selectivity"],

    "RF": [rf_results[t]["mean"] for t in rf_results],
    "ExtraTrees": [et_results[t]["mean"] for t in et_results],
    "XGBoost": [xgb_results[t]["mean"] for t in xgb_results],
    "HGBR (Numeric)": [hgbr_num_results[t]["mean"] for t in hgbr_num_results],
    "HGBR (Full)": [hgbr_full_results[t]["mean"] for t in hgbr_full_results],
})

final_comparison.round(3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from sklearn.model_selection import learning_curve, KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.base import clone
from scipy import stats
from scipy.stats import shapiro

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


def model_diagnostics(pipeline, X, y, model_name, target_name):

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    # ╔══════════════════════════════════════════╗
    # ║  1. LEARNING CURVE                       ║
    # ╚══════════════════════════════════════════╝
    train_sizes, train_scores, val_scores = learning_curve(
        pipeline, X, y,
        cv=kf,
        scoring="neg_root_mean_squared_error",
        train_sizes=np.linspace(0.1, 1.0, 6),
        n_jobs=-1
    )

    train_rmse     = -train_scores.mean(axis=1)
    val_rmse       = -val_scores.mean(axis=1)
    train_rmse_std = (-train_scores).std(axis=1)
    val_rmse_std   = (-val_scores).std(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor(BG)
    style_ax(ax)

    ax.plot(train_sizes, train_rmse, marker="o", color=C1,
            linewidth=2.2, markersize=8, label="Training RMSE",
            markerfacecolor="white", markeredgewidth=2, zorder=4)
    ax.fill_between(train_sizes,
                    train_rmse - train_rmse_std,
                    train_rmse + train_rmse_std,
                    color=C1, alpha=0.13, label="Train ±1σ")

    ax.plot(train_sizes, val_rmse, marker="s", color=C2,
            linewidth=2.2, markersize=8, label="CV RMSE",
            markerfacecolor="white", markeredgewidth=2, zorder=4)
    ax.fill_between(train_sizes,
                    val_rmse - val_rmse_std,
                    val_rmse + val_rmse_std,
                    color=C2, alpha=0.13, label="CV ±1σ")

    gap = val_rmse[-1] - train_rmse[-1]
    ax.annotate(
        f"Generalisation gap\n= {gap:.3f}",
        xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
        xytext=(train_sizes[-2] * 0.78, val_rmse[-1] * 1.05),
        fontsize=8.5, color=C4,
        arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
                  edgecolor=C4, alpha=0.85)
    )

    ax.set_xlabel("Training Set Size")
    ax.set_ylabel("RMSE")
    ax.set_title(f"Learning Curve — {model_name}  |  {target_name}")
    ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
              framealpha=1, ncol=2)

    fig.text(0.5, -0.02,
             "Shaded bands = ±1 standard deviation across CV folds",
             ha="center", fontsize=8, color="#666666", style="italic")

    plt.tight_layout()
    save_fig(fig, f"lc_{model_name}_{target_name}.png".replace(" ", "_"))
    plt.show()

    # ╔══════════════════════════════════════════╗
    # ║  2. OOF PREDICTIONS                      ║
    # ╚══════════════════════════════════════════╝
    y_oof_pred = cross_val_predict(pipeline, X, y, cv=kf, n_jobs=-1)
    y_arr      = np.array(y)
    residuals  = y_arr - y_oof_pred

    # ╔══════════════════════════════════════════╗
    # ║  3. METRICS                              ║
    # ╚══════════════════════════════════════════╝
    rmse = np.sqrt(mean_squared_error(y_arr, y_oof_pred))
    mae  = mean_absolute_error(y_arr, y_oof_pred)
    r2   = r2_score(y_arr, y_oof_pred)
    corr = np.corrcoef(y_arr, y_oof_pred)[0, 1]

    # ╔══════════════════════════════════════════╗
    # ║  4. PARITY PLOT                          ║
    # ╚══════════════════════════════════════════╝
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    fig.patch.set_facecolor(BG)
    style_ax(ax)

    min_val = min(y_arr.min(), y_oof_pred.min())
    max_val = max(y_arr.max(), y_oof_pred.max())
    margin  = (max_val - min_val) * 0.05

    xy   = np.vstack([y_arr, y_oof_pred])
    kern = stats.gaussian_kde(xy)(xy)
    idx  = kern.argsort()

    sc = ax.scatter(
        y_arr[idx], y_oof_pred[idx],
        c=kern[idx], cmap="Spectral_r",
        s=55, alpha=0.88,
        edgecolors="#555555", linewidths=0.3, zorder=3
    )
    cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
    cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
    cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
    cbar.outline.set_edgecolor("#888888")

    ax.plot([min_val, max_val], [min_val, max_val],
            color="#333333", linewidth=1.8, linestyle="--",
            label="Ideal fit  (y = x)", zorder=4)

    band10 = (max_val - min_val) * 0.10
    ax.fill_between([min_val, max_val],
                    [min_val - band10, max_val - band10],
                    [min_val + band10, max_val + band10],
                    color=C2, alpha=0.07, label="±10% error band")

    band5 = (max_val - min_val) * 0.05
    ax.fill_between([min_val, max_val],
                    [min_val - band5, max_val - band5],
                    [min_val + band5, max_val + band5],
                    color=C3, alpha=0.09, label="±5% error band")

    metrics_txt = (f"R²   = {r2:.4f}\n"
                   f"RMSE = {rmse:.3f}\n"
                   f"MAE  = {mae:.3f}")
    ax.text(0.04, 0.97, metrics_txt,
            transform=ax.transAxes,
            fontsize=9.5, color=TEXT_COL,
            family="monospace", va="top",
            bbox=dict(boxstyle="round,pad=0.5",
                      facecolor="white", edgecolor="#333333",
                      alpha=0.92, linewidth=1.2))

    ax.set_xlim(min_val - margin, max_val + margin)
    ax.set_ylim(min_val - margin, max_val + margin)
    ax.set_xlabel(f"Actual {target_name}")
    ax.set_ylabel(f"Predicted {target_name}")
    ax.set_title(f"Parity Plot (CV) — {model_name}  |  {target_name}")
    ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
              framealpha=1, loc="lower right")

    fig.text(0.5, -0.02,
             "Colour encodes local point density  •  Predictions from 5-fold CV",
             ha="center", fontsize=8, color="#666666", style="italic")

    plt.tight_layout()
    save_fig(fig, f"parity_{model_name}_{target_name}.png".replace(" ", "_"))
    plt.show()

    # ╔══════════════════════════════════════════╗
    # ║  5. RESIDUAL DISTRIBUTION                ║
    # ╚══════════════════════════════════════════╝
    fig, ax = plt.subplots(figsize=(7.5, 5))
    fig.patch.set_facecolor(BG)
    style_ax(ax)

    n_bins = 25
    counts, bin_edges, patches = ax.hist(
        residuals, bins=n_bins,
        edgecolor="white", linewidth=0.6, zorder=3
    )

    # Left→right positional gradient — always visible
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    norm_pos    = (bin_centers - bin_centers.min()) / (
                   bin_centers.max() - bin_centers.min() + 1e-9)
    div_cmap    = LinearSegmentedColormap.from_list(
        "br_visible", [
            "#1a6faf", "#5ba3d0", "#a8d1e7",
            "#f4a97f", "#e05c3a", "#c0392b",
        ]
    )
    for patch, np_ in zip(patches, norm_pos):
        patch.set_facecolor(div_cmap(np_))
        patch.set_alpha(0.85)

    kde_x     = np.linspace(residuals.min(), residuals.max(), 400)
    kde_y     = stats.gaussian_kde(residuals)(kde_x)
    bin_width = bin_edges[1] - bin_edges[0]
    scale     = len(residuals) * bin_width

    ax.plot(kde_x, kde_y * scale, color=C2,
            linewidth=2.5, label="KDE", zorder=5)

    mu, sigma = residuals.mean(), residuals.std()
    normal_y  = stats.norm.pdf(kde_x, mu, sigma)
    ax.plot(kde_x, normal_y * scale, color=C3,
            linewidth=2.0, linestyle=":",
            label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

    sw_stat, sw_p = shapiro(residuals)
    ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
    ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
               alpha=0.9, label=f"Mean residual = {mu:.3f}")

    stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
                 f"p-value = {sw_p:.3f}\n"
                 f"Skewness = {stats.skew(residuals):.3f}\n"
                 f"Kurtosis = {stats.kurtosis(residuals):.3f}")
    ax.text(0.97, 0.97, stats_box,
            transform=ax.transAxes,
            fontsize=8.5, color=TEXT_COL,
            family="monospace", va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.5",
                      facecolor="#fff9e6", edgecolor="#ccaa00",
                      alpha=0.95, linewidth=1.1))

    ax.set_xlabel("Residual  (Actual − Predicted)")
    ax.set_ylabel("Frequency")
    ax.set_title(f"Residual Distribution — {model_name}  |  {target_name}")
    ax.legend(fontsize=9, facecolor="white",
              edgecolor="#cccccc", framealpha=1)

    fig.text(0.5, -0.02,
             "Blue = under-prediction  •  Red = over-prediction  •  "
             "Dotted = Gaussian reference",
             ha="center", fontsize=8, color="#666666", style="italic")

    plt.tight_layout()
    save_fig(fig, f"resdist_{model_name}_{target_name}.png".replace(" ", "_"))
    plt.show()

    # ╔══════════════════════════════════════════╗
    # ║  6. RESIDUALS VS PREDICTED               ║
    # ╚══════════════════════════════════════════╝
    fig, ax = plt.subplots(figsize=(7.5, 5))
    fig.patch.set_facecolor(BG)
    style_ax(ax)

    pos = residuals >= 0
    ax.scatter(y_oof_pred[pos],  residuals[pos],
               color=C1, alpha=0.75, s=45, zorder=3,
               edgecolors="white", linewidths=0.4,
               label=f"Positive residuals  (n={pos.sum()})")
    ax.scatter(y_oof_pred[~pos], residuals[~pos],
               color=C2, alpha=0.75, s=45, zorder=3,
               edgecolors="white", linewidths=0.4,
               label=f"Negative residuals  (n={(~pos).sum()})")

    ax.axhline(0, color="#333333", linewidth=1.8,
               linestyle="--", zorder=4, label="Zero line")

    std_r = residuals.std()
    ax.axhspan(-std_r,   std_r,   color=C1, alpha=0.07,
               label=f"±1σ  ({std_r:.2f})")
    ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
               label=f"±2σ  ({2*std_r:.2f})")

    for mult, ls in [(1, "-"), (2, "--")]:
        for sign in [1, -1]:
            ax.axhline(sign * mult * std_r, color="#aaaaaa",
                       linewidth=0.8, linestyle=ls, alpha=0.8)
            ax.text(y_oof_pred.max(), sign * mult * std_r,
                    f"  {'+' if sign>0 else '-'}{mult}σ",
                    va="center", fontsize=7.5, color="#666666")

    ax.set_xlabel(f"Predicted {target_name}")
    ax.set_ylabel("Residual  (Actual − Predicted)")
    ax.set_title(f"Residuals vs Predicted — {model_name}  |  {target_name}")
    ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
              framealpha=1, ncol=2)

    fig.text(0.5, -0.02,
             "Random scatter around zero indicates no systematic bias",
             ha="center", fontsize=8, color="#666666", style="italic")

    plt.tight_layout()
    save_fig(fig, f"resvspred_{model_name}_{target_name}.png".replace(" ", "_"))
    plt.show()

    # ╔══════════════════════════════════════════╗
    # ║  7. CV R² BOXPLOT                        ║
    # ╚══════════════════════════════════════════╝
    cv_r2_scores = []
    for train_idx, test_idx in kf.split(X):
        model = clone(pipeline)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[test_idx])
        cv_r2_scores.append(r2_score(y.iloc[test_idx], preds))

    cv_r2_scores = np.array(cv_r2_scores)

    fig, ax = plt.subplots(figsize=(6, 5.5))
    fig.patch.set_facecolor(BG)
    style_ax(ax, minor_ticks=False)

    bp = ax.boxplot(
        cv_r2_scores,
        patch_artist=True,
        widths=0.38,
        medianprops=dict(color=C2, linewidth=2.8),
        whiskerprops=dict(color=C1, linewidth=1.6, linestyle="--"),
        capprops=dict(color=C1, linewidth=2.2),
        flierprops=dict(marker="D", color=C4, markersize=5,
                        markerfacecolor=C4, alpha=0.8),
        boxprops=dict(linewidth=1.6),
    )
    bp["boxes"][0].set_facecolor("#dbeeff")
    bp["boxes"][0].set_edgecolor(C1)

    np.random.seed(42)
    jitter = np.random.normal(1, 0.045, size=len(cv_r2_scores))
    ax.scatter(jitter, cv_r2_scores, color=C1, alpha=0.65,
               s=45, zorder=4, label="Individual fold scores",
               edgecolors="white", linewidths=0.5)

    mean_r2 = cv_r2_scores.mean()
    std_r2  = cv_r2_scores.std()
    ax.axhline(mean_r2, color=C2, linewidth=1.8, linestyle="-.",
               zorder=3, label=f"Mean R² = {mean_r2:.3f}")

    stats_txt = (f"Mean  = {mean_r2:.3f}\n"
                 f"Std    = {std_r2:.3f}\n"
                 f"Min   = {cv_r2_scores.min():.3f}\n"
                 f"Max  = {cv_r2_scores.max():.3f}")
    ax.text(1.32, mean_r2, stats_txt,
            fontsize=8.5, color=TEXT_COL, family="monospace",
            va="center",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff9e6",
                      edgecolor="#ccaa00", alpha=0.95))

    ax.set_ylabel("R² Score")
    ax.set_title(f"5-Fold CV R² — {model_name}  |  {target_name}")
    ax.set_xticks([])
    ax.set_xlim(0.5, 1.8)
    ax.legend(fontsize=9, facecolor="white",
              edgecolor="#cccccc", framealpha=1)

    fig.text(0.5, -0.02,
             "Each point = one fold  •  Dashed = median  •  "
             "Dash-dot = mean",
             ha="center", fontsize=8, color="#666666", style="italic")

    plt.tight_layout()
    save_fig(fig, f"boxplot_{model_name}_{target_name}.png".replace(" ", "_"))
    plt.show()

    # ╔══════════════════════════════════════════╗
    # ║  8. DIAGNOSTICS SUMMARY                  ║
    # ╚══════════════════════════════════════════╝
    print("\n" + "═" * 50)
    print(f"   DIAGNOSTICS — {model_name} | {target_name}")
    print("═" * 50)
    print(f"  Pearson r              : {corr:.4f}")
    print(f"  OOF R²                 : {r2:.4f}")
    print(f"  OOF RMSE               : {rmse:.4f}")
    print(f"  OOF MAE                : {mae:.4f}")
    print(f"  Mean CV R²  (5-fold)   : {mean_r2:.4f}")
    print(f"  Std  CV R²  (5-fold)   : {cv_r2_scores.std():.4f}")
    print(f"  Residual Mean  (→ 0)   : {residuals.mean():.4f}")
    print(f"  Residual Std           : {residuals.std():.4f}")
    print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
    print("═" * 50)

    return {
        "rmse":         rmse,
        "mae":          mae,
        "r2":           r2,
        "correlation":  corr,
        "cv_r2_scores": cv_r2_scores,
        "mean_cv_r2":   mean_r2,
        "std_cv_r2":    cv_r2_scores.std(),
    }

In [ ]:
# ==============================
# XGBoost — ALL TARGETS (FIXED)
# ==============================

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

targets = [
    "CO2_conversion_efficiency_(%)",
    "DME_Selectivity_(%)",
    "Methanol_Selectivity_(%)",
    "CO_Selectivity_(%)"
]

results = []

for target in targets:

    print("\n" + "="*50)
    print(f"Running for: {target}")
    print("="*50)

    if target not in df.columns:
        print(f"Skipping {target}")
        continue

    # ✅ FIXED: Only drop target NaNs
    data = df[df[target].notna()].copy()

    if len(data) == 0:
        print(f"No data for {target}")
        continue

    X = data[numeric_features + categorical_features]
    y = data[target]

    xgb_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    xgb_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ])

    res = model_diagnostics(
        xgb_pipeline,
        X,
        y,
        model_name="XGBoost",
        target_name=target
    )

    results.append({
        "Target": target,
        "RMSE": res["rmse"],
        "Mean R2": res["mean_cv_r2"],
        "Std R2": res["std_cv_r2"],
        "Correlation": res["correlation"]
    })

results_df = pd.DataFrame(results)

print("\n===== FINAL SUMMARY =====")
print(results_df.sort_values(by="Mean R2", ascending=False))

results_df
results_df["Mean R2"] = results_df["Mean R2"].round(3)
results_df["RMSE"] = results_df["RMSE"].round(2)

In [ ]:
print("NaNs in X:", X.isna().sum().sum())

In [ ]:
# ============================================================
# HGBR (Numeric) — ALL TARGETS (ONE CELL, CLEAN)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline

targets = [
    ("CO2_conversion_efficiency_(%)", "CO₂ Conversion (%)"),
    ("DME_Selectivity_(%)", "DME Selectivity (%)"),
    ("Methanol_Selectivity_(%)", "Methanol Selectivity (%)"),
    ("CO_Selectivity_(%)", "CO Selectivity (%)")
]

results = []

for target, nice_name in targets:

    print("\n" + "="*50)
    print(f"Running for: {nice_name}")
    print("="*50)

    if target not in df.columns:
        print(f"Skipping {target}")
        continue

    # ✅ FIX: only drop target NaNs
    data = df[df[target].notna()].copy()

    X = data[numeric_features]
    y = data[target]

    # Optional safety check
    if X.isna().sum().sum() > 0:
        print("⚠️ NaNs found in numeric features, skipping...")
        continue

    # Fresh model each time
    hgbr_model = HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=300,
        random_state=42
    )

    hgbr_pipeline = Pipeline([
        ("model", hgbr_model)
    ])

    res = model_diagnostics(
        hgbr_pipeline,
        X,
        y,
        model_name="HGBR (Numeric)",
        target_name=nice_name
    )

    results.append({
        "Target": nice_name,
        "RMSE": res["rmse"],
        "Mean R2": res["mean_cv_r2"],
        "Std R2": res["std_cv_r2"]
    })

# ============================
# Summary
# ============================
import pandas as pd

results_df = pd.DataFrame(results)

print("\n===== SUMMARY =====")
print(results_df.sort_values(by="Mean R2", ascending=False))

results_df

In [ ]:
# ============================================================
# HGBR (Full Features) — ALL TARGETS (ONE CELL)
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline

targets = [
    ("CO2_conversion_efficiency_(%)", "CO₂ Conversion (%)"),
    ("DME_Selectivity_(%)", "DME Selectivity (%)"),
    ("Methanol_Selectivity_(%)", "Methanol Selectivity (%)"),
    ("CO_Selectivity_(%)", "CO Selectivity (%)")
]

results = []

for target, nice_name in targets:

    print("\n" + "="*50)
    print(f"Running for: {nice_name}")
    print("="*50)

    if target not in df.columns:
        print(f"Skipping {target}")
        continue

    # ✅ Only drop target NaNs
    data = df[df[target].notna()].copy()

    X = data[numeric_features + categorical_features]
    y = data[target]

    # Safety check
    if X.isna().sum().sum() > 0:
        print("⚠️ NaNs present — check preprocessing")
        continue

    # Model
    hgbr_full_model = HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=300,
        random_state=42
    )

    # IMPORTANT: use preprocessor for categorical handling
    hgbr_full_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", hgbr_full_model)
    ])

    res = model_diagnostics(
        hgbr_full_pipeline,
        X,
        y,
        model_name="HGBR (Full)",
        target_name=nice_name
    )

    results.append({
        "Target": nice_name,
        "RMSE": res["rmse"],
        "Mean R2": res["mean_cv_r2"],
        "Std R2": res["std_cv_r2"]
    })


# ============================
# Summary Table
# ============================
import pandas as pd

results_df = pd.DataFrame(results)

print("\n===== HGBR FULL SUMMARY =====")
print(results_df.sort_values(by="Mean R2", ascending=False))

results_df

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
et_model = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
et_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", et_model)
])


In [ ]:
# ============================================================
# ExtraTrees — ALL TARGETS (FINAL CLEAN)
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

targets = [
    ("CO2_conversion_efficiency_(%)", "CO₂ Conversion (%)"),
    ("DME_Selectivity_(%)", "DME Selectivity (%)"),
    ("Methanol_Selectivity_(%)", "Methanol Selectivity (%)"),
    ("CO_Selectivity_(%)", "CO Selectivity (%)")
]

results_et = []

for target, nice_name in targets:

    print("\n" + "="*50)
    print(f"Running ExtraTrees for: {nice_name}")
    print("="*50)

    if target not in df.columns:
        print(f"Skipping {target}")
        continue

    # ✅ FIX: only drop target NaNs
    data = df[df[target].notna()].copy()

    X = data[numeric_features + categorical_features]
    y = data[target]

    # Fresh model each time (IMPORTANT)
    et_model = ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    et_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", et_model)
    ])

    res = model_diagnostics(
        et_pipeline,
        X,
        y,
        model_name="Extra Trees",
        target_name=nice_name
    )

    results_et.append({
        "Target": nice_name,
        "Model": "ExtraTrees",
        "RMSE": res["rmse"],
        "Mean R2": res["mean_cv_r2"],
        "Std R2": res["std_cv_r2"]
    })


# ============================
# Final Table
# ============================
results_et_df = pd.DataFrame(results_et)

print("\n===== EXTRA TREES SUMMARY =====")
print(results_et_df.sort_values(by="Mean R2", ascending=False))

results_et_df

In [ ]:
# ============================================================
# BEST MODEL PER TARGET — FINAL RESULTS
# ============================================================

import numpy as np
import pandas as pd

best_results = []

# ------------------------------
# CO2 → ExtraTrees
# ------------------------------
best_results.append({
    "Target": "CO2 Conversion (%)",
    "Best Model": "ExtraTrees",
    "Mean CV R2": np.mean(cv_r2_scores_et_co2),
    "CV Std": np.std(cv_r2_scores_et_co2)
})

# ------------------------------
# DME → XGBoost
# ------------------------------
best_results.append({
    "Target": "DME Selectivity (%)",
    "Best Model": "XGBoost",
    "Mean CV R2": np.mean(cv_r2_scores_xgb_dme),
    "CV Std": np.std(cv_r2_scores_xgb_dme)
})

# ------------------------------
# Methanol → XGBoost
# ------------------------------
best_results.append({
    "Target": "Methanol Selectivity (%)",
    "Best Model": "XGBoost",
    "Mean CV R2": np.mean(cv_r2_scores_xgb_meoh),
    "CV Std": np.std(cv_r2_scores_xgb_meoh)
})

# ------------------------------
# CO → ExtraTrees
# ------------------------------
best_results.append({
    "Target": "CO Selectivity (%)",
    "Best Model": "ExtraTrees",
    "Mean CV R2": np.mean(cv_r2_scores_et_co),
    "CV Std": np.std(cv_r2_scores_et_co)
})

# ------------------------------
# Create DataFrame
# ------------------------------
best_models_df = pd.DataFrame(best_results)

# Round values
best_models_df["Mean CV R2"] = best_models_df["Mean CV R2"].round(3)
best_models_df["CV Std"] = best_models_df["CV Std"].round(3)

print("\n===== BEST MODEL SUMMARY =====")
print(best_models_df)

best_models_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

# ── Typography ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

# Matches your light-theme feature importance palette
COLOR_MAP = {
    "CO2 Conversion":       "#1a6faf",   # steel blue
    "DME Selectivity":      "#d4621a",   # burnt orange
    "Methanol Selectivity": "#1e8449",   # forest green
    "CO Selectivity":       "#6b3fa0",   # deep purple
}

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

# ── Prep data ─────────────────────────────────────────────────────────────────
plot_df  = best_models_df.sort_values("Mean CV R2", ascending=False).reset_index(drop=True)
targets  = plot_df["Target"].values
r2_vals  = plot_df["Mean CV R2"].values
colors   = [COLOR_MAP.get(t, "#1a6faf") for t in targets]

# Best model index
best_idx = int(np.argmax(r2_vals))

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6.5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)

for spine in ax.spines.values():
    spine.set_edgecolor("#333333")
    spine.set_linewidth(1.3)

ax.set_axisbelow(True)
ax.grid(True,  axis="y", color=GRID_COL,
        linewidth=0.7, linestyle="--", alpha=1.0)
ax.grid(False, axis="x")
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which="minor", length=3, color="#888888", direction="in")
ax.tick_params(which="major", length=5, color="#333333",
               direction="in", labelcolor=TEXT_COL)
ax.xaxis.label.set_color(TEXT_COL)
ax.yaxis.label.set_color(TEXT_COL)

# ── Bars ──────────────────────────────────────────────────────────────────────
bars = ax.bar(
    targets, r2_vals,
    color=colors,
    edgecolor="white",
    linewidth=0.8,
    width=0.52,
    zorder=3,
)

# Black border on best bar
ax.bar(
    [targets[best_idx]], [r2_vals[best_idx]],
    color="none",
    edgecolor="#000000",
    linewidth=2.2,
    width=0.52,
    zorder=4,
)

# Value labels above each bar
for bar, val, col in zip(bars, r2_vals, colors):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.012,
        f"{val:.4f}",
        ha="center", va="bottom",
        color=col, fontsize=10,
        fontweight="bold", family="monospace"
    )

# Model name labels inside each bar (if column exists)
if "Best Model" in plot_df.columns:
    for bar, model_name, val in zip(bars, plot_df["Best Model"].values, r2_vals):
        if val > 0.12:   # only label if bar tall enough
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val * 0.5,
                model_name,
                ha="center", va="center",
                color="white", fontsize=8.5,
                fontweight="bold", rotation=90,
                alpha=0.9,
            )

# ── Threshold reference lines ─────────────────────────────────────────────────
for thresh, label in [(0.95, "R² = 0.95"),
                       (0.90, "R² = 0.90"),
                       (0.80, "R² = 0.80")]:
    ax.axhline(thresh, color="#aaaaaa", linewidth=1.1,
               linestyle="--", zorder=2)
    ax.text(len(targets) - 0.42, thresh + 0.006,
            label, color="#888888", fontsize=8, style="italic")

# ── Best model callout ────────────────────────────────────────────────────────
best_target = targets[best_idx]
best_val    = r2_vals[best_idx]
ax.annotate(
    f"Best: {best_target}\nR² = {best_val:.4f}",
    xy=(best_idx, best_val),
    xytext=(best_idx + 0.6, best_val - 0.10),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.3,
                    connectionstyle="arc3,rad=-0.2"),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.92)
)

# Std error bars if available
if "Std CV R2" in plot_df.columns:
    ax.errorbar(
        targets, r2_vals,
        yerr=plot_df["Std CV R2"].values,
        fmt="none",
        ecolor="#555555",
        elinewidth=1.5,
        capsize=5,
        capthick=1.5,
        zorder=5,
    )

# ── Labels & title ────────────────────────────────────────────────────────────
ax.set_ylabel("Mean CV R²")
ax.set_xlabel("Target Variable")
ax.set_title("Best Model Performance per Target  |  Cross-Validation")
ax.set_ylim(0, 1.12)
ax.tick_params(axis="x", rotation=20)

fig.text(
    0.5, -0.02,
    "Error bars = ±1 std across CV folds  •  "
    "Black border = best overall model  •  "
    "Dashed lines = R² thresholds",
    ha="center", fontsize=8, color="#666666", style="italic"
)

plt.tight_layout()
save_fig(fig, "best_model_performance.png")
plt.show()

In [ ]:
# Hyperparameter Tuning
import time
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import r2_score
import numpy as np

def nested_cv_tuning(
    pipeline,
    param_dist,
    X,
    y,
    model_name,
    n_iter=20
):
    outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

    outer_scores = []

    total_start = time.time()

    print(f"\n🚀 Starting Nested CV for {model_name}")
    print("="*60)

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), 1):

        fold_start = time.time()

        print(f"\n🔁 Outer Fold {fold}/5")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Inner tuning
        search = RandomizedSearchCV(
            pipeline,
            param_distributions=param_dist,
            n_iter=n_iter,
            scoring="r2",
            cv=3,
            random_state=42,
            n_jobs=-1
        )

        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        preds = best_model.predict(X_test)

        score = r2_score(y_test, preds)
        outer_scores.append(score)

        fold_time = time.time() - fold_start
        elapsed = time.time() - total_start

        remaining = (elapsed / fold) * (5 - fold)

        print(f"✅ Fold R2: {score:.4f}")
        print(f"⏱ Fold Time: {fold_time:.1f}s")
        print(f"⏳ Estimated Remaining: {remaining:.1f}s")
        print(f"🎯 Best Params: {search.best_params_}")

    print("\n" + "="*60)
    print("🎉 NESTED CV COMPLETE")

    print(f"Mean R2: {np.mean(outer_scores):.4f}")
    print(f"Std R2: {np.std(outer_scores):.4f}")

    return {
        "scores": outer_scores,
        "mean_r2": np.mean(outer_scores),
        "std_r2": np.std(outer_scores)
    }

In [ ]:
# CO2: ExtraTrees Model
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline

target = "CO2_conversion_efficiency_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(random_state=42, n_jobs=-1))
])

param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

co2_tuned = nested_cv_tuning(
    pipeline,
    param_dist,
    X,
    y,
    model_name="ExtraTrees (CO2)",
    n_iter=15
)
co2_random_nested = {
    "mean_r2": co2_tuned["mean_r2"],
    "std_r2": co2_tuned["std_r2"],
    "params": "RandomSearch (varies per fold)"
}

In [ ]:
# DME: XGBoost Model - Randomsearch - Nested CV
from xgboost import XGBRegressor

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

param_dist = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [4, 6, 8],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0]
}

dme_tuned = nested_cv_tuning(
    pipeline,
    param_dist,
    X,
    y,
    model_name="XGBoost (DME)",
    n_iter=15
)
dme_random_nested = {
    "mean_r2": dme_tuned["mean_r2"],
    "std_r2": dme_tuned["std_r2"],
    "params": "RandomSearch (varies per fold)"
}

In [ ]:
# =========================================
# METHANOL: XGBoost Model
# (RandomizedSearch + Nested Cross Validation)
# =========================================

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

target = "Methanol_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [3, 5, 7, 10],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
    "model__gamma": [0, 0.1, 0.3],
    "model__reg_alpha": [0, 0.1, 1],
    "model__reg_lambda": [1, 2, 5]
}

# 🔥 RandomizedSearch + Nested CV
methanol_tuned = nested_cv_tuning(
    pipeline,
    param_dist,
    X,
    y,
    model_name="XGBoost (Methanol)",
    n_iter=20
)

methanol_random_nested = {
    "mean_r2": methanol_tuned["mean_r2"],
    "std_r2": methanol_tuned["std_r2"],
    "params": "RandomSearch (varies per fold)"
}

In [ ]:
# =========================================
# CO: ExtraTrees Model
# (RandomizedSearch + Nested Cross Validation)
# =========================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline

target = "CO_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(random_state=42, n_jobs=-1))
])

param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

# 🔥 RandomizedSearch + Nested CV
co_tuned = nested_cv_tuning(
    pipeline,
    param_dist,
    X,
    y,
    model_name="ExtraTrees (CO)",
    n_iter=15
)

co_random_nested = {
    "mean_r2": co_tuned["mean_r2"],
    "std_r2": co_tuned["std_r2"],
    "params": "RandomSearch (varies per fold)"
}

In [ ]:
!pip install optuna

In [ ]:
# =========================================
# CO2: ExtraTrees
# (Bayesian Optimization + Single CV)
# =========================================

import optuna
import time
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
import numpy as np

target = "CO2_conversion_efficiency_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

start_time = time.time()

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }

    model = ExtraTreesRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    )

    # 🔥 FIX: store full CV scores
    trial.set_user_attr("cv_scores", scores)

    mean_score = scores.mean()

    elapsed = time.time() - start_time
    print(f"Trial {trial.number} | R2: {mean_score:.4f} | Time: {elapsed:.1f}s")

    return mean_score


# 🔥 Optional (recommended for reproducibility)
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=30)

print("\n===== BEST RESULT =====")
print("Best R2:", study.best_value)
print("Best Params:", study.best_params)

# 🔥 FIX: extract CV distribution
best_trial = study.best_trial
cv_scores = best_trial.user_attrs["cv_scores"]

co2_optuna_single = {
    "mean_r2": np.mean(cv_scores),
    "std_r2": np.std(cv_scores),
    "params": study.best_params
}

In [ ]:
# ============================================================
# OPTUNA — XGBoost — DME Selectivity (Single CV)
# ============================================================

import optuna
import time
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import numpy as np

target = "DME_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

start_time = time.time()

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 700),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2)
    }

    model = XGBRegressor(
        **params,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    )

    # 🔥 FIX: store full CV scores
    trial.set_user_attr("cv_scores", scores)

    mean_score = scores.mean()

    elapsed = time.time() - start_time
    print(f"Trial {trial.number} | R2: {mean_score:.4f} | Time: {elapsed:.1f}s")

    return mean_score


# 🔥 Optional but recommended (reproducibility)
study_dme = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_dme.optimize(objective, n_trials=30)

print("\n===== BEST RESULT (DME) =====")
print("Best R2:", study_dme.best_value)
print("Best Params:", study_dme.best_params)

# 🔥 FIX: extract CV distribution
best_trial = study_dme.best_trial
cv_scores = best_trial.user_attrs["cv_scores"]

dme_optuna_single = {
    "mean_r2": np.mean(cv_scores),
    "std_r2": np.std(cv_scores),
    "params": study_dme.best_params
}

In [ ]:
# =========================================
# METHANOL: XGBoost
# (Bayesian Optimization + Single CV)
# =========================================

import optuna
import time
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
import numpy as np

target = "Methanol_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

start_time = time.time()

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 0.3),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 5)
    }

    model = XGBRegressor(
        **params,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    )

    # 🔥 ADD THIS (missing in your code)
    trial.set_user_attr("cv_scores", scores)

    mean_score = scores.mean()

    elapsed = time.time() - start_time
    print(f"Trial {trial.number} | R2: {mean_score:.4f} | Time: {elapsed:.1f}s")

    return mean_score


study_methanol = optuna.create_study(direction="maximize")

study_methanol.optimize(objective, n_trials=30)

print("\n===== BEST RESULT =====")
print("Best R2:", study_methanol.best_value)
print("Best Params:", study_methanol.best_params)

best_trial = study_methanol.best_trial
cv_scores = best_trial.user_attrs["cv_scores"]

methanol_optuna_single = {
    "mean_r2": np.mean(cv_scores),
    "std_r2": np.std(cv_scores),
    "params": study_methanol.best_params
}

In [ ]:
# =========================================
# CO: ExtraTrees
# (Bayesian Optimization + Single CV)
# =========================================

import optuna
import time
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
import numpy as np

target = "CO_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

start_time = time.time()

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }

    model = ExtraTreesRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    )

    # 🔥 IMPORTANT LINE
    trial.set_user_attr("cv_scores", scores)

    mean_score = scores.mean()

    elapsed = time.time() - start_time
    print(f"Trial {trial.number} | R2: {mean_score:.4f} | Time: {elapsed:.1f}s")

    return mean_score


study_co = optuna.create_study(direction="maximize")

study_co.optimize(objective, n_trials=30)

print("\n===== BEST RESULT =====")
print("Best R2:", study_co.best_value)
print("Best Params:", study_co.best_params)

best_trial = study_co.best_trial
cv_scores = best_trial.user_attrs["cv_scores"]

co_optuna_single = {
    "mean_r2": np.mean(cv_scores),
    "std_r2": np.std(cv_scores),
    "params": study_co.best_params
}

In [ ]:
# ============================================================
# CO2 — Optuna Bayesian Optimization + Nested CV (CLEAN)
# ============================================================
# ============================================================
# FINAL PIPELINE — CO2 (Nested CV + Final Model + Diagnostics)
# ============================================================

import numpy as np
import pandas as pd
import optuna
import time

from sklearn.model_selection import KFold, cross_val_predict, learning_curve
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from sklearn.model_selection import learning_curve
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats
from scipy.stats import shapiro

# ============================================================
# DATA
# ============================================================
target = "CO2_conversion_efficiency_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

print("\n===== DATA CHECK ====")
print("Shape:", X.shape)
print("NaNs:", X.isna().sum().sum(), y.isna().sum())

# ============================================================
# NESTED CV
# ============================================================
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

outer_scores = []
best_params_all_folds = []

print("\n🚀 Starting Nested Optuna")

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), 1):

    print(f"\n🔁 Fold {fold}/5")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # -------------------------
    # INNER OPTUNA
    # -------------------------
    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 500),
            "max_depth": trial.suggest_int("max_depth", 5, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        }

        model = ExtraTreesRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

        scores = []
        for tr_idx, val_idx in inner_cv.split(X_train):
            pipeline.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
            preds = pipeline.predict(X_train.iloc[val_idx])
            scores.append(r2_score(y_train.iloc[val_idx], preds))

        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=15)

    best_params = study.best_params
    best_params_all_folds.append(best_params)

    # -------------------------
    # OUTER TEST
    # -------------------------
    model = ExtraTreesRegressor(
        **best_params,
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    score = r2_score(y_test, preds)
    outer_scores.append(score)

    print(f"Fold R2: {score:.4f}")

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n===== NESTED CV RESULT ====")
print("Mean R2:", np.mean(outer_scores))
print("Std R2:", np.std(outer_scores))

# Store results in co2_optuna_nested
co2_optuna_nested = {
    "mean_r2": np.mean(outer_scores),
    "std_r2": np.std(outer_scores),
    "params": best_params_all_folds
}

# ============================================================
# FINAL MODEL (MOST COMMON PARAMS)
# ============================================================

best_params_final = Counter(
    [str(p) for p in best_params_all_folds]
).most_common(1)[0][0]

best_params_final = eval(best_params_final)

print("\n===== FINAL PARAMS ====")
print(best_params_final)

pipeline_final = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        **best_params_final,
        random_state=42,
        n_jobs=-1
    ))
])

# Train on FULL DATA
pipeline_final.fit(X, y)

# ============================================================
# 🔍 DATA LEAKAGE CHECK (VERY IMPORTANT)
# ============================================================

print("\n===== LEAKAGE CHECK ====")

# 1. CV vs Train comparison
y_pred_full = pipeline_final.predict(X)

train_r2 = r2_score(y, y_pred_full)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_oof = cross_val_predict(pipeline_final, X, y, cv=kf, n_jobs=-1)

cv_r2 = r2_score(y, y_oof)

print("Train R2:", train_r2)
print("CV R2:", cv_r2)
print("Gap:", train_r2 - cv_r2)

if (train_r2 - cv_r2) > 0.2:
    print("⚠️ Possible Overfitting / Leakage")
else:
    print("✅ No major leakage detected")

# ============================================================
# DIAGNOSTIC GRAPHS
# ============================================================

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ╔══════════════════════════════════════════╗
# ║  1. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline_final, X, y,
    cv=kf,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.75, val_rmse[-1] * 1.06),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — Final Pipeline")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

y_arr      = np.array(y)
residuals  = y_arr - y_oof

r2   = r2_score(y_arr, y_oof)
rmse = np.sqrt(mean_squared_error(y_arr, y_oof))
mae  = mean_absolute_error(y_arr, y_oof)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(y_arr.min(), y_oof.min())
max_val = max(y_arr.max(), y_oof.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_arr, y_oof])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    y_arr[idx], y_oof[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

band10 = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band10, max_val - band10],
                [min_val + band10, max_val + band10],
                color=C2, alpha=0.07, label="±10% error band")

band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

metrics_txt = (f"R²   = {r2:.4f}\n"
               f"RMSE = {rmse:.3f}\n"
               f"MAE  = {mae:.3f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual Values")
ax.set_ylabel("Predicted Values")
ax.set_title("Parity Plot — Final Pipeline  (OOF Predictions)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  OOF = out-of-fold CV predictions",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  3. RESIDUAL DISTRIBUTION                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos    = (bin_centers - bin_centers.min()) / (
               bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap    = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf", "#5ba3d0", "#a8d1e7",
        "#f4a97f", "#e05c3a", "#c0392b",
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

kde_x     = np.linspace(residuals.min(), residuals.max(), 400)
kde_y     = stats.gaussian_kde(residuals)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals) * bin_width

ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

mu, sigma = residuals.mean(), residuals.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

sw_stat, sw_p = shapiro(residuals)
ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — Final Pipeline")
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resdist_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. RESIDUAL VS PREDICTED                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals >= 0
ax.scatter(y_oof[pos],  residuals[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_oof[~pos], residuals[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals.std()
ax.axhspan(-std_r,   std_r,   color=C1, alpha=0.07,
           label=f"±1σ  ({std_r:.2f})")
ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
           label=f"±2σ  ({2*std_r:.2f})")

for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#aaaaaa",
                   linewidth=0.8, linestyle=ls, alpha=0.8)
        ax.text(y_oof.max(), sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5, color="#666666")

ax.set_xlabel("Predicted Values")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — Final Pipeline")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resvspred_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. FINAL METRICS SUMMARY                ║
# ╚══════════════════════════════════════════╝

print("\n" + "═" * 46)
print("   FINAL MODEL METRICS — Pipeline")
print("═" * 46)
print(f"  R²                     : {r2:.4f}")
print(f"  RMSE                   : {rmse:.4f}")
print(f"  MAE                    : {mae:.4f}")
print(f"  CV R²                  : {cv_r2:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals.mean():.4f}")
print(f"  Residual Std           : {residuals.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 46)

In [ ]:
# ============================================================
# NESTED OPTUNA — XGBoost — DME Selectivity (CLEAN)
# ============================================================

import optuna
import time
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# ------------------------------
# DATA
# ------------------------------
target = "DME_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

# ------------------------------
# OUTER CV
# ------------------------------
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

outer_scores = []
best_params_all_folds = []

start_time = time.time()

print("\n🚀 Starting NESTED OPTUNA — DME")
print("="*60)

# ============================================================
# OUTER LOOP
# ============================================================
for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), 1):

    print(f"\n🔁 Outer Fold {fold}/5")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ------------------------------
    # INNER OPTUNA
    # ------------------------------
    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 700),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 2),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2)
        }

        model = XGBRegressor(
            **params,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

        scores = []
        for tr_idx, val_idx in inner_cv.split(X_train):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

            pipeline.fit(X_tr, y_tr)
            preds = pipeline.predict(X_val)
            scores.append(r2_score(y_val, preds))

        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)

    best_params = study.best_params
    best_params_all_folds.append(best_params)

    # ------------------------------
    # TRAIN BEST MODEL (OUTER)
    # ------------------------------
    best_model = XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", best_model)
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    score = r2_score(y_test, preds)
    outer_scores.append(score)

    # ------------------------------
    # TIME TRACKING
    # ------------------------------
    elapsed = time.time() - start_time
    remaining = (elapsed / fold) * (5 - fold)

    print(f"✅ Fold R2: {score:.4f}")
    print(f"🎯 Best Params: {best_params}")
    print(f"⏳ Remaining Time: {remaining:.1f}s")


# ============================================================
# FINAL SAVE (CORRECT POSITION)
# ============================================================

outer_scores_dme_optuna = outer_scores.copy()

dme_optuna_nested = {
    "mean_r2": np.mean(outer_scores_dme_optuna),
    "std_r2": np.std(outer_scores_dme_optuna),
    "params": best_params_all_folds
}

print("\n" + "="*60)
print("🎉 NESTED OPTUNA COMPLETE — DME")
print(f"Mean R2: {dme_optuna_nested['mean_r2']:.4f}")
print(f"Std R2: {dme_optuna_nested['std_r2']:.4f}")

In [ ]:
# ============================================================
# FINAL PIPELINE — CO (Nested CV + Final Model + Diagnostics)
# ============================================================

import numpy as np
import pandas as pd
import optuna
import time

from sklearn.model_selection import KFold, cross_val_predict, learning_curve
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from scipy import stats
from scipy.stats import shapiro

# ============================================================
# DATA
# ============================================================
target = "CO_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

print("\n===== DATA CHECK ====")
print("Shape:", X.shape)
print("NaNs:", X.isna().sum().sum(), y.isna().sum())

# ============================================================
# NESTED CV
# ============================================================
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

outer_scores = []
best_params_all_folds = []

print("\n🚀 Starting Nested Optuna (CO)")

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), 1):

    print(f"\n🔁 Fold {fold}/5")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 500),
            "max_depth": trial.suggest_int("max_depth", 5, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        }

        model = ExtraTreesRegressor(**params, random_state=42, n_jobs=-1)

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

        scores = []
        for tr_idx, val_idx in inner_cv.split(X_train):
            pipeline.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
            preds = pipeline.predict(X_train.iloc[val_idx])
            scores.append(r2_score(y_train.iloc[val_idx], preds))

        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=15)

    best_params = study.best_params
    best_params_all_folds.append(best_params)

    model = ExtraTreesRegressor(**best_params, random_state=42, n_jobs=-1)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    score = r2_score(y_test, preds)
    outer_scores.append(score)

    print(f"Fold R2: {score:.4f}")

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n===== NESTED CV RESULT ====")
print("Mean R2:", np.mean(outer_scores))
print("Std R2:", np.std(outer_scores))

co_optuna_nested = {
    "mean_r2": np.mean(outer_scores),
    "std_r2": np.std(outer_scores),
    "params": best_params_all_folds
}

# ============================================================
# FINAL MODEL
# ============================================================

best_params_final = Counter([str(p) for p in best_params_all_folds]).most_common(1)[0][0]
best_params_final = eval(best_params_final)

print("\n===== FINAL PARAMS ====")
print(best_params_final)

pipeline_final = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(**best_params_final, random_state=42, n_jobs=-1))
])

pipeline_final.fit(X, y)

# ============================================================
# LEAKAGE CHECK
# ============================================================

print("\n===== LEAKAGE CHECK ====")

y_pred_full = pipeline_final.predict(X)
train_r2 = r2_score(y, y_pred_full)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_oof = cross_val_predict(pipeline_final, X, y, cv=kf, n_jobs=-1)

cv_r2 = r2_score(y, y_oof)

print("Train R2:", train_r2)
print("CV R2:", cv_r2)
print("Gap:", train_r2 - cv_r2)

# ============================================================
# (ALL GRAPHS SAME — NO CHANGE NEEDED)
# ============================================================
# ============================================================
# DIAGNOSTIC GRAPHS
# ============================================================

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ╔══════════════════════════════════════════╗
# ║  1. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline_final, X, y,
    cv=kf,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.75, val_rmse[-1] * 1.06),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — Final Pipeline")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

y_arr      = np.array(y)
residuals  = y_arr - y_oof

r2   = r2_score(y_arr, y_oof)
rmse = np.sqrt(mean_squared_error(y_arr, y_oof))
mae  = mean_absolute_error(y_arr, y_oof)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(y_arr.min(), y_oof.min())
max_val = max(y_arr.max(), y_oof.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_arr, y_oof])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    y_arr[idx], y_oof[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

band10 = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band10, max_val - band10],
                [min_val + band10, max_val + band10],
                color=C2, alpha=0.07, label="±10% error band")

band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

metrics_txt = (f"R²   = {r2:.4f}\n"
               f"RMSE = {rmse:.3f}\n"
               f"MAE  = {mae:.3f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual Values")
ax.set_ylabel("Predicted Values")
ax.set_title("Parity Plot — Final Pipeline  (OOF Predictions)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  OOF = out-of-fold CV predictions",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  3. RESIDUAL DISTRIBUTION                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos    = (bin_centers - bin_centers.min()) / (
               bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap    = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf", "#5ba3d0", "#a8d1e7",
        "#f4a97f", "#e05c3a", "#c0392b",
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

kde_x     = np.linspace(residuals.min(), residuals.max(), 400)
kde_y     = stats.gaussian_kde(residuals)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals) * bin_width

ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

mu, sigma = residuals.mean(), residuals.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

sw_stat, sw_p = shapiro(residuals)
ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — Final Pipeline")
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resdist_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. RESIDUAL VS PREDICTED                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals >= 0
ax.scatter(y_oof[pos],  residuals[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_oof[~pos], residuals[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals.std()
ax.axhspan(-std_r,   std_r,   color=C1, alpha=0.07,
           label=f"±1σ  ({std_r:.2f})")
ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
           label=f"±2σ  ({2*std_r:.2f})")

for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#aaaaaa",
                   linewidth=0.8, linestyle=ls, alpha=0.8)
        ax.text(y_oof.max(), sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5, color="#666666")

ax.set_xlabel("Predicted Values")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — Final Pipeline")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resvspred_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. FINAL METRICS SUMMARY                ║
# ╚══════════════════════════════════════════╝

print("\n" + "═" * 46)
print("   FINAL MODEL METRICS — Pipeline")
print("═" * 46)
print(f"  R²                     : {r2:.4f}")
print(f"  RMSE                   : {rmse:.4f}")
print(f"  MAE                    : {mae:.4f}")
print(f"  CV R²                  : {cv_r2:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals.mean():.4f}")
print(f"  Residual Std           : {residuals.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 46)

In [ ]:
# ============================================================
# FINAL PIPELINE — METHANOL (Nested CV + Final Model + Diagnostics)
# ============================================================

import numpy as np
import pandas as pd
import optuna
import time

from sklearn.model_selection import KFold, cross_val_predict, learning_curve
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from scipy import stats
from scipy.stats import shapiro

# ============================================================
# DATA
# ============================================================
target = "Methanol_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

print("\n===== DATA CHECK ====")
print("Shape:", X.shape)
print("NaNs:", X.isna().sum().sum(), y.isna().sum())

# ============================================================
# NESTED CV
# ============================================================
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

outer_scores = []
best_params_all_folds = []

print("\n🚀 Starting Nested Optuna (Methanol)")

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), 1):

    print(f"\n🔁 Fold {fold}/5")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 0.3),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
            "reg_lambda": trial.suggest_float("reg_lambda", 1, 5)
        }

        model = XGBRegressor(**params, random_state=42, n_jobs=-1, verbosity=0)

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

        scores = []
        for tr_idx, val_idx in inner_cv.split(X_train):
            pipeline.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
            preds = pipeline.predict(X_train.iloc[val_idx])
            scores.append(r2_score(y_train.iloc[val_idx], preds))

        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=15)

    best_params = study.best_params
    best_params_all_folds.append(best_params)

    model = XGBRegressor(**best_params, random_state=42, n_jobs=-1, verbosity=0)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    score = r2_score(y_test, preds)
    outer_scores.append(score)

    print(f"Fold R2: {score:.4f}")

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n===== NESTED CV RESULT ====")
print("Mean R2:", np.mean(outer_scores))
print("Std R2:", np.std(outer_scores))

methanol_optuna_nested = {
    "mean_r2": np.mean(outer_scores),
    "std_r2": np.std(outer_scores),
    "params": best_params_all_folds
}

# ============================================================
# FINAL MODEL
# ============================================================

best_params_final = Counter([str(p) for p in best_params_all_folds]).most_common(1)[0][0]
best_params_final = eval(best_params_final)

print("\n===== FINAL PARAMS ====")
print(best_params_final)

pipeline_final = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(**best_params_final, random_state=42, n_jobs=-1, verbosity=0))
])

pipeline_final.fit(X, y)

# ============================================================
# LEAKAGE CHECK
# ============================================================

print("\n===== LEAKAGE CHECK ====")

y_pred_full = pipeline_final.predict(X)
train_r2 = r2_score(y, y_pred_full)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_oof = cross_val_predict(pipeline_final, X, y, cv=kf, n_jobs=-1)

cv_r2 = r2_score(y, y_oof)

print("Train R2:", train_r2)
print("CV R2:", cv_r2)
print("Gap:", train_r2 - cv_r2)

# ============================================================
# GRAPHS → SAME AS CO2 (IMPORTANT)
# ============================================================
# ============================================================
# DIAGNOSTIC GRAPHS
# ============================================================

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ╔══════════════════════════════════════════╗
# ║  1. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline_final, X, y,
    cv=kf,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.75, val_rmse[-1] * 1.06),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — Final Pipeline")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

y_arr      = np.array(y)
residuals  = y_arr - y_oof

r2   = r2_score(y_arr, y_oof)
rmse = np.sqrt(mean_squared_error(y_arr, y_oof))
mae  = mean_absolute_error(y_arr, y_oof)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(y_arr.min(), y_oof.min())
max_val = max(y_arr.max(), y_oof.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_arr, y_oof])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    y_arr[idx], y_oof[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

band10 = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band10, max_val - band10],
                [min_val + band10, max_val + band10],
                color=C2, alpha=0.07, label="±10% error band")

band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

metrics_txt = (f"R²   = {r2:.4f}\n"
               f"RMSE = {rmse:.3f}\n"
               f"MAE  = {mae:.3f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual Values")
ax.set_ylabel("Predicted Values")
ax.set_title("Parity Plot — Final Pipeline  (OOF Predictions)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  OOF = out-of-fold CV predictions",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  3. RESIDUAL DISTRIBUTION                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos    = (bin_centers - bin_centers.min()) / (
               bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap    = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf", "#5ba3d0", "#a8d1e7",
        "#f4a97f", "#e05c3a", "#c0392b",
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

kde_x     = np.linspace(residuals.min(), residuals.max(), 400)
kde_y     = stats.gaussian_kde(residuals)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals) * bin_width

ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

mu, sigma = residuals.mean(), residuals.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

sw_stat, sw_p = shapiro(residuals)
ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — Final Pipeline")
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resdist_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. RESIDUAL VS PREDICTED                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals >= 0
ax.scatter(y_oof[pos],  residuals[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_oof[~pos], residuals[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals.std()
ax.axhspan(-std_r,   std_r,   color=C1, alpha=0.07,
           label=f"±1σ  ({std_r:.2f})")
ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
           label=f"±2σ  ({2*std_r:.2f})")

for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#aaaaaa",
                   linewidth=0.8, linestyle=ls, alpha=0.8)
        ax.text(y_oof.max(), sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5, color="#666666")

ax.set_xlabel("Predicted Values")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — Final Pipeline")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resvspred_final.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. FINAL METRICS SUMMARY                ║
# ╚══════════════════════════════════════════╝

print("\n" + "═" * 46)
print("   FINAL MODEL METRICS — Pipeline")
print("═" * 46)
print(f"  R²                     : {r2:.4f}")
print(f"  RMSE                   : {rmse:.4f}")
print(f"  MAE                    : {mae:.4f}")
print(f"  CV R²                  : {cv_r2:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals.mean():.4f}")
print(f"  Residual Std           : {residuals.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 46)

In [ ]:
import pandas as pd

# =========================================
# COMBINE ALL RESULTS
# =========================================

results_all = [

    # =======================
    # CO2
    # =======================
    {"Target": "CO2", "Method": "Random + Nested", **co2_random_nested},
    {"Target": "CO2", "Method": "Optuna Single", **co2_optuna_single},
    {"Target": "CO2", "Method": "Optuna Nested", **co2_optuna_nested},

    # =======================
    # DME
    # =======================
    {"Target": "DME", "Method": "Random + Nested", **dme_random_nested},
    {"Target": "DME", "Method": "Optuna Single", **dme_optuna_single},
    {"Target": "DME", "Method": "Optuna Nested", **dme_optuna_nested},

    # =======================
    # METHANOL
    # =======================
    {"Target": "Methanol", "Method": "Random + Nested", **methanol_random_nested},
    {"Target": "Methanol", "Method": "Optuna Single", **methanol_optuna_single},
    {"Target": "Methanol", "Method": "Optuna Nested", **methanol_optuna_nested},

    # =======================
    # CO
    # =======================
    {"Target": "CO", "Method": "Random + Nested", **co_random_nested},
    {"Target": "CO", "Method": "Optuna Single", **co_optuna_single},
    {"Target": "CO", "Method": "Optuna Nested", **co_optuna_nested},
]

results_df = pd.DataFrame(results_all)

# =========================================
# CLEAN FORMAT
# =========================================

results_df["mean_r2"] = results_df["mean_r2"].round(3)
results_df["std_r2"] = results_df["std_r2"].round(3)

# =========================================
# SORT (BEST FIRST PER TARGET)
# =========================================

results_df = results_df.sort_values(
    by=["Target", "mean_r2"],
    ascending=[True, False]
)

print("\n===== ALL RESULTS =====")
print(results_df)

In [ ]:
# =========================================
# BEST METHOD PER TARGET
# =========================================

best_df = (
    results_df.loc[
        results_df.groupby("Target")["mean_r2"].idxmax()
    ]
    .sort_values(by="Target")   # keeps order consistent
    .reset_index(drop=True)
)

print("\n===== BEST METHOD =====")
print(best_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

# ── Typography ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

COLOR_MAP = {
    "CO2 Conversion":       "#1a6faf",
    "DME Selectivity":      "#d4621a",
    "Methanol Selectivity": "#1e8449",
    "CO Selectivity":       "#6b3fa0",
}

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")

# ── Prep data ─────────────────────────────────────────────────────────────────
plot_df  = best_df.sort_values("mean_r2", ascending=False).reset_index(drop=True)
targets  = plot_df["Target"].values
r2_vals  = plot_df["mean_r2"].values
colors   = [COLOR_MAP.get(t, "#1a6faf") for t in targets]
best_idx = int(np.argmax(r2_vals))

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6.5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_edgecolor("#333333")
    spine.set_linewidth(1.3)

ax.grid(True,  axis="y", color=GRID_COL,
        linewidth=0.7, linestyle="--", alpha=1.0)
ax.grid(False, axis="x")
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which="minor", length=3, color="#888888", direction="in")
ax.tick_params(which="major", length=5, color="#333333",
               direction="in", labelcolor=TEXT_COL)
ax.xaxis.label.set_color(TEXT_COL)
ax.yaxis.label.set_color(TEXT_COL)

# ── Bars ──────────────────────────────────────────────────────────────────────
bars = ax.bar(
    targets, r2_vals,
    color=colors,
    edgecolor="white",
    linewidth=0.8,
    width=0.52,
    zorder=3,
)

# Black border on best bar
ax.bar(
    [targets[best_idx]], [r2_vals[best_idx]],
    color="none",
    edgecolor="#000000",
    linewidth=2.2,
    width=0.52,
    zorder=4,
)

# Value labels above bars — 4 decimal places, color-matched
for bar, val, col in zip(bars, r2_vals, colors):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.012,
        f"{val:.4f}",
        ha="center", va="bottom",
        color=col, fontsize=10,
        fontweight="bold", family="monospace"
    )

# Model name inside bar if column exists
if "best_model" in plot_df.columns:
    for bar, mname, val in zip(bars, plot_df["best_model"].values, r2_vals):
        if val > 0.12:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val * 0.48,
                mname,
                ha="center", va="center",
                color="white", fontsize=8.5,
                fontweight="bold", rotation=90,
                alpha=0.92,
            )

# ── Threshold lines ───────────────────────────────────────────────────────────
for thresh, label in [(0.95, "R² = 0.95"),
                       (0.90, "R² = 0.90"),
                       (0.80, "R² = 0.80")]:
    ax.axhline(thresh, color="#aaaaaa", linewidth=1.1,
               linestyle="--", zorder=2)
    ax.text(len(targets) - 0.42, thresh + 0.006,
            label, color="#888888",
            fontsize=8, style="italic")

# ── Best model callout ────────────────────────────────────────────────────────
ax.annotate(
    f"Best: {targets[best_idx]}\nR² = {r2_vals[best_idx]:.4f}",
    xy=(best_idx, r2_vals[best_idx]),
    xytext=(best_idx + 0.6, r2_vals[best_idx] - 0.10),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.3,
                    connectionstyle="arc3,rad=-0.2"),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.92)
)

# Std error bars if available
if "std_r2" in plot_df.columns:
    ax.errorbar(
        targets, r2_vals,
        yerr=plot_df["std_r2"].values,
        fmt="none",
        ecolor="#555555",
        elinewidth=1.5,
        capsize=5,
        capthick=1.5,
        zorder=5,
    )

# ── Labels & title ────────────────────────────────────────────────────────────
ax.set_ylabel("Mean R²")
ax.set_xlabel("Target Variable")
ax.set_title("Best Model Performance per Target  |  Cross-Validation")
ax.set_ylim(0, 1.12)
ax.tick_params(axis="x", rotation=15)

fig.text(
    0.5, -0.02,
    "Black border = best overall model  •  "
    "Dashed lines = R² thresholds  •  "
    "Error bars = ±1 std (if available)",
    ha="center", fontsize=8, color="#666666", style="italic"
)

plt.tight_layout()
save_fig(fig, "best_model_performance.png")
plt.show()

In [ ]:
# ============================================================
# OPTUNA — FULL DATA (NO CV) — CO2
# ============================================================

import optuna
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor

target = "CO2_conversion_efficiency_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

print("\n🚀 Running Optuna on FULL DATA (CO2)")

# ------------------------------------------------
# Objective (with CV inside → stable tuning)
# ------------------------------------------------
def objective_co2(trial):   # 🔥 renamed

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }

    model = ExtraTreesRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    pipeline_co2 = Pipeline([   # 🔥 renamed
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # internal CV for stability
    kf_co2 = KFold(n_splits=5, shuffle=True, random_state=42)   # 🔥 renamed

    scores = []

    for train_idx, val_idx in kf_co2.split(X):
        pipeline_co2.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = pipeline_co2.predict(X.iloc[val_idx])
        scores.append(r2_score(y.iloc[val_idx], preds))

    return np.mean(scores)

# ------------------------------------------------
# Run Optuna
# ------------------------------------------------
study_co2_full = optuna.create_study(direction="maximize")   # 🔥 renamed
study_co2_full.optimize(objective_co2, n_trials=25)

best_params_co2_full = study_co2_full.best_params   # 🔥 renamed

print("\n===== BEST PARAMS (FULL DATA — CO2) =====")
print(best_params_co2_full)

In [ ]:
# ============================================================
# EVALUATE BEST PARAMS (CLEAN CV) — CO2: ExtraTrees
# ============================================================

from sklearn.model_selection import cross_val_score, KFold

final_model_co2 = ExtraTreesRegressor(   # 🔥 renamed
    **best_params_co2_full,              # 🔥 fixed
    random_state=42,
    n_jobs=-1
)

pipeline_final_co2 = Pipeline([          # 🔥 renamed
    ("preprocessor", preprocessor),
    ("model", final_model_co2)
])

kf_co2 = KFold(n_splits=5, shuffle=True, random_state=42)   # 🔥 renamed

cv_scores_co2 = cross_val_score(        # 🔥 renamed
    pipeline_final_co2,
    X,
    y,
    cv=kf_co2,
    scoring="r2",
    n_jobs=-1
)

print("\n===== FINAL CV RESULT (FULL OPTUNA — CO2) =====")
print("Scores:", cv_scores_co2)
print("Mean R2:", np.mean(cv_scores_co2))
print("Std:", np.std(cv_scores_co2))

In [ ]:
co2_optuna_full = {
    "mean_r2": np.mean(cv_scores),
    "std_r2": np.std(cv_scores),
    "params": best_params_full
}

In [ ]:
# ============================================================
# FINAL DIAGNOSTICS — USING BEST PARAMS (FULL OPTUNA)
# PUBLICATION EDITION — LIGHT THEME
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from sklearn.model_selection import KFold, cross_val_predict, learning_curve
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
from scipy import stats
from scipy.stats import shapiro

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ── Rebuild final pipeline ────────────────────────────────────────────────────
final_model_co2 = ExtraTreesRegressor(
    **best_params_co2_full,
    random_state=42,
    n_jobs=-1
)

pipeline_final_co2 = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model_co2)
])

kf_co2 = KFold(n_splits=5, shuffle=True, random_state=42)


# ╔══════════════════════════════════════════╗
# ║  1. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline_final_co2, X, y,
    cv=kf_co2,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.75, val_rmse[-1] * 1.06),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — ExtraTrees  |  CO₂ Conversion (Final Model)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_final_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. OOF PREDICTIONS                      ║
# ╚══════════════════════════════════════════╝

y_pred_co2    = cross_val_predict(pipeline_final_co2, X, y, cv=kf_co2, n_jobs=-1)
y_arr         = np.array(y)
residuals_co2 = y_arr - y_pred_co2

r2_co2   = r2_score(y_arr, y_pred_co2)
rmse_co2 = np.sqrt(mean_squared_error(y_arr, y_pred_co2))
mae_co2  = mean_absolute_error(y_arr, y_pred_co2)
corr_co2 = np.corrcoef(y_arr, y_pred_co2)[0, 1]


# ╔══════════════════════════════════════════╗
# ║  3. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(y_arr.min(), y_pred_co2.min())
max_val = max(y_arr.max(), y_pred_co2.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_arr, y_pred_co2])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    y_arr[idx], y_pred_co2[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

band10 = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band10, max_val - band10],
                [min_val + band10, max_val + band10],
                color=C2, alpha=0.07, label="±10% error band")

band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

metrics_txt = (f"R²   = {r2_co2:.4f}\n"
               f"RMSE = {rmse_co2:.3f}\n"
               f"MAE  = {mae_co2:.3f}\n"
               f"r    = {corr_co2:.4f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual CO₂ Conversion (%)")
ax.set_ylabel("Predicted CO₂ Conversion (%)")
ax.set_title("Parity Plot — ExtraTrees  |  CO₂ Conversion (Final Model)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  OOF = out-of-fold CV predictions",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_final_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. RESIDUAL DISTRIBUTION                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals_co2, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos    = (bin_centers - bin_centers.min()) / (
               bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap    = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf", "#5ba3d0", "#a8d1e7",
        "#f4a97f", "#e05c3a", "#c0392b",
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

kde_x     = np.linspace(residuals_co2.min(), residuals_co2.max(), 400)
kde_y     = stats.gaussian_kde(residuals_co2)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals_co2) * bin_width

ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

mu, sigma = residuals_co2.mean(), residuals_co2.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

sw_stat, sw_p = shapiro(residuals_co2)
ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals_co2):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals_co2):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — ExtraTrees  |  CO₂ Conversion (Final Model)")
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resdist_final_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. RESIDUAL SCATTER                     ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals_co2 >= 0
ax.scatter(y_pred_co2[pos],  residuals_co2[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_pred_co2[~pos], residuals_co2[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals_co2.std()
ax.axhspan(-std_r,   std_r,   color=C1, alpha=0.07,
           label=f"±1σ  ({std_r:.2f})")
ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
           label=f"±2σ  ({2*std_r:.2f})")

for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#aaaaaa",
                   linewidth=0.8, linestyle=ls, alpha=0.8)
        ax.text(y_pred_co2.max(), sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5, color="#666666")

ax.set_xlabel("Predicted CO₂ Conversion (%)")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — ExtraTrees  |  CO₂ Conversion (Final Model)")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resvspred_final_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  6. CV R² BOXPLOT                        ║
# ╚══════════════════════════════════════════╝

cv_r2_scores_co2 = []
for train_idx, test_idx in kf_co2.split(X):
    pipeline_final_co2.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds = pipeline_final_co2.predict(X.iloc[test_idx])
    cv_r2_scores_co2.append(r2_score(y.iloc[test_idx], preds))

cv_r2_scores_co2 = np.array(cv_r2_scores_co2)

fig, ax = plt.subplots(figsize=(6, 5.5))
fig.patch.set_facecolor(BG)
style_ax(ax, minor_ticks=False)

bp = ax.boxplot(
    cv_r2_scores_co2,
    patch_artist=True,
    widths=0.38,
    medianprops=dict(color=C2, linewidth=2.8),
    whiskerprops=dict(color=C1, linewidth=1.6, linestyle="--"),
    capprops=dict(color=C1, linewidth=2.2),
    flierprops=dict(marker="D", color=C4, markersize=5,
                    markerfacecolor=C4, alpha=0.8),
    boxprops=dict(linewidth=1.6),
)
bp["boxes"][0].set_facecolor("#dbeeff")
bp["boxes"][0].set_edgecolor(C1)

np.random.seed(42)
jitter = np.random.normal(1, 0.045, size=len(cv_r2_scores_co2))
ax.scatter(jitter, cv_r2_scores_co2, color=C1, alpha=0.65,
           s=45, zorder=4, label="Individual fold scores",
           edgecolors="white", linewidths=0.5)

mean_r2 = cv_r2_scores_co2.mean()
std_r2  = cv_r2_scores_co2.std()
ax.axhline(mean_r2, color=C2, linewidth=1.8, linestyle="-.",
           zorder=3, label=f"Mean R² = {mean_r2:.3f}")

stats_txt = (f"Mean  = {mean_r2:.3f}\n"
             f"Std    = {std_r2:.3f}\n"
             f"Min   = {cv_r2_scores_co2.min():.3f}\n"
             f"Max  = {cv_r2_scores_co2.max():.3f}")
ax.text(1.32, mean_r2, stats_txt,
        fontsize=8.5, color=TEXT_COL, family="monospace",
        va="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff9e6",
                  edgecolor="#ccaa00", alpha=0.95))

ax.set_ylabel("R² Score")
ax.set_title("5-Fold CV R² — ExtraTrees  |  CO₂ Conversion (Final Model)")
ax.set_xticks([])
ax.set_xlim(0.5, 1.8)
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Each point = one fold  •  Dashed = median  •  Dash-dot = mean",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "boxplot_final_co2.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  7. FINAL METRICS SUMMARY                ║
# ╚══════════════════════════════════════════╝

print("\n" + "═" * 50)
print("   FINAL DIAGNOSTICS — ExtraTrees | CO₂ Conversion")
print("═" * 50)
print(f"  Pearson r              : {corr_co2:.4f}")
print(f"  OOF R²                 : {r2_co2:.4f}")
print(f"  OOF RMSE               : {rmse_co2:.4f}")
print(f"  OOF MAE                : {mae_co2:.4f}")
print(f"  Mean CV R²  (5-fold)   : {mean_r2:.4f}")
print(f"  Std  CV R²  (5-fold)   : {std_r2:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals_co2.mean():.4f}")
print(f"  Residual Std           : {residuals_co2.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 50)

In [ ]:
import joblib
import json
import numpy as np


# =========================================
# SAVE FULL PIPELINE
# =========================================

joblib.dump(pipeline_final_co2, "co2_model_pipeline.joblib")   # 🔥 fixed

print("✅ CO2 model saved successfully!")


# =========================================
# METADATA (IMPORTANT)
# =========================================

metadata_co2 = {   # 🔥 renamed
    "target": target,
    "features": list(X.columns),
    "model_type": "ExtraTrees",
    "mean_r2": float(np.mean(cv_scores_co2)),   # 🔥 fixed
    "std_r2": float(np.std(cv_scores_co2)),
    "params": best_params_co2_full              # 🔥 fixed
}

with open("co2_model_metadata.json", "w") as f:
    json.dump(metadata_co2, f, indent=4)

print("✅ CO2 metadata saved!")


# =========================================
# COMPLETE PACKAGE (PIPELINE + INFO)
# =========================================

joblib.dump({
    "pipeline": pipeline_final_co2,   # 🔥 fixed
    "features": list(X.columns),
    "params": best_params_co2_full    # 🔥 fixed
}, "co2_complete_package.joblib")

print("✅ Full CO2 package saved!")

In [ ]:
# ============================================================
# OPTUNA — FULL DATA (NO CV) — METHANOL
# ============================================================

import optuna
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

target = "Methanol_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

print("\n🚀 Running Optuna on FULL DATA (Methanol)")

# ------------------------------------------------
# Objective (with CV inside → stable tuning)
# ------------------------------------------------
def objective_methanol(trial):   # 🔥 renamed for clarity

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 0.3),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 5)
    }

    model = XGBRegressor(
        **params,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )

    pipeline_methanol = Pipeline([   # 🔥 renamed (important)
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    for train_idx, val_idx in kf.split(X):
        pipeline_methanol.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = pipeline_methanol.predict(X.iloc[val_idx])
        scores.append(r2_score(y.iloc[val_idx], preds))

    return np.mean(scores)

# ------------------------------------------------
# Run Optuna
# ------------------------------------------------
study_methanol_full = optuna.create_study(direction="maximize")
study_methanol_full.optimize(objective_methanol, n_trials=25)  # 🔥 updated

best_params_methanol_full = study_methanol_full.best_params

print("\n===== BEST PARAMS (FULL DATA — METHANOL) =====")
print(best_params_methanol_full)

In [ ]:
# ============================================================
# EVALUATE BEST PARAMS (CLEAN CV) — METHANOL: XGBoost
# ============================================================

from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import numpy as np

# use params from your full-data optuna
final_model_methanol = XGBRegressor(   # 🔥 renamed
    **best_params_methanol_full,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

pipeline_final_methanol = Pipeline([   # 🔥 renamed (VERY IMPORTANT)
    ("preprocessor", preprocessor),
    ("model", final_model_methanol)
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_methanol = cross_val_score(   # 🔥 renamed
    pipeline_final_methanol,
    X,
    y,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("\n===== FINAL CV RESULT (FULL OPTUNA — METHANOL) =====")
print("Scores:", cv_scores_methanol)
print("Mean R2:", np.mean(cv_scores_methanol))
print("Std:", np.std(cv_scores_methanol))

# ============================================================
# STORE RESULT (IMPORTANT FOR COMPARISON TABLE)
# ============================================================

methanol_optuna_full = {
    "mean_r2": np.mean(cv_scores_methanol),
    "std_r2": np.std(cv_scores_methanol),
    "params": best_params_methanol_full
}

In [ ]:
target = "Methanol_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

In [ ]:
# ============================================================
# FINAL DIAGNOSTICS — USING BEST PARAMS (FULL OPTUNA)-methanol
# PUBLICATION EDITION — LIGHT THEME
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from sklearn.model_selection import KFold, cross_val_predict, learning_curve
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from scipy import stats
from scipy.stats import shapiro

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ── Rebuild final pipeline ────────────────────────────────────────────────────
final_model_methanol = XGBRegressor(
    **best_params_methanol_full,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

pipeline_final_methanol = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model_methanol)
])

kf_methanol = KFold(n_splits=5, shuffle=True, random_state=42)


# ╔══════════════════════════════════════════╗
# ║  1. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline_final_methanol, X, y,
    cv=kf_methanol,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.75, val_rmse[-1] * 1.06),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — XGBoost  |  Methanol Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_final_methanol.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. OOF PREDICTIONS                      ║
# ╚══════════════════════════════════════════╝

y_pred_methanol    = cross_val_predict(pipeline_final_methanol, X, y, cv=kf_methanol, n_jobs=-1)
y_arr              = np.array(y)
residuals_methanol = y_arr - y_pred_methanol

r2_methanol   = r2_score(y_arr, y_pred_methanol)
rmse_methanol = np.sqrt(mean_squared_error(y_arr, y_pred_methanol))
mae_methanol  = mean_absolute_error(y_arr, y_pred_methanol)
corr_methanol = np.corrcoef(y_arr, y_pred_methanol)[0, 1]


# ╔══════════════════════════════════════════╗
# ║  3. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(y_arr.min(), y_pred_methanol.min())
max_val = max(y_arr.max(), y_pred_methanol.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_arr, y_pred_methanol])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    y_arr[idx], y_pred_methanol[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

band10 = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band10, max_val - band10],
                [min_val + band10, max_val + band10],
                color=C2, alpha=0.07, label="±10% error band")

band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

metrics_txt = (f"R²   = {r2_methanol:.4f}\n"
               f"RMSE = {rmse_methanol:.3f}\n"
               f"MAE  = {mae_methanol:.3f}\n"
               f"r    = {corr_methanol:.4f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual Methanol Selectivity (%)")
ax.set_ylabel("Predicted Methanol Selectivity (%)")
ax.set_title("Parity Plot — XGBoost  |  Methanol Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  OOF = out-of-fold CV predictions",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_final_methanol.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. RESIDUAL DISTRIBUTION                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals_methanol, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos    = (bin_centers - bin_centers.min()) / (
               bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap    = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf", "#5ba3d0", "#a8d1e7",
        "#f4a97f", "#e05c3a", "#c0392b",
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

kde_x     = np.linspace(residuals_methanol.min(), residuals_methanol.max(), 400)
kde_y     = stats.gaussian_kde(residuals_methanol)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals_methanol) * bin_width

ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

mu, sigma = residuals_methanol.mean(), residuals_methanol.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

sw_stat, sw_p = shapiro(residuals_methanol)
ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals_methanol):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals_methanol):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — XGBoost  |  Methanol Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resdist_final_methanol.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. RESIDUAL SCATTER                     ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals_methanol >= 0
ax.scatter(y_pred_methanol[pos],  residuals_methanol[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_pred_methanol[~pos], residuals_methanol[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals_methanol.std()
ax.axhspan(-std_r,   std_r,   color=C1, alpha=0.07,
           label=f"±1σ  ({std_r:.2f})")
ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
           label=f"±2σ  ({2*std_r:.2f})")

for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#aaaaaa",
                   linewidth=0.8, linestyle=ls, alpha=0.8)
        ax.text(y_pred_methanol.max(), sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5, color="#666666")

ax.set_xlabel("Predicted Methanol Selectivity (%)")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — XGBoost  |  Methanol Selectivity (Final Model)")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resvspred_final_methanol.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  6. CV R² BOXPLOT                        ║
# ╚══════════════════════════════════════════╝

cv_r2_scores_methanol = []
for train_idx, test_idx in kf_methanol.split(X):
    pipeline_final_methanol.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds = pipeline_final_methanol.predict(X.iloc[test_idx])
    cv_r2_scores_methanol.append(r2_score(y.iloc[test_idx], preds))

cv_r2_scores_methanol = np.array(cv_r2_scores_methanol)

fig, ax = plt.subplots(figsize=(6, 5.5))
fig.patch.set_facecolor(BG)
style_ax(ax, minor_ticks=False)

bp = ax.boxplot(
    cv_r2_scores_methanol,
    patch_artist=True,
    widths=0.38,
    medianprops=dict(color=C2, linewidth=2.8),
    whiskerprops=dict(color=C1, linewidth=1.6, linestyle="--"),
    capprops=dict(color=C1, linewidth=2.2),
    flierprops=dict(marker="D", color=C4, markersize=5,
                    markerfacecolor=C4, alpha=0.8),
    boxprops=dict(linewidth=1.6),
)
bp["boxes"][0].set_facecolor("#dbeeff")
bp["boxes"][0].set_edgecolor(C1)

np.random.seed(42)
jitter = np.random.normal(1, 0.045, size=len(cv_r2_scores_methanol))
ax.scatter(jitter, cv_r2_scores_methanol, color=C1, alpha=0.65,
           s=45, zorder=4, label="Individual fold scores",
           edgecolors="white", linewidths=0.5)

mean_r2 = cv_r2_scores_methanol.mean()
std_r2  = cv_r2_scores_methanol.std()
ax.axhline(mean_r2, color=C2, linewidth=1.8, linestyle="-.",
           zorder=3, label=f"Mean R² = {mean_r2:.3f}")

stats_txt = (f"Mean  = {mean_r2:.3f}\n"
             f"Std    = {std_r2:.3f}\n"
             f"Min   = {cv_r2_scores_methanol.min():.3f}\n"
             f"Max  = {cv_r2_scores_methanol.max():.3f}")
ax.text(1.32, mean_r2, stats_txt,
        fontsize=8.5, color=TEXT_COL, family="monospace",
        va="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff9e6",
                  edgecolor="#ccaa00", alpha=0.95))

ax.set_ylabel("R² Score")
ax.set_title("5-Fold CV R² — XGBoost  |  Methanol Selectivity (Final Model)")
ax.set_xticks([])
ax.set_xlim(0.5, 1.8)
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Each point = one fold  •  Dashed = median  •  Dash-dot = mean",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "boxplot_final_methanol.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  7. FINAL METRICS SUMMARY                ║
# ╚══════════════════════════════════════════╝

print("\n" + "═" * 50)
print("   FINAL DIAGNOSTICS — XGBoost | Methanol Selectivity")
print("═" * 50)
print(f"  Pearson r              : {corr_methanol:.4f}")
print(f"  OOF R²                 : {r2_methanol:.4f}")
print(f"  OOF RMSE               : {rmse_methanol:.4f}")
print(f"  OOF MAE                : {mae_methanol:.4f}")
print(f"  Mean CV R²  (5-fold)   : {mean_r2:.4f}")
print(f"  Std  CV R²  (5-fold)   : {std_r2:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals_methanol.mean():.4f}")
print(f"  Residual Std           : {residuals_methanol.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 50)

In [ ]:
import joblib
import json
import numpy as np   # 🔥 missing import


# =========================================
# SAVE FULL PIPELINE
# =========================================

joblib.dump(pipeline_final_methanol, "methanol_model_pipeline.joblib")   # 🔥 fixed

print("✅ Methanol model saved successfully!")


# =========================================
# METADATA (IMPORTANT)
# =========================================

metadata_methanol = {   # 🔥 renamed (avoid overwrite)
    "target": target,
    "features": list(X.columns),
    "model_type": "XGBoost",
    "mean_r2": float(np.mean(cv_scores_methanol)),   # 🔥 fixed
    "std_r2": float(np.std(cv_scores_methanol)),
    "params": best_params_methanol_full
}

with open("methanol_model_metadata.json", "w") as f:
    json.dump(metadata_methanol, f, indent=4)

print("✅ Methanol metadata saved!")


# =========================================
# COMPLETE PACKAGE (PIPELINE + INFO)
# =========================================

joblib.dump({
    "pipeline": pipeline_final_methanol,   # 🔥 fixed
    "features": list(X.columns),
    "params": best_params_methanol_full
}, "methanol_complete_package.joblib")

print("✅ Full Methanol package saved!")

In [ ]:
# ============================================================
# OPTUNA — FULL DATA (NO NESTED) — DME (XGBoost)
# ============================================================

import optuna
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

target = "DME_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

print("\n🚀 Running Optuna on FULL DATA — DME")

# ------------------------------------------------
# Objective (with CV inside)
# ------------------------------------------------
def objective_dme(trial):   # 🔥 renamed

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 700),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2)
    }

    model = XGBRegressor(
        **params,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    pipeline_dme = Pipeline([   # 🔥 renamed
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    kf_dme = KFold(n_splits=5, shuffle=True, random_state=42)   # 🔥 renamed

    scores = []
    for train_idx, val_idx in kf_dme.split(X):
        pipeline_dme.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = pipeline_dme.predict(X.iloc[val_idx])
        scores.append(r2_score(y.iloc[val_idx], preds))

    return np.mean(scores)

# ------------------------------------------------
# Run Optuna
# ------------------------------------------------
study_dme_full = optuna.create_study(direction="maximize")   # 🔥 renamed
study_dme_full.optimize(objective_dme, n_trials=30)

best_params_dme_full = study_dme_full.best_params

print("\n===== BEST PARAMS (DME FULL) =====")
print(best_params_dme_full)

In [ ]:
# ============================================================
# FINAL CV EVALUATION — DME
# ============================================================

from sklearn.model_selection import cross_val_score, KFold

final_model_dme = XGBRegressor(
    **best_params_dme_full,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

pipeline_final_dme = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model_dme)
])

kf_dme = KFold(n_splits=5, shuffle=True, random_state=42)   # 🔥 renamed

cv_scores_dme = cross_val_score(
    pipeline_final_dme,
    X,
    y,
    cv=kf_dme,   # 🔥 updated
    scoring="r2",
    n_jobs=-1
)

print("\n===== FINAL CV RESULT — DME =====")
print("Scores:", cv_scores_dme)
print("Mean R2:", np.mean(cv_scores_dme))
print("Std:", np.std(cv_scores_dme))

In [ ]:
dme_optuna_full = {
    "mean_r2": np.mean(cv_scores_dme),
    "std_r2": np.std(cv_scores_dme),
    "params": best_params_dme_full
}

In [ ]:
# ============================================================
# FINAL DIAGNOSTICS — DME (BEST MODEL)
# PUBLICATION EDITION — LIGHT THEME
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from sklearn.model_selection import cross_val_predict, learning_curve
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats
from scipy.stats import shapiro

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ╔══════════════════════════════════════════╗
# ║  1. OOF PREDICTIONS                      ║
# ╚══════════════════════════════════════════╝

y_pred_dme    = cross_val_predict(
    pipeline_final_dme, X, y, cv=kf_dme, n_jobs=-1
)
y_arr_dme     = np.array(y)
residuals_dme = y_arr_dme - y_pred_dme

r2_dme   = r2_score(y_arr_dme, y_pred_dme)
rmse_dme = np.sqrt(mean_squared_error(y_arr_dme, y_pred_dme))
mae_dme  = mean_absolute_error(y_arr_dme, y_pred_dme)
corr_dme = np.corrcoef(y_arr_dme, y_pred_dme)[0, 1]


# ╔══════════════════════════════════════════╗
# ║  2. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline_final_dme, X, y,
    cv=kf_dme,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.75, val_rmse[-1] * 1.06),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — XGBoost  |  DME Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_final_dme.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  3. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(y_arr_dme.min(), y_pred_dme.min())
max_val = max(y_arr_dme.max(), y_pred_dme.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_arr_dme, y_pred_dme])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    y_arr_dme[idx], y_pred_dme[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

band10 = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band10, max_val - band10],
                [min_val + band10, max_val + band10],
                color=C2, alpha=0.07, label="±10% error band")

band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

metrics_txt = (f"R²   = {r2_dme:.4f}\n"
               f"RMSE = {rmse_dme:.3f}\n"
               f"MAE  = {mae_dme:.3f}\n"
               f"r    = {corr_dme:.4f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual DME Selectivity (%)")
ax.set_ylabel("Predicted DME Selectivity (%)")
ax.set_title("Parity Plot — XGBoost  |  DME Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  OOF = out-of-fold CV predictions",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_final_dme.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. RESIDUAL DISTRIBUTION                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals_dme, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos    = (bin_centers - bin_centers.min()) / (
               bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap    = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf", "#5ba3d0", "#a8d1e7",
        "#f4a97f", "#e05c3a", "#c0392b",
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

kde_x     = np.linspace(residuals_dme.min(), residuals_dme.max(), 400)
kde_y     = stats.gaussian_kde(residuals_dme)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals_dme) * bin_width

ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

mu, sigma = residuals_dme.mean(), residuals_dme.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

sw_stat, sw_p = shapiro(residuals_dme)
ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals_dme):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals_dme):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — XGBoost  |  DME Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resdist_final_dme.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. RESIDUAL SCATTER                     ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals_dme >= 0
ax.scatter(y_pred_dme[pos],  residuals_dme[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_pred_dme[~pos], residuals_dme[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals_dme.std()
ax.axhspan(-std_r,   std_r,   color=C1, alpha=0.07,
           label=f"±1σ  ({std_r:.2f})")
ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
           label=f"±2σ  ({2*std_r:.2f})")

for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#aaaaaa",
                   linewidth=0.8, linestyle=ls, alpha=0.8)
        ax.text(y_pred_dme.max(), sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5, color="#666666")

ax.set_xlabel("Predicted DME Selectivity (%)")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — XGBoost  |  DME Selectivity (Final Model)")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resvspred_final_dme.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  6. FINAL METRICS SUMMARY                ║
# ╚══════════════════════════════════════════╝

print("\n" + "═" * 50)
print("   FINAL DIAGNOSTICS — XGBoost | DME Selectivity")
print("═" * 50)
print(f"  Pearson r              : {corr_dme:.4f}")
print(f"  OOF R²                 : {r2_dme:.4f}")
print(f"  OOF RMSE               : {rmse_dme:.4f}")
print(f"  OOF MAE                : {mae_dme:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals_dme.mean():.4f}")
print(f"  Residual Std           : {residuals_dme.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 50)

In [ ]:
import joblib
import json
import numpy as np

# =========================================
# 🔥 FINAL FIT BEFORE SAVING (CRITICAL)
# =========================================


# =========================================
# SAVE FULL PIPELINE
# =========================================

joblib.dump(pipeline_final_dme, "dme_model_pipeline.joblib")

print("✅ DME model saved successfully!")


# =========================================
# METADATA (IMPORTANT)
# =========================================

metadata_dme = {
    "target": target,
    "features": list(X.columns),
    "model_type": "XGBoost",
    "mean_r2": float(np.mean(cv_scores_dme)),
    "std_r2": float(np.std(cv_scores_dme)),
    "params": best_params_dme_full
}

with open("dme_model_metadata.json", "w") as f:
    json.dump(metadata_dme, f, indent=4)

print("✅ DME metadata saved!")


# =========================================
# COMPLETE PACKAGE
# =========================================

joblib.dump({
    "pipeline": pipeline_final_dme,
    "features": list(X.columns),
    "params": best_params_dme_full
}, "dme_complete_package.joblib")

print("✅ Full DME package saved!")

In [ ]:
# ============================================================
# OPTUNA — FULL DATA (NO CV) — CO
# ============================================================

import optuna
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor

target = "CO_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

print("\n🚀 Running Optuna on FULL DATA (CO)")

# ------------------------------------------------
# Objective (with CV inside → stable tuning)
# ------------------------------------------------
def objective_co(trial):   # 🔥 renamed

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }

    model = ExtraTreesRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    pipeline_co = Pipeline([   # 🔥 renamed
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    kf_co = KFold(n_splits=5, shuffle=True, random_state=42)   # 🔥 renamed

    scores = []

    for train_idx, val_idx in kf_co.split(X):
        pipeline_co.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = pipeline_co.predict(X.iloc[val_idx])
        scores.append(r2_score(y.iloc[val_idx], preds))

    return np.mean(scores)

# ------------------------------------------------
# Run Optuna
# ------------------------------------------------
study_co_full = optuna.create_study(direction="maximize")
study_co_full.optimize(objective_co, n_trials=25)   # 🔥 updated

best_params_co_full = study_co_full.best_params

print("\n===== BEST PARAMS (FULL DATA — CO) =====")
print(best_params_co_full)

In [ ]:
# ============================================================
# EVALUATE BEST PARAMS (CLEAN CV) — CO: ExtraTrees
# ============================================================

from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
import numpy as np

# rebuild model using best params
final_model_co = ExtraTreesRegressor(   # 🔥 renamed
    **best_params_co_full,
    random_state=42,
    n_jobs=-1
)

pipeline_final_co = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model_co)   # 🔥 updated
])

kf_co = KFold(n_splits=5, shuffle=True, random_state=42)   # 🔥 renamed

cv_scores_co = cross_val_score(
    pipeline_final_co,
    X,
    y,
    cv=kf_co,   # 🔥 updated
    scoring="r2",
    n_jobs=-1
)

print("\n===== FINAL CV RESULT (FULL OPTUNA — CO) =====")
print("Scores:", cv_scores_co)
print("Mean R2:", np.mean(cv_scores_co))
print("Std:", np.std(cv_scores_co))


# =========================================
# STORE RESULT (IMPORTANT)
# =========================================

co_optuna_full = {
    "mean_r2": np.mean(cv_scores_co),
    "std_r2": np.std(cv_scores_co),
    "params": best_params_co_full
}

In [ ]:
target = "CO_Selectivity_(%)"

data = df[df[target].notna()].copy()
X = data[numeric_features + categorical_features]
y = data[target]

In [ ]:
# ============================================================
# FINAL DIAGNOSTICS — CO SELECTIVITY (BEST PARAMS)
# PUBLICATION EDITION — LIGHT THEME
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from sklearn.model_selection import KFold, cross_val_predict, learning_curve
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
from scipy import stats
from scipy.stats import shapiro

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"
C2       = "#c0392b"
C3       = "#1e8449"
C4       = "#7d3c98"
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ── Build final pipeline ──────────────────────────────────────────────────────
final_model_co = ExtraTreesRegressor(
    **best_params_co_full,
    random_state=42,
    n_jobs=-1
)

pipeline_final_co = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model_co)
])

# Sanity check
print(type(pipeline_final_co.named_steps["model"]))

kf_co = KFold(n_splits=5, shuffle=True, random_state=42)


# ╔══════════════════════════════════════════╗
# ║  1. LEARNING CURVE                       ║
# ╚══════════════════════════════════════════╝

train_sizes, train_scores, val_scores = learning_curve(
    pipeline_final_co, X, y,
    cv=kf_co,
    scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_rmse     = np.sqrt(-train_scores.mean(axis=1))
val_rmse       = np.sqrt(-val_scores.mean(axis=1))
train_rmse_std = np.sqrt(-train_scores).std(axis=1)
val_rmse_std   = np.sqrt(-val_scores).std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

ax.plot(train_sizes, train_rmse, marker="o", color=C1,
        linewidth=2.2, markersize=8, label="Training RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                train_rmse - train_rmse_std,
                train_rmse + train_rmse_std,
                color=C1, alpha=0.13, label="Train ±1σ")

ax.plot(train_sizes, val_rmse, marker="s", color=C2,
        linewidth=2.2, markersize=8, label="CV RMSE",
        markerfacecolor="white", markeredgewidth=2, zorder=4)
ax.fill_between(train_sizes,
                val_rmse - val_rmse_std,
                val_rmse + val_rmse_std,
                color=C2, alpha=0.13, label="CV ±1σ")

gap = val_rmse[-1] - train_rmse[-1]
ax.annotate(
    f"Generalisation gap\n= {gap:.3f}",
    xy=(train_sizes[-1], (val_rmse[-1] + train_rmse[-1]) / 2),
    xytext=(train_sizes[-2] * 0.75, val_rmse[-1] * 1.06),
    fontsize=8.5, color=C4,
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.85)
)

ax.set_xlabel("Training Set Size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve — ExtraTrees  |  CO Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Shaded bands = ±1 standard deviation across CV folds",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "lc_final_co.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. OOF PREDICTIONS                      ║
# ╚══════════════════════════════════════════╝

y_pred_co  = cross_val_predict(pipeline_final_co, X, y, cv=kf_co, n_jobs=-1)
y_arr_co   = np.array(y)
residuals_co = y_arr_co - y_pred_co

r2_co   = r2_score(y_arr_co, y_pred_co)
rmse_co = np.sqrt(mean_squared_error(y_arr_co, y_pred_co))
mae_co  = mean_absolute_error(y_arr_co, y_pred_co)
corr_co = np.corrcoef(y_arr_co, y_pred_co)[0, 1]


# ╔══════════════════════════════════════════╗
# ║  3. PARITY PLOT                          ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor(BG)
style_ax(ax)

min_val = min(y_arr_co.min(), y_pred_co.min())
max_val = max(y_arr_co.max(), y_pred_co.max())
margin  = (max_val - min_val) * 0.05

xy   = np.vstack([y_arr_co, y_pred_co])
kern = stats.gaussian_kde(xy)(xy)
idx  = kern.argsort()

sc = ax.scatter(
    y_arr_co[idx], y_pred_co[idx],
    c=kern[idx], cmap="Spectral_r",
    s=55, alpha=0.88,
    edgecolors="#555555", linewidths=0.3, zorder=3
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Point Density (KDE)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

ax.plot([min_val, max_val], [min_val, max_val],
        color="#333333", linewidth=1.8, linestyle="--",
        label="Ideal fit  (y = x)", zorder=4)

band10 = (max_val - min_val) * 0.10
ax.fill_between([min_val, max_val],
                [min_val - band10, max_val - band10],
                [min_val + band10, max_val + band10],
                color=C2, alpha=0.07, label="±10% error band")

band5 = (max_val - min_val) * 0.05
ax.fill_between([min_val, max_val],
                [min_val - band5, max_val - band5],
                [min_val + band5, max_val + band5],
                color=C3, alpha=0.09, label="±5% error band")

metrics_txt = (f"R²   = {r2_co:.4f}\n"
               f"RMSE = {rmse_co:.3f}\n"
               f"MAE  = {mae_co:.3f}\n"
               f"r    = {corr_co:.4f}")
ax.text(0.04, 0.97, metrics_txt,
        transform=ax.transAxes,
        fontsize=9.5, color=TEXT_COL,
        family="monospace", va="top",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", edgecolor="#333333",
                  alpha=0.92, linewidth=1.2))

ax.set_xlim(min_val - margin, max_val + margin)
ax.set_ylim(min_val - margin, max_val + margin)
ax.set_xlabel("Actual CO Selectivity (%)")
ax.set_ylabel("Predicted CO Selectivity (%)")
ax.set_title("Parity Plot — ExtraTrees  |  CO Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Colour encodes local point density  •  OOF = out-of-fold CV predictions",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "parity_final_co.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. RESIDUAL DISTRIBUTION                ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

n_bins = 25
counts, bin_edges, patches = ax.hist(
    residuals_co, bins=n_bins,
    edgecolor="white", linewidth=0.6, zorder=3
)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
norm_pos    = (bin_centers - bin_centers.min()) / (
               bin_centers.max() - bin_centers.min() + 1e-9)
div_cmap    = LinearSegmentedColormap.from_list(
    "br_visible", [
        "#1a6faf", "#5ba3d0", "#a8d1e7",
        "#f4a97f", "#e05c3a", "#c0392b",
    ]
)
for patch, np_ in zip(patches, norm_pos):
    patch.set_facecolor(div_cmap(np_))
    patch.set_alpha(0.85)

kde_x     = np.linspace(residuals_co.min(), residuals_co.max(), 400)
kde_y     = stats.gaussian_kde(residuals_co)(kde_x)
bin_width = bin_edges[1] - bin_edges[0]
scale     = len(residuals_co) * bin_width

ax.plot(kde_x, kde_y * scale, color=C2,
        linewidth=2.5, label="KDE", zorder=5)

mu, sigma = residuals_co.mean(), residuals_co.std()
normal_y  = stats.norm.pdf(kde_x, mu, sigma)
ax.plot(kde_x, normal_y * scale, color=C3,
        linewidth=2.0, linestyle=":",
        label=f"Normal fit  (μ={mu:.2f}, σ={sigma:.2f})", zorder=5)

sw_stat, sw_p = shapiro(residuals_co)
ax.axvline(0,  color="#333333", linewidth=1.5, linestyle="--", alpha=0.8)
ax.axvline(mu, color=C4,        linewidth=1.5, linestyle="-.",
           alpha=0.9, label=f"Mean residual = {mu:.3f}")

stats_box = (f"Shapiro-Wilk W = {sw_stat:.3f}\n"
             f"p-value = {sw_p:.3f}\n"
             f"Skewness = {stats.skew(residuals_co):.3f}\n"
             f"Kurtosis = {stats.kurtosis(residuals_co):.3f}")
ax.text(0.97, 0.97, stats_box,
        transform=ax.transAxes,
        fontsize=8.5, color=TEXT_COL,
        family="monospace", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="#fff9e6", edgecolor="#ccaa00",
                  alpha=0.95, linewidth=1.1))

ax.set_xlabel("Residual  (Actual − Predicted)")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution — ExtraTrees  |  CO Selectivity (Final Model)")
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Blue = under-prediction  •  Red = over-prediction  •  "
         "Dotted = Gaussian reference",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resdist_final_co.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. RESIDUAL SCATTER                     ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor(BG)
style_ax(ax)

pos = residuals_co >= 0
ax.scatter(y_pred_co[pos],  residuals_co[pos],
           color=C1, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Positive residuals  (n={pos.sum()})")
ax.scatter(y_pred_co[~pos], residuals_co[~pos],
           color=C2, alpha=0.75, s=45, zorder=3,
           edgecolors="white", linewidths=0.4,
           label=f"Negative residuals  (n={(~pos).sum()})")

ax.axhline(0, color="#333333", linewidth=1.8,
           linestyle="--", zorder=4, label="Zero line")

std_r = residuals_co.std()
ax.axhspan(-std_r,   std_r,   color=C1, alpha=0.07,
           label=f"±1σ  ({std_r:.2f})")
ax.axhspan(-2*std_r, 2*std_r, color=C1, alpha=0.04,
           label=f"±2σ  ({2*std_r:.2f})")

for mult, ls in [(1, "-"), (2, "--")]:
    for sign in [1, -1]:
        ax.axhline(sign * mult * std_r, color="#aaaaaa",
                   linewidth=0.8, linestyle=ls, alpha=0.8)
        ax.text(y_pred_co.max(), sign * mult * std_r,
                f"  {'+' if sign>0 else '-'}{mult}σ",
                va="center", fontsize=7.5, color="#666666")

ax.set_xlabel("Predicted CO Selectivity (%)")
ax.set_ylabel("Residual  (Actual − Predicted)")
ax.set_title("Residual Plot — ExtraTrees  |  CO Selectivity (Final Model)")
ax.legend(fontsize=8.5, facecolor="white", edgecolor="#cccccc",
          framealpha=1, ncol=2)

fig.text(0.5, -0.02,
         "Random scatter around zero indicates no systematic bias",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "resvspred_final_co.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  6. CV R² BOXPLOT                        ║
# ╚══════════════════════════════════════════╝

cv_r2_scores_co = []
for train_idx, test_idx in kf_co.split(X):
    pipeline_final_co.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds = pipeline_final_co.predict(X.iloc[test_idx])
    cv_r2_scores_co.append(r2_score(y.iloc[test_idx], preds))

cv_r2_scores_co = np.array(cv_r2_scores_co)

fig, ax = plt.subplots(figsize=(6, 5.5))
fig.patch.set_facecolor(BG)
style_ax(ax, minor_ticks=False)

bp = ax.boxplot(
    cv_r2_scores_co,
    patch_artist=True,
    widths=0.38,
    medianprops=dict(color=C2, linewidth=2.8),
    whiskerprops=dict(color=C1, linewidth=1.6, linestyle="--"),
    capprops=dict(color=C1, linewidth=2.2),
    flierprops=dict(marker="D", color=C4, markersize=5,
                    markerfacecolor=C4, alpha=0.8),
    boxprops=dict(linewidth=1.6),
)
bp["boxes"][0].set_facecolor("#dbeeff")
bp["boxes"][0].set_edgecolor(C1)

np.random.seed(42)
jitter = np.random.normal(1, 0.045, size=len(cv_r2_scores_co))
ax.scatter(jitter, cv_r2_scores_co, color=C1, alpha=0.65,
           s=45, zorder=4, label="Individual fold scores",
           edgecolors="white", linewidths=0.5)

mean_r2_co = cv_r2_scores_co.mean()
std_r2_co  = cv_r2_scores_co.std()
ax.axhline(mean_r2_co, color=C2, linewidth=1.8, linestyle="-.",
           zorder=3, label=f"Mean R² = {mean_r2_co:.3f}")

stats_txt = (f"Mean  = {mean_r2_co:.3f}\n"
             f"Std    = {std_r2_co:.3f}\n"
             f"Min   = {cv_r2_scores_co.min():.3f}\n"
             f"Max  = {cv_r2_scores_co.max():.3f}")
ax.text(1.32, mean_r2_co, stats_txt,
        fontsize=8.5, color=TEXT_COL, family="monospace",
        va="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff9e6",
                  edgecolor="#ccaa00", alpha=0.95))

ax.set_ylabel("R² Score")
ax.set_title("5-Fold CV R² — ExtraTrees  |  CO Selectivity (Final Model)")
ax.set_xticks([])
ax.set_xlim(0.5, 1.8)
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Each point = one fold  •  Dashed = median  •  Dash-dot = mean",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "boxplot_final_co.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  7. FINAL METRICS SUMMARY                ║
# ╚══════════════════════════════════════════╝

print("\n" + "═" * 50)
print("   FINAL DIAGNOSTICS — ExtraTrees | CO Selectivity")
print("═" * 50)
print(f"  Pearson r              : {corr_co:.4f}")
print(f"  OOF R²                 : {r2_co:.4f}")
print(f"  OOF RMSE               : {rmse_co:.4f}")
print(f"  OOF MAE                : {mae_co:.4f}")
print(f"  Mean CV R²  (5-fold)   : {mean_r2_co:.4f}")
print(f"  Std  CV R²  (5-fold)   : {std_r2_co:.4f}")
print(f"  Residual Mean  (→ 0)   : {residuals_co.mean():.4f}")
print(f"  Residual Std           : {residuals_co.std():.4f}")
print(f"  Shapiro-Wilk p         : {sw_p:.4f}")
print("═" * 50)

In [ ]:
import joblib
import json
import numpy as np

# =========================================
# SAVE FULL PIPELINE
# =========================================

joblib.dump(pipeline_final_co, "co_model_pipeline.joblib")

print("✅ CO model saved successfully!")


# =========================================
# METADATA (IMPORTANT)
# =========================================

metadata_co = {
    "target": target,   # 🔥 safe (dynamic)
    "features": list(X.columns),
    "model_type": "ExtraTrees",
    "mean_r2": float(np.mean(cv_scores_co)),
    "std_r2": float(np.std(cv_scores_co)),
    "params": best_params_co_full
}

with open("co_model_metadata.json", "w") as f:
    json.dump(metadata_co, f, indent=4)

print("✅ CO metadata saved!")


# =========================================
# COMPLETE PACKAGE
# =========================================

joblib.dump({
    "pipeline": pipeline_final_co,
    "features": list(X.columns),
    "params": best_params_co_full
}, "co_complete_package.joblib")

print("✅ Full CO package saved!")

In [ ]:
import pandas as pd
import numpy as np

final_best = pd.DataFrame([
    {
        "Target": "CO2",
        "mean_r2": np.mean(cv_scores_co2),
        "std_r2": np.std(cv_scores_co2)
    },
    {
        "Target": "DME",
        "mean_r2": np.mean(cv_scores_dme),
        "std_r2": np.std(cv_scores_dme)
    },
    {
        "Target": "Methanol",
        "mean_r2": np.mean(cv_scores_methanol),
        "std_r2": np.std(cv_scores_methanol)
    },
    {
        "Target": "CO",
        "mean_r2": np.mean(cv_scores_co),
        "std_r2": np.std(cv_scores_co)
    }
])

# clean formatting
final_best["mean_r2"] = final_best["mean_r2"].round(3)
final_best["std_r2"] = final_best["std_r2"].round(3)

print("\n===== FINAL BEST RESULTS (CORRECT) =====")
print(final_best)

In [ ]:
import joblib

# Load all trained model packages
co2_package = joblib.load("co2_complete_package.joblib")
dme_package = joblib.load("dme_complete_package.joblib")
methanol_package = joblib.load("methanol_complete_package.joblib")
co_package = joblib.load("co_complete_package.joblib")

print("✅ All models loaded successfully")

# Optional: quick check
print("\n===== LOADED MODELS =====")
print("CO2 model:", type(co2_package["pipeline"].named_steps["model"]))
print("DME model:", type(dme_package["pipeline"].named_steps["model"]))
print("Methanol model:", type(methanol_package["pipeline"].named_steps["model"]))
print("CO model:", type(co_package["pipeline"].named_steps["model"]))

In [ ]:
print(co2_package.keys())
print(dme_package.keys())
print(methanol_package.keys())
print(co_package.keys())

In [ ]:
import joblib

co2_package = joblib.load("co2_complete_package.joblib")
dme_package = joblib.load("dme_complete_package.joblib")
methanol_package = joblib.load("methanol_complete_package.joblib")
co_package = joblib.load("co_complete_package.joblib")

# Extract pipelines
model_co2 = co2_package["pipeline"]
model_dme = dme_package["pipeline"]
model_methanol = methanol_package["pipeline"]
model_co = co_package["pipeline"]

print("✅ All models loaded successfully")

In [ ]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

final_model_dme = XGBRegressor(
    **best_params_dme_full,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

pipeline_final_dme = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model_dme)
])

In [ ]:
pipeline_final_dme.fit(X, y)

In [ ]:
import joblib

joblib.dump({
    "pipeline": pipeline_final_dme,
    "features": list(X.columns),
    "params": best_params_dme_full
}, "dme_complete_package.joblib")

print("✅ DME fully rebuilt and saved")

In [ ]:
dme_package = joblib.load("dme_complete_package.joblib")
model_dme = dme_package["pipeline"]

In [ ]:
sample = X.iloc[:2]

print("CO2:", model_co2.predict(sample))
print("DME:", model_dme.predict(sample))
print("Methanol:", model_methanol.predict(sample))
print("CO:", model_co.predict(sample))

In [ ]:
# ============================================================
# FASTAPI APP — FINAL (ALL 4 TARGETS)
# ============================================================

from fastapi import FastAPI
import joblib
import pandas as pd

app = FastAPI()

# =========================================
# LOAD ALL MODELS
# =========================================

co2_package = joblib.load("co2_complete_package.joblib")
dme_package = joblib.load("dme_complete_package.joblib")
methanol_package = joblib.load("methanol_complete_package.joblib")
co_package = joblib.load("co_complete_package.joblib")

co2_model = co2_package["pipeline"]
dme_model = dme_package["pipeline"]
methanol_model = methanol_package["pipeline"]
co_model = co_package["pipeline"]

# Use one consistent feature set (all same ideally)
features = co2_package["features"]

# =========================================
# HOME
# =========================================

@app.get("/")
def home():
    return {"message": "API Running 🚀"}

# =========================================
# PREDICT
# =========================================

@app.post("/predict")
def predict(data: dict):

    df = pd.DataFrame([data])

    # Feature alignment
    try:
        df = df[features]
    except:
        return {"error": "Feature mismatch. Check input keys."}

    # Predictions
    co2 = co2_model.predict(df)[0]
    dme = dme_model.predict(df)[0]
    methanol = methanol_model.predict(df)[0]
    co_val = co_model.predict(df)[0]

    return {
        "CO2_Conversion": float(co2),
        "DME_Selectivity": float(dme),
        "Methanol_Selectivity": float(methanol),
        "CO_Selectivity": float(co_val)
    }

In [ ]:
pip install fastapi uvicorn joblib pandas scikit-learn xgboost

In [ ]:
import os
print(os.listdir())

In [ ]:
%%writefile app.py
from fastapi import FastAPI
import joblib
import pandas as pd

app = FastAPI()

# =========================================
# LOAD ALL MODELS
# =========================================

co2_package = joblib.load("co2_complete_package.joblib")
dme_package = joblib.load("dme_complete_package.joblib")
methanol_package = joblib.load("methanol_complete_package.joblib")
co_package = joblib.load("co_complete_package.joblib")

co2_model = co2_package["pipeline"]
dme_model = dme_package["pipeline"]
methanol_model = methanol_package["pipeline"]
co_model = co_package["pipeline"]

# Use consistent features
features = co2_package["features"]

# =========================================
# HOME
# =========================================

@app.get("/")
def home():
    return {"message": "API Running 🚀"}

# =========================================
# PREDICT
# =========================================

@app.post("/predict")
def predict(data: dict):
    df = pd.DataFrame([data])

    try:
        df = df[features]
    except:
        return {"error": "Feature mismatch"}

    co2 = co2_model.predict(df)[0]
    dme = dme_model.predict(df)[0]
    methanol = methanol_model.predict(df)[0]
    co_val = co_model.predict(df)[0]

    return {
        "CO2_Conversion": float(co2),
        "DME_Selectivity": float(dme),
        "Methanol_Selectivity": float(methanol),
        "CO_Selectivity": float(co_val)
    }

In [ ]:
# !pip install gradio joblib pandas scikit-learn xgboost

In [ ]:
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

import gradio as gr
import joblib
import pandas as pd

# =========================================
# LOAD ALL MODELS
# =========================================

co2_package = joblib.load("co2_complete_package.joblib")
dme_package = joblib.load("dme_complete_package.joblib")
methanol_package = joblib.load("methanol_complete_package.joblib")
co_package = joblib.load("co_complete_package.joblib")

co2_model = co2_package["pipeline"]
dme_model = dme_package["pipeline"]
methanol_model = methanol_package["pipeline"]
co_model = co_package["pipeline"]

features = co2_package["features"]

# ------------------------------------------------
# DEFINE INPUT TYPES
# ------------------------------------------------

inputs = []

for f in features:
    if f in numeric_features:
        inputs.append(gr.Number(label=f))
    else:
        choices = df[f].dropna().unique().tolist()
        inputs.append(gr.Dropdown(choices=choices, label=f))

# ------------------------------------------------
# PREDICTION FUNCTION
# ------------------------------------------------

def predict_fn(*inputs_values):

    data = dict(zip(features, inputs_values))
    df_input = pd.DataFrame([data])

    co2 = co2_model.predict(df_input)[0]
    dme = dme_model.predict(df_input)[0]
    methanol = methanol_model.predict(df_input)[0]
    co_val = co_model.predict(df_input)[0]

    return co2, dme, methanol, co_val

# ------------------------------------------------
# INTERFACE
# ------------------------------------------------

app = gr.Interface(
    fn=predict_fn,
    inputs=inputs,
    outputs=[
        gr.Number(label="CO2 Conversion"),
        gr.Number(label="DME Selectivity"),
        gr.Number(label="Methanol Selectivity"),
        gr.Number(label="CO Selectivity")
    ],
    title="CO2 → DME Multi-Output Predictor"
)

app.launch(share=True)

In [ ]:
# !pip install pymoo

In [ ]:
import os

files = os.listdir()
for f in files:
    print(f)

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.optimize import minimize
from pymoo.termination import get_termination

# =========================================
# LOAD ALL PACKAGES (ONLY ONCE)
# =========================================

co2_pkg = joblib.load("co2_complete_package.joblib")
dme_pkg = joblib.load("dme_complete_package.joblib")
methanol_pkg = joblib.load("methanol_complete_package.joblib")
co_pkg = joblib.load("co_complete_package.joblib")

# Extract pipelines
co2_pipeline = co2_pkg["pipeline"]
dme_pipeline = dme_pkg["pipeline"]
methanol_pipeline = methanol_pkg["pipeline"]
co_pipeline = co_pkg["pipeline"]

# Features (same for all)
features = co2_pkg["features"]

print("✅ All models loaded successfully")

# =========================================
# TEST INPUT
# =========================================

test_df = df.iloc[[0]][features]

print("\n===== TEST PREDICTIONS =====")
print("CO2:", co2_pipeline.predict(test_df))
print("DME:", dme_pipeline.predict(test_df))
print("Methanol:", methanol_pipeline.predict(test_df))
print("CO:", co_pipeline.predict(test_df))

In [ ]:
from itertools import product
import numpy as np

# -----------------------------
# 1. CATEGORICAL VALUES (KEEP ALL INCLUDING UNKNOWN)
# -----------------------------
selected_cat_cols = [
    "acid_component_type",
    "Integration_method",
    "Dimentionality_of_Zeolite"
]

cat_values = {}

for col in selected_cat_cols:
    # ✅ keep all values including "Unknown"
    values = df[col].dropna().unique().tolist()
    cat_values[col] = values


# -----------------------------
# 2. ALL COMBINATIONS
# -----------------------------
all_combinations = list(product(
    cat_values["acid_component_type"],
    cat_values["Integration_method"],
    cat_values["Dimentionality_of_Zeolite"]
))

combination_dicts = []

for comb in all_combinations:
    combination_dicts.append({
        "acid_component_type": comb[0],
        "Integration_method": comb[1],
        "Dimentionality_of_Zeolite": comb[2]
    })

valid_combinations = combination_dicts.copy()

print("Total combinations:", len(valid_combinations))


# -----------------------------
# 3. NUMERICAL COLUMNS
# -----------------------------
num_cols = [col for col in features if col not in selected_cat_cols]

X_full = df[num_cols]

xl = X_full.min().values
xu = X_full.max().values


# -----------------------------
# 4. DEFINE INTEGER INDEXES
# -----------------------------
integer_cols = [
    "metal_1_atomic",
    "metal_2_atomic",
    "metal_3_atomic",
    "metal_4_atomic"
]

int_indices = [num_cols.index(col) for col in integer_cols]

In [ ]:
# valid_combinations = valid_combinations[:2940]

In [ ]:
# =========================================
# CHECK UNIQUE CATEGORIES (DETAILED)
# =========================================

selected_cat_cols = [
    "acid_component_type",
    "Integration_method",
    "Dimentionality_of_Zeolite"
]

for col in selected_cat_cols:
    print(f"\n===== {col} =====")

    # unique values
    values = df[col].unique()

    print("Total unique:", len(values))
    print("Values:")
    print(values)

    # frequency
    print("\nTop frequencies:")
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
results_all = []

for i, fixed_cat in enumerate(valid_combinations):

    print(f"\nRunning {i+1}/{len(valid_combinations)}")
    print(fixed_cat)

    class CO2_DME_Problem(Problem):

        def __init__(self):
            super().__init__(
                n_var=len(num_cols),
                n_obj=2,
                xl=xl,
                xu=xu
            )

        def _evaluate(self, X, out, *args, **kwargs):

            results = []

            for row in X:

                input_dict = {}

                # 🔥 FIX: enforce atomic numbers
                for j, col in enumerate(num_cols):

                    if j in int_indices:
                        val = int(round(row[j]))        # round
                        val = max(0, min(val, 118))     # clamp to [0,118]
                        input_dict[col] = val
                    else:
                        input_dict[col] = row[j]

                # categorical
                for k, v in fixed_cat.items():
                    input_dict[k] = v

                # correct feature order
                input_df = pd.DataFrame([input_dict])[features]

                co2_pred = co2_pipeline.predict(input_df)[0]
                dme_pred = dme_pipeline.predict(input_df)[0]

                results.append([-co2_pred, -dme_pred])

            out["F"] = np.array(results)

    problem = CO2_DME_Problem()

    res = minimize(
        problem,
        SPEA2(pop_size=20),
        get_termination("n_gen", 10),
        seed=42,
        verbose=False
    )

    co2_opt = -res.F[:, 0]
    dme_opt = -res.F[:, 1]

    best_idx = np.argmax(co2_opt + dme_opt)
    best_X = res.X[best_idx]

    best_input = {}

    # 🔥 FIX AGAIN (important)
    for j, col in enumerate(num_cols):

        if j in int_indices:
            val = int(round(best_X[j]))
            val = max(1, min(val, 118))
            best_input[col] = val
        else:
            best_input[col] = best_X[j]

    for k, v in fixed_cat.items():
        best_input[k] = v

    results_all.append({
        "combination": fixed_cat,
        "best_co2": co2_opt[best_idx],
        "best_dme": dme_opt[best_idx],
        "best_input": best_input
    })

print("\nTOTAL RESULTS:", len(results_all))

In [ ]:
results_df = pd.DataFrame(results_all)

# expand input dict into columns
input_expanded = results_df["best_input"].apply(pd.Series)

final_df = pd.concat([results_df.drop(columns=["best_input"]), input_expanded], axis=1)

final_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"   # steel blue
C2       = "#c0392b"   # brick red
C3       = "#1e8449"   # forest green
C4       = "#7d3c98"   # purple
C5       = "#d4621a"   # burnt orange
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ── Find optimal point ────────────────────────────────────────────────────────
best_idx = np.argmax(results_df["best_co2"] + results_df["best_dme"])
best_co2 = results_df.iloc[best_idx]["best_co2"]
best_dme = results_df.iloc[best_idx]["best_dme"]


# ╔══════════════════════════════════════════╗
# ║  1. PARETO SCATTER PLOT                  ║
# ╚══════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor(BG)
style_ax(ax)

# Color points by combined score
scores     = results_df["best_co2"] + results_df["best_dme"]
norm_score = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
pt_cmap    = plt.cm.get_cmap("RdYlBu")
pt_colors  = [pt_cmap(0.15 + 0.7 * s) for s in norm_score]

sc = ax.scatter(
    results_df["best_co2"],
    results_df["best_dme"],
    c=scores, cmap="RdYlBu",
    s=60, alpha=0.80,
    edgecolors="#555555", linewidths=0.4,
    zorder=3, label="Pareto solutions"
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar.set_label("Combined Score  (CO₂ + DME)", color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

# Optimal point
ax.scatter(
    best_co2, best_dme,
    s=220, marker="*",
    color=C2, edgecolors="#333333",
    linewidths=0.8, zorder=5,
    label=f"Optimal  (CO₂={best_co2:.2f}, DME={best_dme:.2f})"
)

# Dashed crosshair at optimal
ax.axvline(best_co2, color=C2, linewidth=1.0,
           linestyle=":", alpha=0.6, zorder=2)
ax.axhline(best_dme, color=C2, linewidth=1.0,
           linestyle=":", alpha=0.6, zorder=2)

# Annotation
ax.annotate(
    f"Optimal\nCO₂ = {best_co2:.2f}\nDME = {best_dme:.2f}",
    xy=(best_co2, best_dme),
    xytext=(best_co2 * 0.85, best_dme * 1.06),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2,
                    connectionstyle="arc3,rad=-0.2"),
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.92)
)

ax.set_xlabel("Best CO₂ Conversion (%)")
ax.set_ylabel("Best DME Selectivity (%)")
ax.set_title("SPEA2 Multi-Objective Optimisation  |  Pareto Trade-off")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1, loc="lower right")

fig.text(0.5, -0.02,
         "Each point = one Pareto solution  •  "
         "Colour = combined objective score  •  "
         "★ = balanced optimum",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "pareto_scatter.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. PARETO FRONT LINE                    ║
# ╚══════════════════════════════════════════╝

sorted_df = results_df.sort_values(by="best_co2").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor(BG)
style_ax(ax)

# Shade area under Pareto front
ax.fill_between(
    sorted_df["best_co2"],
    sorted_df["best_dme"],
    alpha=0.08, color=C1,
    label="Pareto feasible region"
)

# Line + markers colored by position
n_pts      = len(sorted_df)
line_norm  = np.linspace(0, 1, n_pts)
line_cmap  = plt.cm.get_cmap("RdYlBu")

for i in range(n_pts - 1):
    ax.plot(
        sorted_df["best_co2"].iloc[i:i+2],
        sorted_df["best_dme"].iloc[i:i+2],
        color=line_cmap(0.15 + 0.7 * line_norm[i]),
        linewidth=2.2, zorder=3
    )

sc2 = ax.scatter(
    sorted_df["best_co2"],
    sorted_df["best_dme"],
    c=line_norm, cmap="RdYlBu",
    s=55, zorder=4,
    edgecolors="#333333", linewidths=0.4,
)
cbar2 = fig.colorbar(sc2, ax=ax, fraction=0.04, pad=0.03, aspect=28)
cbar2.set_label("Position along Pareto front", color=TEXT_COL, fontsize=9)
cbar2.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar2.outline.set_edgecolor("#888888")

# Optimal point on front
ax.scatter(
    best_co2, best_dme,
    s=220, marker="*",
    color=C2, edgecolors="#333333",
    linewidths=0.8, zorder=5,
    label="Balanced optimum"
)

ax.set_xlabel("CO₂ Conversion (%)")
ax.set_ylabel("DME Selectivity (%)")
ax.set_title("Pareto Front — Trade-off Curve  |  SPEA2 Optimisation")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc",
          framealpha=1)

fig.text(0.5, -0.02,
         "Trade-off: increasing CO₂ conversion generally reduces DME selectivity",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "pareto_front.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  3. HISTOGRAMS — SIDE BY SIDE            ║
# ╚══════════════════════════════════════════╝

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor(BG)

for ax, col, data, label, color in [
    (ax1, "best_co2", results_df["best_co2"],
     "CO₂ Conversion (%)", C1),
    (ax2, "best_dme", results_df["best_dme"],
     "DME Selectivity (%)", C5),
]:
    style_ax(ax)

    n_bins = 15
    counts, bin_edges, patches = ax.hist(
        data, bins=n_bins,
        edgecolor="white", linewidth=0.6, zorder=3
    )

    # Gradient fill light → full color
    norm_b = np.linspace(0, 1, n_bins)
    base_r = int(color[1:3], 16) / 255
    base_g = int(color[3:5], 16) / 255
    base_b = int(color[5:7], 16) / 255
    for patch, nb in zip(patches, norm_b):
        patch.set_facecolor((
            1 - nb * (1 - base_r),
            1 - nb * (1 - base_g),
            1 - nb * (1 - base_b),
        ))
        patch.set_alpha(0.88)

    # KDE overlay
    from scipy import stats as sp_stats
    kde_x = np.linspace(data.min(), data.max(), 300)
    kde_y = sp_stats.gaussian_kde(data)(kde_x)
    bw    = bin_edges[1] - bin_edges[0]
    ax.plot(kde_x, kde_y * len(data) * bw,
            color=C2, linewidth=2.2, label="KDE", zorder=4)

    # Mean & median lines
    mu  = data.mean()
    med = data.median()
    ax.axvline(mu,  color=C4, linewidth=1.6, linestyle="-.",
               label=f"Mean = {mu:.2f}")
    ax.axvline(med, color=C3, linewidth=1.6, linestyle="--",
               label=f"Median = {med:.2f}")

    # Optimal value marker
    opt_val = best_co2 if col == "best_co2" else best_dme
    ax.axvline(opt_val, color=C2, linewidth=1.8, linestyle="-",
               label=f"Optimal = {opt_val:.2f}")

    ax.set_xlabel(label)
    ax.set_ylabel("Frequency")
    title_str = ("CO₂ Conversion Distribution"
                 if col == "best_co2"
                 else "DME Selectivity Distribution")
    ax.set_title(title_str)
    ax.legend(fontsize=8.5, facecolor="white",
              edgecolor="#cccccc", framealpha=1)

fig.suptitle("Objective Distribution — SPEA2 Pareto Solutions",
             fontsize=14, fontweight="bold", color=TEXT_COL, y=1.02)
fig.text(0.5, -0.02,
         "Dashed = median  •  Dash-dot = mean  •  Solid red = optimal solution value",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "pareto_distributions.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. OPTIMAL SOLUTION SUMMARY             ║
# ╚══════════════════════════════════════════╝

print("\n" + "═" * 52)
print("   OPTIMAL SOLUTION  —  SPEA2 (Balanced)")
print("═" * 52)
print(f"  CO₂ Conversion   : {best_co2:.4f}")
print(f"  DME Selectivity  : {best_dme:.4f}")
print(f"  Combined Score   : {best_co2 + best_dme:.4f}")

if "combination" in results_df.columns:
    print(f"\n  Category Combination:")
    print(f"  {results_df.iloc[best_idx]['combination']}")

print(f"\n  Optimal Conditions:")
for k, v in results_df.iloc[best_idx].items():
    if k not in ["combination", "best_co2", "best_dme"]:
        print(f"  {k:<30} : {v}")
print("═" * 52)

In [ ]:
# ============================================================
# 🔥 SPEA2 — 4 TARGET OPTIMIZATION (ISOLATED BLOCK)
# ============================================================

from pymoo.core.problem import Problem
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.optimize import minimize
from pymoo.termination import get_termination
import numpy as np
import pandas as pd

# =========================================
# USE EXISTING MODELS (DO NOT RELOAD)
# =========================================

co2_pipeline_4 = co2_pkg["pipeline"]
dme_pipeline_4 = dme_pkg["pipeline"]
methanol_pipeline_4 = methanol_pkg["pipeline"]
co_pipeline_4 = co_pkg["pipeline"]

features_4 = co2_pkg["features"]

print("✅ 4-target pipelines ready")


# =========================================
# STORAGE (NEW VARIABLE — SAFE)
# =========================================

results_all_4obj = []


# =========================================
# LOOP OVER CATEGORICAL COMBINATIONS
# =========================================

for i, fixed_cat in enumerate(valid_combinations):

    print(f"\n[4-OBJ] Running {i+1}/{len(valid_combinations)}")
    print(fixed_cat)

    class Problem_4OBJ(Problem):

        def __init__(self):
            super().__init__(
                n_var=len(num_cols),
                n_obj=4,
                xl=xl,
                xu=xu
            )

        def _evaluate(self, X, out, *args, **kwargs):

            results = []

            for row in X:

                input_dict = {}

                # 🔥 numeric handling (same as your original)
                for j, col in enumerate(num_cols):

                    if j in int_indices:
                        val = int(round(row[j]))
                        val = max(0, min(val, 118))
                        input_dict[col] = val
                    else:
                        input_dict[col] = row[j]

                # categorical
                for k, v in fixed_cat.items():
                    input_dict[k] = v

                # feature alignment
                input_df = pd.DataFrame([input_dict])[features_4]

                # 🔥 predictions (ALL 4)
                co2_pred = co2_pipeline_4.predict(input_df)[0]
                dme_pred = dme_pipeline_4.predict(input_df)[0]
                methanol_pred = methanol_pipeline_4.predict(input_df)[0]
                co_pred = co_pipeline_4.predict(input_df)[0]

                # 🔥 objectives
                results.append([
                    -co2_pred,         # maximize
                    -dme_pred,         # maximize
                    -methanol_pred,   # maximize
                    co_pred           # minimize
                ])

            out["F"] = np.array(results)

    # =========================================
    # RUN OPTIMIZATION
    # =========================================

    problem_4 = Problem_4OBJ()

    res_4 = minimize(
        problem_4,
        SPEA2(pop_size=20),
        get_termination("n_gen", 10),
        seed=42,
        verbose=False
    )

    # =========================================
    # EXTRACT RESULTS
    # =========================================

    co2_opt_4 = -res_4.F[:, 0]
    dme_opt_4 = -res_4.F[:, 1]
    methanol_opt_4 = -res_4.F[:, 2]
    co_opt_4 = res_4.F[:, 3]

    # 🔥 balanced scoring
    score_4 = co2_opt_4 + dme_opt_4 + methanol_opt_4 - co_opt_4
    best_idx_4 = np.argmax(score_4)

    best_X_4 = res_4.X[best_idx_4]

    # =========================================
    # BUILD BEST INPUT
    # =========================================

    best_input_4 = {}

    for j, col in enumerate(num_cols):

        if j in int_indices:
            val = int(round(best_X_4[j]))
            val = max(0, min(val, 118))
            best_input_4[col] = val
        else:
            best_input_4[col] = best_X_4[j]

    for k, v in fixed_cat.items():
        best_input_4[k] = v

    # =========================================
    # STORE RESULTS
    # =========================================

    results_all_4obj.append({
        "combination": fixed_cat,
        "best_co2": co2_opt_4[best_idx_4],
        "best_dme": dme_opt_4[best_idx_4],
        "best_methanol": methanol_opt_4[best_idx_4],
        "best_co": co_opt_4[best_idx_4],
        "best_input": best_input_4
    })


print("\n🔥 TOTAL 4-OBJECTIVE RESULTS:", len(results_all_4obj))

In [ ]:
# ============================================================
# 4-OBJECTIVE PARETO ANALYSIS — SPEA2 OPTIMISATION
# PUBLICATION EDITION — LIGHT THEME
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats

# ── Typography & palette ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BG       = "#ffffff"
PANEL    = "#fafafa"
C1       = "#1a6faf"   # steel blue
C2       = "#c0392b"   # brick red
C3       = "#1e8449"   # forest green
C4       = "#7d3c98"   # purple
C5       = "#d4621a"   # burnt orange
GRID_COL = "#e0e0e0"
TEXT_COL = "#1a1a1a"

def style_ax(ax, minor_ticks=True):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.7,
            linestyle="--", alpha=1.0, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.3)
    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="minor", length=3,
                       color="#888888", direction="in")
        ax.tick_params(which="major", length=5,
                       color="#333333", direction="in")

def save_fig(fig, fname):
    fig.savefig(fname, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")


# ── Build dataframe & compute scores ─────────────────────────────────────────
df_opt_4 = pd.DataFrame(results_all_4obj)

score    = (df_opt_4["best_co2"] +
            df_opt_4["best_dme"] +
            df_opt_4["best_methanol"] -
            df_opt_4["best_co"])

best_idx   = np.argmax(score)
best_point = df_opt_4.iloc[best_idx]

print(df_opt_4.head())
print("\n" + "═" * 52)
print("   BEST TRADE-OFF POINT — 4-Objective SPEA2")
print("═" * 52)
for k, v in best_point.items():
    print(f"  {k:<30} : {v:.4f}" if isinstance(v, float) else
          f"  {k:<30} : {v}")
print(f"  {'Combined Score':<30} : {score.iloc[best_idx]:.4f}")
print("═" * 52)


# ╔══════════════════════════════════════════╗
# ║  1. 3D PARETO SCATTER                    ║
# ╚══════════════════════════════════════════╝

fig = plt.figure(figsize=(11, 8))
fig.patch.set_facecolor(BG)
ax  = fig.add_subplot(111, projection="3d")
ax.set_facecolor(PANEL)

x = df_opt_4["best_co2"].values
y = df_opt_4["best_dme"].values
z = df_opt_4["best_methanol"].values
c = df_opt_4["best_co"].values

sc = ax.scatter(
    x, y, z,
    c=c, cmap="RdYlBu_r",
    s=55, alpha=0.82,
    edgecolors="#555555", linewidths=0.3,
    depthshade=True, zorder=3
)

# Best point — red star
ax.scatter(
    best_point["best_co2"],
    best_point["best_dme"],
    best_point["best_methanol"],
    color=C2, s=280, marker="*",
    edgecolors="#333333", linewidths=0.8,
    label=f"Optimal  (CO₂={best_point['best_co2']:.1f}, "
          f"DME={best_point['best_dme']:.1f}, "
          f"MeOH={best_point['best_methanol']:.1f})",
    zorder=6
)

# Colorbar
cbar = fig.colorbar(sc, ax=ax, fraction=0.03,
                    pad=0.1, aspect=25, shrink=0.7)
cbar.set_label("CO Selectivity (minimise →)",
               color=TEXT_COL, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar.outline.set_edgecolor("#888888")

# Axis styling
ax.set_xlabel("CO₂ Conversion (%)",  labelpad=10,
              color=TEXT_COL, fontsize=10)
ax.set_ylabel("DME Selectivity (%)", labelpad=10,
              color=TEXT_COL, fontsize=10)
ax.set_zlabel("Methanol Selectivity (%)", labelpad=10,
              color=TEXT_COL, fontsize=10)
ax.tick_params(colors=TEXT_COL, labelsize=8)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.xaxis.pane.set_edgecolor("#cccccc")
ax.yaxis.pane.set_edgecolor("#cccccc")
ax.zaxis.pane.set_edgecolor("#cccccc")
ax.grid(True, color=GRID_COL, linewidth=0.5, linestyle="--")

ax.set_title("4-Objective Pareto Space  |  SPEA2 Optimisation",
             fontsize=13, fontweight="bold", color=TEXT_COL, pad=18)
ax.legend(fontsize=8.5, facecolor="white",
          edgecolor="#cccccc", framealpha=1,
          loc="upper left")

fig.text(0.5, 0.01,
         "Colour encodes CO selectivity  •  ★ = balanced optimum  •  "
         "Objectives: maximise CO₂, DME, MeOH  |  minimise CO",
         ha="center", fontsize=8, color="#666666", style="italic")

save_fig(fig, "pareto_3d.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  2. OBJECTIVE CORRELATION HEATMAP        ║
# ╚══════════════════════════════════════════╝

obj_cols  = ["best_co2", "best_dme", "best_methanol", "best_co"]
obj_labels = ["CO₂ Conversion", "DME Selectivity",
              "Methanol Selectivity", "CO Selectivity"]

corr_mat = df_opt_4[obj_cols].corr()

# Custom diverging colormap
div_cmap = LinearSegmentedColormap.from_list(
    "research_div", [
        "#08306b", "#2171b5", "#6baed6", "#c6dbef",
        "#ffffff",
        "#fcbba1", "#cb181d", "#67000d",
    ], N=512
)

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

im = ax.imshow(corr_mat.values, cmap=div_cmap,
               vmin=-1, vmax=1, aspect="auto")

# Cell grid lines
n = len(obj_cols)
for v in np.arange(-0.5, n, 1):
    ax.axhline(v, color="#cccccc", linewidth=0.5)
    ax.axvline(v, color="#cccccc", linewidth=0.5)

# Annotate
for i in range(n):
    for j in range(n):
        val = corr_mat.values[i, j]
        tc  = "white" if abs(val) > 0.55 else TEXT_COL
        fw  = "bold"  if abs(val) > 0.7  else "normal"
        ax.text(j, i, f"{val:.2f}",
                ha="center", va="center",
                fontsize=11, color=tc,
                fontweight=fw, family="monospace")

cbar2 = fig.colorbar(im, ax=ax, fraction=0.04,
                     pad=0.02, aspect=25)
cbar2.set_label("Pearson r", color=TEXT_COL, fontsize=9)
cbar2.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar2.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar2.set_ticklabels(["-1.0", "-0.5", "0.0", "+0.5", "+1.0"])
cbar2.outline.set_edgecolor("#888888")

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(obj_labels, rotation=25,
                   ha="right", fontsize=10, color=TEXT_COL)
ax.set_yticklabels(obj_labels, fontsize=10, color=TEXT_COL)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_edgecolor("#333333")
    spine.set_linewidth(1.3)

ax.set_title("Objective Correlation — 4-Objective Pareto Solutions",
             fontsize=13, fontweight="bold", pad=14, color=TEXT_COL)

fig.text(0.5, -0.02,
         "Pearson correlation between optimised objectives  •  "
         "Values ≥ |0.7| bolded",
         ha="center", fontsize=8, color="#666666", style="italic")

save_fig(fig, "pareto_corr_heatmap.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  3. OBJECTIVE DISTRIBUTIONS (2×2)        ║
# ╚══════════════════════════════════════════╝

dist_colors = [C1, C5, C3, C2]
dist_titles = ["CO₂ Conversion (%)", "DME Selectivity (%)",
               "Methanol Selectivity (%)", "CO Selectivity (%)"]
opt_vals    = [best_point["best_co2"],  best_point["best_dme"],
               best_point["best_methanol"], best_point["best_co"]]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.patch.set_facecolor(BG)
axes = axes.flatten()

for ax, col, color, title, opt_v in zip(
        axes, obj_cols, dist_colors, dist_titles, opt_vals):

    style_ax(ax)
    data = df_opt_4[col].values

    n_bins = 18
    counts, bin_edges, patches = ax.hist(
        data, bins=n_bins,
        edgecolor="white", linewidth=0.6, zorder=3
    )

    # Light → full color gradient
    norm_b  = np.linspace(0, 1, n_bins)
    base_r  = int(color[1:3], 16) / 255
    base_g  = int(color[3:5], 16) / 255
    base_b  = int(color[5:7], 16) / 255
    for patch, nb in zip(patches, norm_b):
        patch.set_facecolor((
            1 - nb * (1 - base_r),
            1 - nb * (1 - base_g),
            1 - nb * (1 - base_b),
        ))
        patch.set_alpha(0.88)

    # KDE
    kde_x = np.linspace(data.min(), data.max(), 300)
    kde_y = stats.gaussian_kde(data)(kde_x)
    bw    = bin_edges[1] - bin_edges[0]
    ax.plot(kde_x, kde_y * len(data) * bw,
            color=C2, linewidth=2.2, label="KDE", zorder=4)

    # Mean / median / optimal
    mu  = data.mean()
    med = np.median(data)
    ax.axvline(mu,    color=C4, linewidth=1.6,
               linestyle="-.", label=f"Mean = {mu:.2f}")
    ax.axvline(med,   color=C3, linewidth=1.6,
               linestyle="--", label=f"Median = {med:.2f}")
    ax.axvline(opt_v, color=C2, linewidth=1.8,
               linestyle="-",  label=f"Optimal = {opt_v:.2f}")

    # Stats box
    sb = (f"μ = {mu:.2f}\n"
          f"σ = {data.std():.2f}\n"
          f"Min = {data.min():.2f}\n"
          f"Max = {data.max():.2f}")
    ax.text(0.97, 0.97, sb,
            transform=ax.transAxes,
            fontsize=8, color=TEXT_COL,
            family="monospace", va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.4",
                      facecolor="#fff9e6",
                      edgecolor="#ccaa00", alpha=0.95))

    ax.set_xlabel(title)
    ax.set_ylabel("Frequency")
    ax.set_title(f"Distribution — {title}")
    ax.legend(fontsize=8, facecolor="white",
              edgecolor="#cccccc", framealpha=1)

fig.suptitle("Optimised Objective Distributions  |  SPEA2 4-Objective",
             fontsize=14, fontweight="bold",
             color=TEXT_COL, y=1.02)
fig.text(0.5, -0.01,
         "Dashed = median  •  Dash-dot = mean  •  Solid red = optimal point value",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "pareto_distributions_4obj.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  4. LOW-CO REGION HIGHLIGHT              ║
# ╚══════════════════════════════════════════╝

co_thresh = df_opt_4["best_co"].quantile(0.30)
low_co    = df_opt_4[df_opt_4["best_co"] < co_thresh]
high_co   = df_opt_4[df_opt_4["best_co"] >= co_thresh]

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor(BG)
style_ax(ax)

# All points — grey background
ax.scatter(
    high_co["best_co2"], high_co["best_dme"],
    color="#bbbbbb", s=45, alpha=0.55,
    edgecolors="white", linewidths=0.3,
    zorder=2, label="Other solutions"
)

# Low CO points — colored by CO value
sc2 = ax.scatter(
    low_co["best_co2"], low_co["best_dme"],
    c=low_co["best_co"], cmap="YlGn_r",
    s=65, alpha=0.88,
    edgecolors="#333333", linewidths=0.4,
    zorder=3, label=f"Low CO  (<{co_thresh:.1f}%,  n={len(low_co)})"
)
cbar3 = fig.colorbar(sc2, ax=ax, fraction=0.04,
                     pad=0.03, aspect=28)
cbar3.set_label("CO Selectivity (%)", color=TEXT_COL, fontsize=9)
cbar3.ax.yaxis.set_tick_params(color=TEXT_COL, labelsize=8)
plt.setp(cbar3.ax.yaxis.get_ticklabels(), color=TEXT_COL)
cbar3.outline.set_edgecolor("#888888")

# Best point
ax.scatter(
    best_point["best_co2"], best_point["best_dme"],
    color=C2, s=220, marker="*",
    edgecolors="#333333", linewidths=0.8,
    zorder=5, label=f"Optimal point"
)

# Threshold annotation
ax.annotate(
    f"Low CO threshold\n< {co_thresh:.1f}%  (30th pct)",
    xy=(low_co["best_co2"].mean(), low_co["best_dme"].mean()),
    xytext=(low_co["best_co2"].mean() * 0.80,
            low_co["best_dme"].mean() * 1.12),
    fontsize=8.5, color=C4, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=C4, lw=1.2,
                    connectionstyle="arc3,rad=-0.2"),
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#f3e5f5",
              edgecolor=C4, alpha=0.92)
)

ax.set_xlabel("CO₂ Conversion (%)")
ax.set_ylabel("DME Selectivity (%)")
ax.set_title("Low-CO Region  |  Bottom 30% CO Selectivity Highlighted")
ax.legend(fontsize=9, facecolor="white",
          edgecolor="#cccccc", framealpha=1)

fig.text(0.5, -0.02,
         "Green intensity encodes CO selectivity within low-CO subset  •  "
         "Grey = remaining Pareto solutions",
         ha="center", fontsize=8, color="#666666", style="italic")

save_fig(fig, "pareto_low_co_region.png")
plt.show()


# ╔══════════════════════════════════════════╗
# ║  5. 2×2 PAIRWISE PARETO SCATTER GRID    ║
# ╚══════════════════════════════════════════╝

pairs  = [("best_co2",  "best_dme"),
          ("best_co2",  "best_methanol"),
          ("best_dme",  "best_methanol"),
          ("best_co2",  "best_co")]
xlabels = ["CO₂ Conversion (%)", "CO₂ Conversion (%)",
           "DME Selectivity (%)", "CO₂ Conversion (%)"]
ylabels = ["DME Selectivity (%)", "Methanol Selectivity (%)",
           "Methanol Selectivity (%)", "CO Selectivity (%)"]
titles  = ["CO₂ vs DME", "CO₂ vs Methanol",
           "DME vs Methanol", "CO₂ vs CO  (trade-off)"]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.patch.set_facecolor(BG)
axes = axes.flatten()

for ax, (cx, cy), xl, yl, ttl in zip(
        axes, pairs, xlabels, ylabels, titles):

    style_ax(ax)

    sc_p = ax.scatter(
        df_opt_4[cx], df_opt_4[cy],
        c=score, cmap="RdYlBu",
        s=50, alpha=0.80,
        edgecolors="#555555", linewidths=0.3, zorder=3
    )
    fig.colorbar(sc_p, ax=ax, fraction=0.04,
                 pad=0.02, aspect=25).set_label(
        "Combined score", color=TEXT_COL, fontsize=8)

    # Best point
    ax.scatter(
        best_point[cx], best_point[cy],
        color=C2, s=180, marker="*",
        edgecolors="#333333", linewidths=0.8,
        zorder=5, label="Optimal"
    )

    # Trend line
    m, b, r, p, _ = stats.linregress(
        df_opt_4[cx], df_opt_4[cy]
    )
    xline = np.linspace(df_opt_4[cx].min(),
                        df_opt_4[cx].max(), 100)
    ax.plot(xline, m * xline + b,
            color=C4, linewidth=1.5,
            linestyle="--", alpha=0.7,
            label=f"r = {r:.2f}")

    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(ttl)
    ax.legend(fontsize=8.5, facecolor="white",
              edgecolor="#cccccc", framealpha=1)

fig.suptitle("Pairwise Objective Trade-offs  |  SPEA2 4-Objective Pareto",
             fontsize=14, fontweight="bold",
             color=TEXT_COL, y=1.02)
fig.text(0.5, -0.01,
         "Colour = combined objective score  •  ★ = optimal solution  •  "
         "Dashed = linear trend  •  r = Pearson correlation",
         ha="center", fontsize=8, color="#666666", style="italic")

plt.tight_layout()
save_fig(fig, "pareto_pairwise_grid.png")
plt.show()